# DR-VERGE — Final Research Notebook

**A rigorous investigation of complementarity-aware knowledge transfer and INT8 deployment for
lightweight two-field diabetic retinopathy grading.**

Implements `revision/dr-verge-rev.md` on top of the validated rev3 core. Supersedes
`full_pipeline_notebook_rev3.ipynb`.

---

## Research questions (locked before running)

**RQ1 — Knowledge transfer.** *To what extent can Complementarity-Shift Distillation transfer the
dual-view decision benefit of a two-field teacher to a lightweight student, compared with no
distillation, standard logit distillation, and feature distillation?*

Judged on **two independent axes**, so the finding is informative regardless of which way it lands:
- *Predictive*: QWK (primary), Accuracy, Macro-F1, MAE, Severe-Error Rate
- *Mechanistic*: ShiftL1/ShiftMAE, Cosine agreement, Benefit correlation, internal/external dual-view gain

**The comparison ladder, named precisely.** Feature-KD and CSD each add ONE term on top of the same
tuned logit-KD baseline, so the four conditions are

```
no distillation  ->  logit-KD  ->  logit-KD + feature-KD  ->  logit-KD + CSD
```

Write it as *"CSD augmentation and feature-distillation augmentation over a standard logit-KD
baseline"* — never a bare "CSD vs Feature-KD", which would imply two disjoint methods.
`table_condition_labels.csv` carries the exact label for every condition.

**Δ is an operational proxy.** `Δ = p_dual − (p_macula + p_disc)/2` is measured through three heads,
so it can also absorb head discrepancy and calibration discrepancy. Describe it as an **operational
proxy of the dual-view decision shift**, not as pure anatomical complementarity. The same-head
counterfactual ablation (`abl_csd_counterfactual`) is the control that bounds this concern.

If QWK(CSD) ≈ QWK(KD) but ShiftFidelity(CSD) > ShiftFidelity(KD), that is still a scientific
finding. If CSD fails on both, that is a valid answer too.

**RQ2 — Quantization / deployment.** *To what extent can INT8 post-training quantization and
quantization-aware training reduce model size and CPU latency while preserving categorical and
ordinal grading performance of the best lightweight dual-view model?*

Compares `M*_FP32` vs `M*_PTQ-INT8` vs `M*_QAT-INT8` (plus a matched FP32 fine-tuning control),
where `M*` is selected **on validation only**. PTQ and QAT quantize the **identical operator set**
(eager, backbone-only) so the comparison isolates the training procedure, not the scope.

---

## Locked protocol (do not change after the first full run)

| Item | Value |
|---|---|
| Primary metric | **QWK** (ordinal; DR grades are 0<1<2<3<4) |
| Core seeds | 42, 123, 2026, 3407, 8888 (**5**) |
| Baseline seeds | 42, 123, 2026 (**3**) |
| Model selection | `argmax QWK_val`; ties (<0.005) → Macro-F1 → lower SER → lower MAE → simpler method |
| Test set | DRTiD official test — not used for selection **within this locked run** (see note) |
| External validation | DeepDRiD — frozen, no tuning, evaluated last |
| Statistics | Hierarchical paired cluster bootstrap over **matched seeds** + cluster permutation test, B=10,000, Holm-corrected |
| Deployment | Every exported artifact is **re-loaded from disk** and re-checked, quantized ones included |
| Pre-registered comparisons | RQ1: CSD vs {NoDistill, LogitKD, FeatureKD}. RQ2: {PTQ, QAT} vs FP32, QAT vs PTQ |

**Everything is saved.** Every figure ships PNG+PDF+SVG **and** a companion CSV — no number lives
only inside an image. Per-sample predictions, per-epoch gradient contributions, configs, metadata,
and a model registry are all written to disk.

---

## What this adds over rev3

rev3 fixed the three defects that made rev2's RQ1 test uninformative (collapsed CORAL thresholds,
40×-undersized student, CSD with no gradient). That core is **kept unchanged**. This notebook adds:

1. 5 seeds on core conditions (was 3)
2. Complete categorical metrics: Accuracy, Balanced Accuracy, macro/weighted P/R/F1, per-grade
   P/R/F1/specificity/support
3. Confusion matrices (raw + normalized) with **automatic prediction-collapse warnings**
4. **QAT** alongside PTQ — RQ2 becomes a three-way FP32/PTQ/QAT comparison
5. Quantization with a **matched scope**: PTQ and QAT both eager backbone-only, so RQ2 compares the
   procedure and not the operator set. PT2E (`torch.export` + `prepare_pt2e`) is run as a
   **supplementary** deployment-path demonstration, reported separately and excluded from RQ2.
   Export artifacts are `checkpoint.pt` / `model_object.pt` / `model.pt2` / `model.onnx`
   (TorchScript is deprecated and is no longer the deployment path)
6. Full efficiency suite: params, serialized size, compression ratio, mean/median/p95/p99 latency,
   throughput, speedup, memory — under a standardized benchmark protocol
7. Performance-retention metrics (INT8 vs FP32)
8. Statistics: hierarchical paired cluster bootstrap over MATCHED seeds + permutation p-values
9. DeepDRiD **external confirmatory validation**, frozen
10. Deployment artifacts + `predict_dr()` inference wrapper + parity checks + model registry
11. Ten publication-grade figures, each with a companion data CSV
12. Gates 0–9 with a final consolidated gate report

---

## What changed after the first pre-flight

The first pre-flight ran end-to-end (15/17 gates) and exposed a set of validity and hygiene issues
that a successful run alone would never have surfaced:

| Issue found | Fix |
|---|---|
| Deployment model chosen from **DRTiD test** results | The whole decision moved to validation and is frozen before the test set is read |
| The "severe error must not worsen" rule consulted the **QWK** interval | `SevereErrorRate` is now a bootstrap metric and the rule reads ΔSER |
| `best_fp32`/PTQ on 1 seed vs QAT on 3 — no `QAT_s ↔ FP32_s` pairing | Every RQ2 variant derives from `FP32_s` on all 5 core seeds |
| `M*` = the single best `(method, seed)` row out of 20 — a seed lottery | Two-stage: method by mean validation QWK, then a checkpoint within it |
| Hyperparameters chosen on seed 42 alone | Grids scored as mean validation QWK over 3 tuning seeds |
| PTQ calibrated on a **shuffled 512-eye subset** | Deterministic: `shuffle=False`, all 800 training eyes, manifest + SHA-256 |
| `PTQ ops == QAT ops == 15` treated as proof of the same scope | The sorted quantized **module paths** must be identical (blocking) |
| `gnorm_aux = 0.0` — the probe could not see the auxiliary loss | Probes on the **shared backbone**, where every loss term lands |
| CSD/task gradient balance measured before the teacher existed | Gate 4b re-measures it with the frozen teacher and the real global scale |
| 210 `can only test a child process` DataLoader assertions | `num_workers=0` everywhere; data staged on local SSD |
| Gate 0 passed unconditionally | Validates versions, CUDA, quant engine, dependency conflicts |
| PT2E import failed (`torch.ao.quantization.quantize_pt2e` removed in torch 2.11) | Imports from torchao first; still supplementary, never in RQ2 |
| Every ONNX export failed (`onnxscript` missing) | Installed; FP32 ONNX + Runtime parity is now its own gate |
| One `Gate8_Export` hid ONNX and `.pt2` failures | Split into 8a mandatory / 8b FP32 ONNX / 8c optional `.pt2` / 8d reload |
| Gate 3 failed the whole study when any ablation missed a rare grade | 3A core conditions (blocking) vs 3B diagnostics (reported) |
| DeepDRiD Set-A silently yielded 597 of 600 eyes | Per-eye audit CSV; the cause (patients 77/164) is named |
| Set-B had already been inspected, so it is not "untouched" | **Set-C** (`Online-Challenge1&2-Evaluation`) is the confirmatory set |
| `predict_dr()` always served the FP32 student | Serves `DEPLOY_CHOICE`, loaded from the exported artifact |
| Checkpoint reuse checked only key/shape | `PROTOCOL_HASH` must match, and `RESUME_EXACT` gates reuse at all |

## 01 — Environment & Reproducibility (Gate 0)

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab ships a CUDA-enabled torch -- we deliberately do NOT reinstall it. Everything else is
# PINNED, because quantization and export APIs are highly version-sensitive.
#
# Two installs were added after the first preflight, which failed on missing optional deps:
#   onnxscript  -- torch.onnx.export(dynamo=True) requires it; without it EVERY ONNX export failed
#   torchao>=0.14 -- torch 2.11 removed torch.ao.quantization.quantize_pt2e; PT2E now lives at
#                   torchao.quantization.pt2e.quantize_pt2e. The preflight's PT2E gate failed purely
#                   because of that moved import, not because PT2E is broken.
# tqdm/scikit-learn floors were raised to clear the pip dependency-resolver conflicts the first
# preflight printed. What matters is not "newest" but "preflighted, then locked".
!pip install -q "albumentations==1.4.21" "scikit-learn>=1.6,<1.8" "pandas==2.2.2" "tqdm>=4.67,<5"                "pyyaml==6.0.2" "psutil==6.0.0" "onnx>=1.17" "onnxruntime>=1.19" "scipy>=1.14" "openpyxl>=3.1"
!pip install -q "onnxscript>=0.1.0" || echo "onnxscript unavailable -- ONNX export will report as FAILED (not silently skipped)"
!pip install -q -U "torchao>=0.14" || echo "torchao unavailable -- PT2E path will report as unavailable (not silently faked)"

import torch, torchvision, numpy, sklearn, platform, subprocess, json, os
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("numpy       :", numpy.__version__)
print("sklearn     :", sklearn.__version__)
for _m in ("torchao", "onnxscript", "onnxruntime"):
    try:
        _mod = __import__(_m); print(f"{_m:12s}:", getattr(_mod, "__version__", "(no __version__)"))
    except Exception as _e:
        print(f"{_m:12s}: NOT AVAILABLE ->", _e)
print("python      :", platform.python_version())
print("CUDA avail  :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("quant engines:", torch.backends.quantized.supported_engines)
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU."

## 02–03 — Locked configuration & paths

In [ ]:
import os, json, hashlib, platform, subprocess, math, random, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

# ---------------- EDIT THESE ----------------
DRIVE_BASE = "/content/drive/MyDrive/DR-VERGE"

# PREFLIGHT=True runs the ENTIRE pipeline end-to-end at tiny scale into a SEPARATE namespace:
#   backbones 1 epoch | teacher 1+1 | students 2 epochs | 1 seed everywhere | grids truncated
#   PTQ 4 calibration batches | QAT + FP32-FT 1 epoch | bootstrap/permutations 300 | benchmark 2x10
# It proves every stage EXECUTES and every required RQ column is populated. It proves NOTHING about
# accuracy -- the models are deliberately undertrained, so the performance gates are non-blocking
# here and their preflight verdicts are meaningless. Read the gate report for "did it run", not
# "is it good".
#
# In the FINAL run (PREFLIGHT=False) these gates BLOCK: Gate0 environment, Gate2 teacher dual-view
# advantage, Gate3A core student viability, Gate4/4b CSD signal and gradient balance,
# Gate6/6a/6c PTQ integrity and scope matching, Gate7 QAT integrity, Gate8a/8b/8d deployment
# artifacts, Gate9/9c external validation, and RQ completeness. A blocking gate raises GateFailure
# and stops the run, because nothing computed past a failed gate is interpretable.
PREFLIGHT = True

# RUN_TAG isolates this run's artifacts. NEVER reuse a previous final namespace: the protocol
# changed after the first preflight (multi-seed RQ2, validation-only deployment selection,
# deterministic PTQ calibration), so artifacts from `final_locked_v1` are NOT comparable.
RUN_TAG = "preflight_v2" if PREFLIGHT else "final_locked_v2_20260809"

# RESUME_EXACT=False means: ignore every existing checkpoint and retrain from scratch. Set it True
# ONLY to continue a run that a Colab disconnect interrupted -- and even then, a checkpoint is
# reused only when its stored PROTOCOL_HASH equals this run's, so a checkpoint produced under a
# different protocol can never be silently inherited.
#
# The final run trains ~99 models (5 core seeds, 3 tuning seeds per grid point, 5-seed RQ2), which
# will not finish inside one Colab session. The intended workflow is:
#     session 1:  RESUME_EXACT = False   (fresh; artifacts land in artifacts_<RUN_TAG>)
#     session N:  RESUME_EXACT = True    (same RUN_TAG; completed models are skipped, the rest train)
# Because reuse is gated on PROTOCOL_HASH, resuming can never mix protocols -- if anything about the
# configuration or the splits changed, every checkpoint is rejected and retrained.
RESUME_EXACT = False

# Determinism for the final scientific run. torch's docs are explicit that bit-exact reproducibility
# is not guaranteed across versions/platforms, but disabling cuDNN benchmarking and requesting
# deterministic algorithms removes the controllable sources of nondeterminism.
FINAL_DETERMINISTIC = True

# DataLoader workers. The first preflight emitted 210 "AssertionError: can only test a child process"
# tracebacks from _MultiProcessingDataLoaderIter.__del__ -- a worker-teardown race that does not
# corrupt results but fills a scientific run with exception noise. Single-process loading removes it
# outright; the datasets here are small enough that the throughput cost is irrelevant.
DEFAULT_NUM_WORKERS = 0

# Optionally stage the read-only image data on Colab's local SSD. Drive/FUSE is slow and is the
# reason worker processes stall. Artifacts ALWAYS go to Drive regardless of this setting.
USE_LOCAL_DATA_CACHE = True
LOCAL_DATA_ROOT = "/content/dr_verge_data"
# --------------------------------------------

DATASET_ROOT = f"{DRIVE_BASE}/dataset"

def _resolve_drtid_root(root):
    for cand in (f"{root}/DRTiD/DRTiD", f"{root}/DRTiD"):
        if os.path.exists(f"{cand}/Ground Truths/DR_grade/a. DR_grade_Training.csv"):
            return cand
    return f"{root}/DRTiD/DRTiD"

def _resolve_deepdrid_root(root):
    for cand in (f"{root}/DeepDRiD-master/regular_fundus_images",
                 f"{root}/DeepDRiD/regular_fundus_images",
                 f"{root}/DeepDRiD-master", f"{root}/DeepDRiD"):
        if os.path.exists(f"{cand}/regular-fundus-validation/regular-fundus-validation.csv"):
            return cand
    return None

def _stage_local(src_root):
    """Copy the read-only dataset tree to local SSD once. Returns the path actually used."""
    if not (USE_LOCAL_DATA_CACHE and os.path.isdir(src_root)):
        return src_root
    dst = f"{LOCAL_DATA_ROOT}/{os.path.basename(src_root.rstrip('/'))}"
    marker = f"{dst}/.stage_complete"
    if os.path.exists(marker):
        print(f"  local cache HIT  {dst}")
        return dst
    import shutil
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f"  staging {src_root} -> {dst} (one-off copy; artifacts still go to Drive)")
    try:
        shutil.copytree(src_root, dst, dirs_exist_ok=True)
        open(marker, "w").write("ok")
        return dst
    except Exception as e:
        print(f"  local staging failed ({e!r}) -- reading directly from Drive instead")
        return src_root

DRTID_ROOT    = _stage_local(_resolve_drtid_root(DATASET_ROOT))
APTOS_ROOT    = _stage_local(f"{DATASET_ROOT}/APTOS")
_dd           = _resolve_deepdrid_root(DATASET_ROOT)
DEEPDRID_ROOT = _stage_local(_dd) if _dd else None

ART          = f"{DRIVE_BASE}/artifacts_{RUN_TAG}"
SPLITS_DIR   = f"{ART}/splits"
CKPT_DIR     = f"{ART}/checkpoints"
MODELS_DIR   = f"{ART}/models"
RESULTS_DIR  = f"{ART}/results"
FIGURES_DIR  = f"{RESULTS_DIR}/figures"
TABLES_DIR   = f"{RESULTS_DIR}/tables"
METRICS_DIR  = f"{RESULTS_DIR}/metrics"
PREDS_DIR    = f"{RESULTS_DIR}/predictions"
LOGS_DIR     = f"{RESULTS_DIR}/logs"
CONFIG_DIR   = f"{ART}/configs"
for d in [ART, SPLITS_DIR, CKPT_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
          METRICS_DIR, PREDS_DIR, LOGS_DIR, CONFIG_DIR,
          f"{CKPT_DIR}/pretrained_backbones", f"{CKPT_DIR}/teacher", f"{CKPT_DIR}/student"]:
    os.makedirs(d, exist_ok=True)

# ================= LOCKED EXPERIMENT PROTOCOL =================
SEEDS_CORE     = [42, 123, 2026, 3407, 8888]   # no-distill / logit-KD / feature-KD / CSD
SEEDS_BASELINE = [42, 123, 2026]               # single-view baselines, ablations
# RQ2 now runs on the SAME seeds as RQ1. Every quantization variant is derived from FP32_s, so
# FP32_s / PTQ_s / QAT_s / FP32FT_s / FT_PTQ_s form an exactly matched set per seed and the paired
# bootstrap never needs a positional-pairing fallback.
SEEDS_QAT      = list(SEEDS_CORE)
# Hyperparameters are scored as the MEAN validation QWK over these seeds, not on one lucky seed.
SEEDS_TUNING   = [42, 123, 2026]
PRIMARY_SEED   = 42
# Pre-registered inferential seed: statistics use ALL matched core seeds, but where a single seed
# must be named it is this one -- fixed in advance, never the best-performing one.
INFERENTIAL_SEED = 42

if PREFLIGHT:
    SEEDS_CORE, SEEDS_BASELINE, SEEDS_QAT, SEEDS_TUNING = [42], [42], [42], [42]

IMG_SIZE       = 224
NUM_CLASSES    = 5
NUM_THRESHOLDS = NUM_CLASSES - 1
POS_WEIGHT_MODE   = "sqrt"                                  # none | sqrt | full
STUDENT_CHANNELS  = (32, 64, 96, 128, 160, 192, 224)        # ~330K-param student
FUSION_TYPE       = "interaction_mlp"

# Model selection (validation only) -- tie-break chain fixed in advance
SELECTION_METRIC   = "QWK"
SELECTION_TIE_EPS  = 0.005
SELECTION_TIEBREAK = ["MacroF1", "-SevereErrorRate", "-MAE"]

# Statistics
BOOTSTRAP_B      = 10000
BOOTSTRAP_ALPHA  = 0.05
PREREGISTERED_COMPARISONS = {
    "RQ1": [("dual_csd", "dual_no_distill"), ("dual_csd", "dual_logitkd"), ("dual_csd", "dual_featkd")],
    # qat_int8 vs fp32_ft_control isolates fake-quantization adaptation from the effect of simply
    # giving the model extra fine-tuning epochs -- without it, any QAT gain is confounded.
    # Every RQ2 pair is now matched seed-for-seed, because all variants derive from FP32_s.
    #   qat vs fp32_ft_control -> is the QAT gain more than extra fine-tuning?
    #   qat vs ft_ptq_int8     -> is adapting to quantization noise better than fine-tuning first
    #                             and quantizing after?
    "RQ2": [("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"), ("qat_int8", "ptq_int8"),
            ("qat_int8", "fp32_ft_control"), ("qat_int8", "ft_ptq_int8")],
}

# Standardized CPU benchmark protocol. `repeats` independent repetitions of `runs` inferences each:
# Colab's CPU is shared, so a single 500-run block can be biased by a noisy neighbour. We report the
# median-of-medians across repetitions plus the IQR between them.
BENCH = {"batch_size": 1, "warmup": 50, "runs": 500, "threads": 1, "repeats": 5}
BENCH_PREFLIGHT = {"batch_size": 1, "warmup": 3, "runs": 10, "threads": 1, "repeats": 2}

# Teacher / student / quantization training constants -- named here so CONFIG_SNAPSHOT can record
# every load-bearing value instead of leaving them buried in default arguments.
TEACHER_CFG = {"freeze_epochs": 5, "finetune_epochs": 20, "freeze_lr": 3e-4, "finetune_lr": 1e-4,
               "batch_size": 16, "patience": 5, "lambda_aux": 0.3}
STUDENT_CFG = {"epochs": 40, "lr": 1e-3, "batch_size": 32, "patience": 8, "lambda_aux": 0.5}
QAT_CFG     = {"epochs": 10, "patience": 4, "batch_size": 16, "lr_grid": [1e-5, 3e-5, 1e-4]}
PTQ_CFG     = {"calibration": "full_training_split", "batch_size": 16, "shuffle": False}

# DeepDRiD field-order is NOT documented in its public CSVs (no column says which of _1/_2 is
# macula- vs disc-centred). Rather than hide that behind an assumption, external validation is
# evaluated under BOTH orderings and both are reported -- turning the unknown into a robustness check.
# PRE-REGISTERED primary ordering, fixed before any DeepDRiD label-performance is inspected.
# It matches DRTiD's documented convention (field 1 = macula-centred), which is the only prior we
# have. The reverse ordering is run as a SENSITIVITY analysis and reported as supplementary --
# whichever scores higher must NOT be promoted to the headline result after the fact.
DEEPDRID_PRIMARY_FIELD_ORDER = "_1=macula"
DEEPDRID_FIELD_ORDERS = [DEEPDRID_PRIMARY_FIELD_ORDER, "_1=disc"]

CONFIG_SNAPSHOT = dict(
    run_tag=RUN_TAG, preflight=PREFLIGHT, resume_exact=RESUME_EXACT,
    final_deterministic=FINAL_DETERMINISTIC, num_workers=DEFAULT_NUM_WORKERS,
    seeds_core=SEEDS_CORE, seeds_baseline=SEEDS_BASELINE, seeds_qat=SEEDS_QAT,
    seeds_tuning=SEEDS_TUNING, primary_seed=PRIMARY_SEED, inferential_seed=INFERENTIAL_SEED,
    img_size=IMG_SIZE, num_classes=NUM_CLASSES, pos_weight_mode=POS_WEIGHT_MODE,
    student_channels=list(STUDENT_CHANNELS), fusion_type=FUSION_TYPE,
    teacher_cfg=TEACHER_CFG, student_cfg=STUDENT_CFG, qat_cfg=QAT_CFG, ptq_cfg=PTQ_CFG,
    selection_metric=SELECTION_METRIC, selection_tie_eps=SELECTION_TIE_EPS,
    selection_tiebreak=SELECTION_TIEBREAK, selection_rule="two_stage_method_then_checkpoint",
    bootstrap_B=BOOTSTRAP_B, bootstrap_alpha=BOOTSTRAP_ALPHA,
    preregistered_comparisons={k: [list(p) for p in v] for k, v in PREREGISTERED_COMPARISONS.items()},
    holm_per_family=True, bench=BENCH,
    deepdrid_primary_field_order=DEEPDRID_PRIMARY_FIELD_ORDER,
    deepdrid_field_orders=DEEPDRID_FIELD_ORDERS,
)

_expected = [f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv",
             f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv",
             f"{DRTID_ROOT}/Original Images",
             f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/valid.csv",
             f"{APTOS_ROOT}/train_images/train_images", f"{APTOS_ROOT}/val_images/val_images"]
_missing = [p for p in _expected if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError("Dataset missing:\n" + "\n".join(_missing))

print("DRTID_ROOT    :", DRTID_ROOT)
print("APTOS_ROOT    :", APTOS_ROOT)
print("DEEPDRID_ROOT :", DEEPDRID_ROOT or "NOT FOUND -- external validation will be SKIPPED (reported, not hidden)")
print("artifacts     :", ART)
print("run mode      :", "PREFLIGHT rehearsal" if PREFLIGHT else "FINAL LOCKED RUN",
      f"| resume_exact={RESUME_EXACT} deterministic={FINAL_DETERMINISTIC} workers={DEFAULT_NUM_WORKERS}")

In [ ]:
# ---- Gate 0: environment lock + provenance ----
def _pip_freeze():
    try:
        return subprocess.check_output(["pip", "freeze"], text=True)
    except Exception as e:
        return f"(pip freeze failed: {e})"

def _git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True,
                                        stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "unavailable (not a git checkout in this runtime)"

import torchvision, sklearn
try:
    import torchao; _torchao_v = torchao.__version__
except Exception:
    _torchao_v = None

ENVIRONMENT = {
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "torchao": _torchao_v, "numpy": np.__version__, "sklearn": sklearn.__version__,
    "python": platform.python_version(), "platform": platform.platform(),
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "quantized_engines": list(torch.backends.quantized.supported_engines),
    "git_commit": _git_commit(), "timestamp": pd.Timestamp.now().isoformat(),
}
with open(f"{CONFIG_DIR}/environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)
with open(f"{CONFIG_DIR}/pip_freeze.txt", "w") as f:
    f.write(_pip_freeze())
with open(f"{CONFIG_DIR}/config_locked.json", "w") as f:
    json.dump(CONFIG_SNAPSHOT, f, indent=2)

GATES = {}
class GateFailure(RuntimeError):
    pass

def record_gate(name, passed, detail="", blocking=False):
    """A gate that only prints a warning is a log line, not a gate. blocking=True raises and stops
    the run, because anything computed past a failed upstream gate is not interpretable."""
    GATES[name] = {"passed": bool(passed), "detail": detail, "blocking": bool(blocking)}
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    if blocking and not passed:
        raise GateFailure(
            f"{name} FAILED and is blocking: {detail}. Fix the upstream cause before continuing."
        )
    return passed

# ---- Gate 0 actually validates something now ----
# The previous version was `record_gate("Gate0_Environment", True, ...)` -- literally any environment
# passed, which makes it a log line rather than a gate. These are the load-bearing packages: torch
# and torchvision decide autograd/quantization behaviour, sklearn provides the QWK reference,
# numpy/scipy the statistics. Minor-version drift inside a family is tolerated; a major-version jump
# is not, because that is exactly where quantization and export APIs move.
EXPECTED_ENV = {"torch": "2.11", "torchvision": "0.26", "numpy": "2", "sklearn": "1", "python": "3.12"}

def _major_minor(v, parts=2):
    return ".".join(str(v).split("+")[0].split(".")[:parts])

env_problems = []
for pkg, want in EXPECTED_ENV.items():
    got = str(ENVIRONMENT.get(pkg, ""))
    if _major_minor(got, len(want.split("."))) != want:
        env_problems.append(f"{pkg}: expected {want}.x, got {got}")

# A GPU is required: the final run is far too slow on CPU and CPU/GPU kernels are not bit-identical.
if not torch.cuda.is_available():
    env_problems.append("no CUDA device visible")

# The eager INT8 path needs an x86 engine; without one, PTQ/QAT cannot run at all.
if not ({"fbgemm", "x86", "onednn"} & set(ENVIRONMENT["quantized_engines"])):
    env_problems.append(f"no usable quantized engine in {ENVIRONMENT['quantized_engines']}")

# `pip check` -- the first preflight ran with unresolved dependency conflicts (tqdm, scikit-learn).
# Conflicts are split by relevance: a clash involving a package DR-VERGE actually computes with is a
# blocking problem, whereas Colab's pre-installed tensorflow/streamlit disagreeing about numpy or
# packaging has nothing to do with this study and must not be able to abort a locked run.
LOAD_BEARING_PKGS = ("torch", "torchvision", "torchao", "numpy", "scipy", "scikit-learn", "sklearn",
                     "pandas", "albumentations", "onnx", "onnxruntime", "onnxscript", "opencv",
                     "pillow", "tqdm")
try:
    _pipchk = subprocess.run(["pip", "check"], capture_output=True, text=True, timeout=300)
    PIP_CHECK_OK, PIP_CHECK_OUT = _pipchk.returncode == 0, (_pipchk.stdout + _pipchk.stderr).strip()
except Exception as e:
    PIP_CHECK_OK, PIP_CHECK_OUT = False, f"pip check could not run: {e!r}"
PIP_CONFLICT_LINES = [l for l in PIP_CHECK_OUT.splitlines() if l.strip()]
PIP_RELEVANT_CONFLICTS = [l for l in PIP_CONFLICT_LINES
                          if any(p in l.lower() for p in LOAD_BEARING_PKGS)]
with open(f"{CONFIG_DIR}/pip_check.txt", "w") as f:
    f.write(f"returncode_ok={PIP_CHECK_OK}\nrelevant_conflicts={len(PIP_RELEVANT_CONFLICTS)}"
            f"\n\n{PIP_CHECK_OUT}")
if not PIP_CHECK_OK:
    print(f"pip check: {len(PIP_CONFLICT_LINES)} conflict line(s), "
          f"{len(PIP_RELEVANT_CONFLICTS)} involving load-bearing packages")
    print(PIP_CHECK_OUT[:2000])
if PIP_RELEVANT_CONFLICTS:
    env_problems.append(f"dependency conflicts involve load-bearing packages: {PIP_RELEVANT_CONFLICTS[:3]}")

ENVIRONMENT["expected_env"] = EXPECTED_ENV
ENVIRONMENT["env_problems"] = env_problems
ENVIRONMENT["pip_check_ok"] = PIP_CHECK_OK
with open(f"{CONFIG_DIR}/environment.json", "w") as f:
    json.dump(ENVIRONMENT, f, indent=2)

record_gate("Gate0_Environment", not env_problems,
            f"torch={torch.__version__} torchvision={torchvision.__version__} torchao={_torchao_v} "
            f"engines={ENVIRONMENT['quantized_engines']}"
            + ("" if not env_problems else " | PROBLEMS: " + "; ".join(env_problems)),
            blocking=not PREFLIGHT)
# Reported, never blocking on its own: unrelated pre-installed packages disagreeing with each other
# say nothing about whether DR-VERGE's own stack is sound.
record_gate("Gate0b_DependencyCheck", PIP_CHECK_OK,
            "pip check clean" if PIP_CHECK_OK else
            f"{len(PIP_CONFLICT_LINES)} conflict line(s), {len(PIP_RELEVANT_CONFLICTS)} load-bearing "
            "(see configs/pip_check.txt)")
print(json.dumps(ENVIRONMENT, indent=2))

## 04 — Reproducibility utilities

In [ ]:
def set_seed(seed: int, deterministic: bool = None):
    """Seeds every RNG. `deterministic` defaults to FINAL_DETERMINISTIC so the final run is
    deterministic by default instead of only when a caller remembers to ask.

    torch's own documentation is clear that bit-exact reproducibility is not guaranteed across
    versions or platforms; what this removes is the controllable nondeterminism (cuDNN autotuning and
    nondeterministic kernels). `warn_only=True` keeps ops without a deterministic implementation
    working -- they warn instead of raising, which is the right trade for a long run.
    """
    if deterministic is None:
        deterministic = bool(globals().get("FINAL_DETERMINISTIC", False))
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required by deterministic cuBLAS
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as e:
            print(f"  deterministic algorithms unavailable here: {e!r}")
    else:
        torch.backends.cudnn.benchmark = True

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def make_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def robust_torch_load(path, map_location=None, retries=6, delay=1.0):
    """Drive's FUSE mount can lag behind its own writes -- retry rather than crash a long run."""
    last = None
    for i in range(retries):
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except (FileNotFoundError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_load retry {i+1}/{retries} for {path}")
                time.sleep(delay); delay *= 1.5
    raise last

def robust_torch_save(obj, path, retries=6, delay=1.0):
    """Also stamps the protocol hash onto every dict-shaped checkpoint, so a saved artifact always
    carries the identity of the protocol that produced it."""
    if isinstance(obj, dict) and "protocol_hash" not in obj and globals().get("PROTOCOL_HASH"):
        obj = {**obj, "protocol_hash": PROTOCOL_HASH, "run_tag": RUN_TAG}
    last = None
    parent = os.path.dirname(path)
    for i in range(retries):
        try:
            if parent: os.makedirs(parent, exist_ok=True)
            torch.save(obj, path); return
        except (RuntimeError, OSError) as e:
            last = e
            if i < retries - 1:
                print(f"  robust_torch_save retry {i+1}/{retries} for {path}: {e}")
                time.sleep(delay); delay *= 1.5
    raise last

def compute_protocol_hash():
    """SHA256 over the locked config + the exact data splits.

    Key/shape compatibility is NOT enough to justify reusing a checkpoint: two runs can produce
    identically-shaped weights from different losses, learning rates, transforms or splits. Binding
    reuse to this hash means a checkpoint from an earlier protocol can never be silently inherited.
    """
    parts = [json.dumps(CONFIG_SNAPSHOT, sort_keys=True, default=str)]
    for p in (DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV):
        parts.append(f"{os.path.basename(p)}:{sha256_file(p) if os.path.exists(p) else 'absent'}")
    return hashlib.sha256("|".join(parts).encode()).hexdigest()

PROTOCOL_HASH = None      # set once the splits exist (Section 05)

def prepared_checkpoint_reusable(ckpt_path):
    """Reuse test for checkpoints whose state_dict belongs to a PREPARED quantization graph.

    Those keys (fused Conv-BN, observer buffers) cannot be validated against a plain module, so the
    ordinary key/shape check does not apply. The protocol identity still must: without this, a fresh
    run would silently inherit a QAT or FP32-control checkpoint produced under a different protocol.
    """
    if not (RESUME_EXACT and os.path.exists(ckpt_path)):
        return False
    try:
        raw = robust_torch_load(ckpt_path, map_location="cpu")
        if PROTOCOL_HASH is not None and isinstance(raw, dict):
            if raw.get("protocol_hash") != PROTOCOL_HASH:
                print(f"  {os.path.basename(ckpt_path)} was produced under a DIFFERENT protocol "
                      f"({str(raw.get('protocol_hash'))[:12]}... vs {PROTOCOL_HASH[:12]}...) -- retraining.")
                return False
        return True
    except Exception as e:
        print(f"  {os.path.basename(ckpt_path)} unreadable ({e!r}) -- retraining.")
        return False

def checkpoint_is_compatible(ckpt_path, model, unwrap_key="model_state"):
    """Side-effect-free key/shape check -- never partially mutates `model`.

    Also refuses any checkpoint whose stored protocol_hash differs from this run's, and refuses
    everything when RESUME_EXACT is False (a fresh run must not inherit anything)."""
    if not os.path.exists(ckpt_path): return False
    if not RESUME_EXACT:
        return False
    try:
        raw = robust_torch_load(ckpt_path, map_location="cpu")
        if isinstance(raw, dict) and PROTOCOL_HASH is not None:
            saved = raw.get("protocol_hash")
            if saved != PROTOCOL_HASH:
                print(f"  {os.path.basename(ckpt_path)} was produced under a DIFFERENT protocol "
                      f"({str(saved)[:12]}... vs {PROTOCOL_HASH[:12]}...) -- retraining.")
                return False
        state = raw[unwrap_key] if (unwrap_key and isinstance(raw, dict) and unwrap_key in raw) else raw
        cur = model.state_dict()
        if set(state.keys()) != set(cur.keys()):
            miss = list(set(cur) - set(state))[:4]; unexp = list(set(state) - set(cur))[:4]
            raise RuntimeError(f"key mismatch missing={miss} unexpected={unexp}")
        for k in state:
            if state[k].shape != cur[k].shape:
                raise RuntimeError(f"shape mismatch '{k}': {tuple(state[k].shape)} vs {tuple(cur[k].shape)}")
        return True
    except Exception as e:
        print(f"  {os.path.basename(ckpt_path)} incompatible with current architecture ({e}) -- retraining.")
        return False

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

print("Reproducibility utilities defined.")

## 05 — DRTiD integrity, splits & exploratory statistics (Gate 1)

DRTiD ships an **official** train/test split, used as-is. We only carve train/val out of the
official training rows. `_1` = Macula, `_2` = Optic disc — confirmed against the CrossFiT
reference loader (DRTiD's own benchmark authors' code).

**Scope note (verified, not assumed):** every `ID` in DRTiD's ground truth appears exactly once and
none carries both an `L` and `R` row, so `ID` is a per-**eye** identifier with no patient linkage
exposed. Splits and bootstrap clustering group by `ID` because it is the finest key the data
provides — that is eye-wise, *not* verified patient-wise. Reported as a limitation, not papered over.

In [ ]:
from sklearn.model_selection import train_test_split

def make_drtid_splits(seed=42, val_fraction=0.2, force=False):
    out = {k: f"{SPLITS_DIR}/drtid_{k}.csv" for k in ("train", "val", "test")}
    images_dir = f"{DRTID_ROOT}/Original Images"
    off_train = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/a. DR_grade_Training.csv")
    off_test  = pd.read_csv(f"{DRTID_ROOT}/Ground Truths/DR_grade/b. DR_grade_Testing.csv")

    overlap = set(off_train["ID"]) & set(off_test["ID"])
    assert not overlap, f"Gate 1 FAILED: official train/test share IDs: {sorted(overlap)[:10]}"

    def std(df):
        # NOTE: DRTiD's public ground truth exposes `ID` only. Every ID occurs exactly once and none
        # carries both an L and R row, and the official CrossFiT loader does not treat it as a patient
        # key (it reads Grade/Macula/Optic disc and leaves ID commented out). We therefore call it
        # record_id, NOT patient_id, and all clustering built on it is EYE/RECORD-level -- never
        # described as patient-level in the paper.
        return pd.DataFrame({
            "record_id": df["ID"],
            "macula_path": df["Macula"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "disc_path":   df["Optic disc"].apply(lambda s: f"{images_dir}/{s}.jpg"),
            "grade":       df["Grade"],
            "laterality":  df["LR"],
        })

    if not force and all(os.path.exists(p) for p in out.values()):
        print("Splits already exist on Drive -- reusing (guarantees identical splits across sessions).")
    else:
        # STRATIFIED by grade. Grade 4 is only ~3.9% of the official training rows, so an unstratified
        # random split makes the validation grade composition highly seed-sensitive -- and validation
        # is what every model-selection decision rests on.
        tr_ids, va_ids = train_test_split(off_train["ID"].values, test_size=val_fraction,
                                          random_state=seed, stratify=off_train["Grade"].values)
        std(off_train[off_train["ID"].isin(tr_ids)]).to_csv(out["train"], index=False)
        std(off_train[off_train["ID"].isin(va_ids)]).to_csv(out["val"], index=False)
        std(off_test).to_csv(out["test"], index=False)

    dfs = {k: pd.read_csv(v) for k, v in out.items()}
    assert not (set(dfs["train"].record_id) & set(dfs["val"].record_id)), "Gate 1 FAILED: train/val overlap"
    assert not (set(dfs["val"].record_id) & set(dfs["test"].record_id)),  "Gate 1 FAILED: val/test overlap"
    assert not (set(dfs["train"].record_id) & set(dfs["test"].record_id)), "Gate 1 FAILED: train/test overlap"

    rows, ok = [], True
    for name, df in dfs.items():
        missing = [p for c in ("macula_path", "disc_path") for p in df[c] if not os.path.exists(p)]
        if missing:
            ok = False; print(f"  MISSING {len(missing)} images in {name}, e.g. {missing[:3]}")
        dist = df["grade"].value_counts().sort_index()
        absent = sorted(set(range(NUM_CLASSES)) - set(dist.index))
        if absent:
            ok = False; print(f"  {name}: grades {absent} ABSENT")
        rows.append({"split": name, "n_records_eyes": len(df), "n_images": 2 * len(df),
                     **{f"grade_{g}": int(dist.get(g, 0)) for g in range(NUM_CLASSES)}})
    stats = pd.DataFrame(rows)
    stats.to_csv(f"{TABLES_DIR}/table_00_dataset_statistics.csv", index=False)
    print(stats.to_string(index=False))

    manifest = {k: {"path": v, "sha256": sha256_file(v), "rows": len(dfs[k])} for k, v in out.items()}
    save_json(manifest, f"{CONFIG_DIR}/split_manifest.json")
    record_gate("Gate1_Dataset", ok, f"train/val/test = {len(dfs['train'])}/{len(dfs['val'])}/{len(dfs['test'])} eyes; "
                                     f"no ID overlap; all grades present; all images resolve")
    return out["train"], out["val"], out["test"]

DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV = make_drtid_splits()

# The protocol identity: locked config + the exact split files. Every checkpoint saved from here on
# carries it, and RESUME_EXACT will only reuse a checkpoint whose stored hash matches. Key/shape
# compatibility alone was never sufficient -- two runs can produce identically-shaped weights from
# different losses, learning rates, transforms or splits.
PROTOCOL_HASH = compute_protocol_hash()
save_json({"protocol_hash": PROTOCOL_HASH, "run_tag": RUN_TAG, "resume_exact": RESUME_EXACT,
           "config": CONFIG_SNAPSHOT,
           "split_sha256": {os.path.basename(p): sha256_file(p)
                            for p in (DRTID_TRAIN_CSV, DRTID_VAL_CSV, DRTID_TEST_CSV)}},
          f"{CONFIG_DIR}/protocol_hash.json")
print(f"PROTOCOL_HASH : {PROTOCOL_HASH}")
print(f"  checkpoint reuse is {'ENABLED for matching-hash checkpoints' if RESUME_EXACT else 'DISABLED (fresh run)'}")

## 06 — Preprocessing & augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# DRTiD channel stats from the CrossFiT authors' own loader -- keeps preprocessing aligned with
# the benchmark this work is positioned against.
DRTID_MEAN, DRTID_STD = [0.372487, 0.217266, 0.119367], [0.281526, 0.179457, 0.109162]
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_transforms(train, mean, std, geometric=True, photometric=True):
    # Horizontal flip deliberately OMITTED: it risks changing macula/disc laterality semantics, and
    # the CrossFiT reference implementation has its flip code commented out for the same reason.
    ops = [A.Resize(IMG_SIZE, IMG_SIZE)]
    if train and geometric:
        ops += [A.Rotate(limit=15, p=0.7)]
    if train and photometric:
        ops += [A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5)]
    ops += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(ops)

class PairedDualViewTransform:
    """Applies the SAME geometric transform to both fields, with independent photometric jitter.

    The two fields of one eye share an acquisition geometry, and CrossFiT's own loader applies its
    geometric ops jointly to the pair while keeping colour jitter separate. Rotating macula and disc
    by different random angles injects a spurious geometric discrepancy into exactly the quantity CSD
    is trying to learn (the difference between the two views), so geometry is shared here.
    """
    def __init__(self, mean, std, train=True):
        self.train = train
        self.geo = A.ReplayCompose([A.Resize(IMG_SIZE, IMG_SIZE)] +
                                   ([A.Rotate(limit=15, p=0.7)] if train else []))
        self.photo = A.Compose(([A.RandomBrightnessContrast(brightness_limit=0.15,
                                                            contrast_limit=0.15, p=0.5)] if train else []) +
                               [A.Normalize(mean=mean, std=std), ToTensorV2()])

    def __call__(self, img_macula, img_disc):
        g = self.geo(image=img_macula)                       # sample geometry once...
        a = g["image"]
        b = A.ReplayCompose.replay(g["replay"], image=img_disc)["image"]   # ...and replay it
        return self.photo(image=a)["image"], self.photo(image=b)["image"]

train_transform = build_transforms(True,  DRTID_MEAN, DRTID_STD)
eval_transform  = build_transforms(False, DRTID_MEAN, DRTID_STD)
paired_train_transform = PairedDualViewTransform(DRTID_MEAN, DRTID_STD, train=True)
aptos_train_transform = build_transforms(True,  IMAGENET_MEAN, IMAGENET_STD)
aptos_eval_transform  = build_transforms(False, IMAGENET_MEAN, IMAGENET_STD)

PREPROCESSING_META = {"input_size": [IMG_SIZE, IMG_SIZE], "normalization_mean": DRTID_MEAN,
                      "normalization_std": DRTID_STD, "horizontal_flip": False,
                      "views": ["macula", "optic_disc"], "ordinal_threshold": 0.5}

def _rgb(path): return np.array(Image.open(path).convert("RGB"))

class DRTiDDualViewDataset(Dataset):
    def __init__(self, split_csv, transform=None, paired_transform=None):
        self.df = pd.read_csv(split_csv)
        self.transform = transform if transform is not None else eval_transform
        self.paired_transform = paired_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        if self.paired_transform is not None:
            a, b = self.paired_transform(_rgb(r["macula_path"]), _rgb(r["disc_path"]))
            return {"macula": a, "disc": b,
                    "label": torch.tensor(int(r["grade"]), dtype=torch.long),
                    "cluster_id": int(r["record_id"])}
        return {"macula": self.transform(image=_rgb(r["macula_path"]))["image"],
                "disc":   self.transform(image=_rgb(r["disc_path"]))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["record_id"])}   # eye/record level -- see Gate 1 note

class APTOSSingleViewDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path); self.root_dir = root_dir
        self.transform = transform if transform is not None else aptos_eval_transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = _rgb("{}/{}.png".format(self.root_dir, r["id_code"]))
        return {"image": self.transform(image=img)["image"],
                "label": torch.tensor(int(r["diagnosis"]), dtype=torch.long)}

def make_loader(ds, batch_size, shuffle, seed=None, workers=None):
    """Single place that decides worker count. `workers=None` -> DEFAULT_NUM_WORKERS (0), which is
    what removes the 210 "can only test a child process" teardown assertions the first preflight
    produced. Callers may still pass an explicit number, but nothing in this notebook does."""
    workers = DEFAULT_NUM_WORKERS if workers is None else workers
    kw = {}
    if seed is not None:
        kw = {"worker_init_fn": seed_worker, "generator": make_generator(seed)}
    if workers > 0:
        kw.update({"persistent_workers": True, "prefetch_factor": 2})
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=workers, **kw)

# ---- Persistent evaluation loaders (RUNTIME BLOCKER FIX) ----
# Validation is read by Gate 2, Gate 4, every grid search and model selection. The earlier code
# created a throwaway validation loader inside the teacher cell and deleted it at the end of that
# cell; the logit-KD grid then called run_grid(), which still referenced that name, and the run died
# with NameError before a single grid point finished. One loader, defined here, never deleted.
VAL_DS     = DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform)
VAL_LOADER = make_loader(VAL_DS, 16, False)

print(f"Preprocessing defined. Persistent VAL_LOADER over {len(VAL_DS)} validation eyes.")

## 07–09 — Model architecture, CORAL initialization & unit tests

**CORAL thresholds are initialized from the empirical marginal** `b_k = logit(P(Y>k))`. rev2
initialized all four thresholds within 0.15 logits of each other while DRTiD needs a 3.24-logit
spread; because CORAL gives each sample one scalar score compared against all thresholds, collapsed
thresholds make intermediate grades unreachable — measured rev2 sensitivity for Grades 1–3 was
0.00–0.04 across every condition including the teacher. Kept from rev3, with assertions.

In [ ]:
import torchvision.models as tv

def compute_pos_weights(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", mode=POS_WEIGHT_MODE):
    """pos_weight_k = N_neg/N_pos. Raw ratio is 24.8x at k=3 on DRTiD (31/800 eyes are Grade 4),
    which drove rev2's collapse onto the extreme grades. 'sqrt' keeps the correction's direction
    without its degeneracy."""
    g = pd.read_csv(train_csv)[grade_col].values
    w = []
    for k in range(num_thresholds):
        pos, neg = int((g > k).sum()), int((g <= k).sum())
        if pos == 0 or neg == 0:
            raise ValueError(f"degenerate threshold k={k}: pos={pos} neg={neg}")
        r = neg / pos
        w.append({"full": r, "sqrt": math.sqrt(r), "none": 1.0}[mode])
    return torch.tensor(w, dtype=torch.float32)

def compute_init_thresholds(train_csv, num_thresholds=NUM_THRESHOLDS, grade_col="grade", eps=1e-3):
    g = pd.read_csv(train_csv)[grade_col].values
    return [math.log(min(max(float((g > k).mean()), eps), 1 - eps) /
                     (1 - min(max(float((g > k).mean()), eps), 1 - eps))) for k in range(num_thresholds)]


class CORALHead(nn.Module):
    """Monotone cumulative outputs P(y>k) by construction (ordered non-negative softplus steps)."""
    def __init__(self, in_dim, num_classes=NUM_CLASSES, init_thresholds=None):
        super().__init__()
        self.num_thresholds = num_classes - 1
        self.fc = nn.Linear(in_dim, 1, bias=False)
        if init_thresholds is None:
            init_thresholds = [-0.7 * i for i in range(self.num_thresholds)]
        t = torch.tensor(list(init_thresholds), dtype=torch.float32)
        if t.numel() != self.num_thresholds:
            raise ValueError(f"need {self.num_thresholds} thresholds, got {t.numel()}")
        gaps = (t[:-1] - t[1:]).clamp_min(1e-4)
        self.base_bias  = nn.Parameter(t[0].clone())
        self.bias_steps = nn.Parameter(torch.log(torch.expm1(gaps)).clone())

    def _ordered_biases(self):
        steps = F.softplus(self.bias_steps)
        cum = torch.cat([torch.zeros(1, device=steps.device), torch.cumsum(steps, dim=0)])
        return self.base_bias - cum

    def forward(self, z):
        logits = self.fc(z) + self._ordered_biases().unsqueeze(0)
        return logits, torch.sigmoid(logits)


class InteractionFusion(nn.Module):
    """Concat + |diff| + product through a small MLP (judge.md Flag 2: a bare linear fusion can
    only form a weighted sum and cannot represent cross-view interaction). LayerNorm rather than
    BatchNorm removes batch-size sensitivity at the small batch sizes used here.

    All submodules are defined unconditionally: TorchScript/export statically analyses every branch,
    and rev2's Gate 5 failed with "has no attribute 'norm'" because submodules were created only
    inside one branch of an if."""
    def __init__(self, feat_dim, fusion_type=FUSION_TYPE, hidden_dim=None):
        super().__init__()
        if fusion_type not in ("linear", "interaction_mlp"):
            raise ValueError(fusion_type)
        self.fusion_type = fusion_type
        hidden_dim = hidden_dim or feat_dim
        self.norm     = nn.LayerNorm(feat_dim * 2)
        self.norm_in  = nn.LayerNorm(feat_dim * 4)
        self.proj     = nn.Linear(feat_dim * 4, hidden_dim)
        self.act      = nn.ReLU(inplace=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.out_dim  = feat_dim * 2 if fusion_type == "linear" else hidden_dim

    def forward(self, z_m, z_d):
        if self.fusion_type == "linear":
            return self.norm(torch.cat([z_m, z_d], dim=1))
        combined = self.norm_in(torch.cat([z_m, z_d, torch.abs(z_m - z_d), z_m * z_d], dim=1))
        return self.norm_out(self.act(self.proj(combined)))


class DepthwiseSeparableBlock(nn.Module):
    """ReLU (not ReLU6): eager-mode fuse_modules has no fuser for Conv-BN-ReLU6."""
    def __init__(self, i, o, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(i, i, 3, stride=stride, padding=1, groups=i, bias=False)
        self.bn1 = nn.BatchNorm2d(i); self.act1 = nn.ReLU(inplace=True)
        self.pw = nn.Conv2d(i, o, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(o); self.act2 = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act2(self.bn2(self.pw(self.act1(self.bn1(self.dw(x))))))
    def fuse(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["dw", "bn1", "act1"], ["pw", "bn2", "act2"]], inplace=True)


class LightweightBackbone(nn.Module):
    """~125K-param feature extractor. rev2's was 8,176 params (its fusion MLP was 75% of the whole
    34K model), ~40x below the technical doc's 0.3-0.4M target, which capacity-capped every
    dual-view condition at the same QWK and made RQ1 untestable."""
    def __init__(self, channels=None):
        super().__init__()
        ch = tuple(channels or STUDENT_CHANNELS)
        self.stem_conv = nn.Conv2d(3, ch[0], 3, stride=2, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(ch[0]); self.stem_act = nn.ReLU(inplace=True)
        strides = [2 if i % 2 == 0 else 1 for i in range(len(ch) - 1)]
        self.blocks = nn.ModuleList([DepthwiseSeparableBlock(ch[i], ch[i+1], strides[i])
                                     for i in range(len(ch) - 1)])
        self.gap = nn.AdaptiveAvgPool2d(1); self.out_dim = ch[-1]
    def forward(self, x):
        x = self.stem_act(self.stem_bn(self.stem_conv(x)))
        for b in self.blocks: x = b(x)
        return self.gap(x).flatten(1)
    def fuse_model(self, qat=False):
        fn = torch.ao.quantization.fuse_modules_qat if qat else torch.ao.quantization.fuse_modules
        fn(self, [["stem_conv", "stem_bn", "stem_act"]], inplace=True)
        for b in self.blocks: b.fuse(qat=qat)


class _DualViewBase(nn.Module):
    def forward(self, macula, disc):
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        z_f = self.fusion(z_m, z_d)
        ld, pd_ = self.main_head(z_f)
        lm, pm = self.macula_head(z_m)
        ldd, pdd = self.disc_head(z_d)
        return {"p_dual": pd_, "logit_dual": ld, "p_macula": pm, "logit_macula": lm,
                "p_disc": pdd, "logit_disc": ldd, "z_fused": z_f}
    def forward_single(self, x, which="macula"):
        z = self.backbone(x)
        logit, p = (self.macula_head if which == "macula" else self.disc_head)(z)
        return {"logit": logit, "p": p}
    def counterfactual_forward(self, macula, disc):
        """Same-head counterfactual (judge.md Flag 1/3): dual / macula-only / disc-only all go
        through the SAME main_head, so their difference cannot be head discrepancy."""
        z_m, z_d = self.backbone(macula), self.backbone(disc)
        zero = torch.zeros_like(z_m)
        _, p_dual = self.main_head(self.fusion(z_m, z_d))
        _, p_m    = self.main_head(self.fusion(z_m, zero))
        _, p_d    = self.main_head(self.fusion(zero, z_d))
        return {"p_dual": p_dual, "p_macula_cf": p_m, "p_disc_cf": p_d}


class DualViewResNetTeacher(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=2048, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        bb = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); bb.fc = nn.Identity()
        self.backbone = bb
        self.fusion = InteractionFusion(feat_dim, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(feat_dim, num_classes, init_thresholds)
        self.disc_head   = CORALHead(feat_dim, num_classes, init_thresholds)


class DualViewLightStudent(_DualViewBase):
    def __init__(self, num_classes=NUM_CLASSES, backbone=None, fusion_type=FUSION_TYPE, init_thresholds=None):
        super().__init__()
        self.backbone = backbone or LightweightBackbone()
        fd = self.backbone.out_dim
        self.fusion = InteractionFusion(fd, fusion_type)
        self.main_head   = CORALHead(self.fusion.out_dim, num_classes, init_thresholds)
        self.macula_head = CORALHead(fd, num_classes, init_thresholds)
        self.disc_head   = CORALHead(fd, num_classes, init_thresholds)
    def fuse_model(self, qat=False):
        if hasattr(self.backbone, "fuse_model"): self.backbone.fuse_model(qat=qat)

INIT_THRESHOLDS = compute_init_thresholds(DRTID_TRAIN_CSV)
POS_WEIGHT = compute_pos_weights(DRTID_TRAIN_CSV)
print("CORAL init thresholds :", [round(t, 4) for t in INIT_THRESHOLDS])
print("  implied P(y>k)      :", [round(float(torch.sigmoid(torch.tensor(t))), 4) for t in INIT_THRESHOLDS])
print(f"pos_weight ({POS_WEIGHT_MODE:4s})     :", [round(float(w), 3) for w in POS_WEIGHT])

In [ ]:
# ---- Unit tests on the ordinal head (fail loudly, before any training) ----
def test_coral_head():
    h = CORALHead(16, NUM_CLASSES, INIT_THRESHOLDS)
    b = h._ordered_biases().detach()
    assert torch.all(b[:-1] >= b[1:]), "thresholds must be non-increasing"
    spread = float(b[0] - b[-1])
    assert spread > 1.5, f"threshold spread {spread:.3f} too small -- predictions will collapse to extremes"
    _, p = h(torch.randn(32, 16))
    assert torch.all(p[:, :-1] >= p[:, 1:] - 1e-6), "P(y>k) must be non-increasing in k"
    emp = [float(torch.sigmoid(torch.tensor(t))) for t in INIT_THRESHOLDS]
    got = [float(torch.sigmoid(x)) for x in b]
    assert max(abs(a - c) for a, c in zip(emp, got)) < 1e-5, "init must reproduce empirical marginals"
    print(f"  CORAL unit tests PASSED (spread={spread:.3f} logits, monotone, matches marginals)")
    return spread

_spread = test_coral_head()

def test_fusion_interaction():
    f = InteractionFusion(8, "interaction_mlp").eval()
    a, b = torch.randn(4, 8), torch.randn(4, 8)
    assert not torch.allclose(f(a, b), f(b, a)), "fusion must not be order-invariant (it models interaction)"
    print("  InteractionFusion unit test PASSED (view-order sensitive => genuine interaction)")

test_fusion_interaction()
record_gate("Gate_CORAL_UnitTests", True, f"threshold spread {_spread:.3f} logits; monotone; matches marginals")

## 10 — Loss definitions

`L = L_task + λ·L_aux + α·L_logitKD + β·L_CSD (+ γ·L_featKD)`

**Fixed-teacher scope.** Every student seed distils from the SAME teacher checkpoint, so
seed-to-seed variability is measured *conditional on a fixed teacher*. The paper must say so:
"student variability is evaluated conditional on a fixed teacher checkpoint". Crossing teacher seeds
with student seeds would be a different (much larger) experiment and is deliberately out of scope.

**What Δ is, and is not.** `Δ` is computed through three separate heads, so besides genuine
complementarity it can also carry head discrepancy and calibration discrepancy. It is therefore an
**operational proxy of the dual-view decision shift** — the wording the paper must use. The same-head
counterfactual (`counterfactual_forward`, ablation `abl_csd_counterfactual`) routes dual / macula /
disc through the *same* `main_head` and bounds how much of Δ could be head discrepancy.

**CSD (normalized).** `Δ = p_dual − (p_macula+p_disc)/2` for teacher and student; both are divided
by `s = mean(|Δ^T|)` (detached) before a Huber loss. Because the divisor is detached and identical
on both sides, the optimum is unchanged but the gradient becomes usable: rev2 logged `L_CSD≈0.014`
against `L_task≈0.82` (<0.5% of the objective, essentially no gradient), which is why its RQ1 test
could not have detected any CSD effect.

In [ ]:
def coral_loss(logits, labels, num_thresholds=NUM_THRESHOLDS, pos_weight=None):
    levels = torch.arange(num_thresholds, device=logits.device).unsqueeze(0)
    y_k = (labels.unsqueeze(1) > levels).float()
    return F.binary_cross_entropy_with_logits(logits, y_k, pos_weight=pos_weight)

def aux_loss(out, labels, pos_weight=None):
    return (coral_loss(out["logit_macula"], labels, pos_weight=pos_weight) +
            coral_loss(out["logit_disc"],   labels, pos_weight=pos_weight))

def logit_kd_loss(logit_t, logit_s, tau=2.0):
    """No tau^2 factor (judge.md Flag 17): alpha and tau are therefore coupled -- do not claim
    independent temperature tuning in the paper."""
    return F.binary_cross_entropy(torch.sigmoid(logit_s / tau), torch.sigmoid(logit_t.detach() / tau))

def _delta(p_dual, p_m, p_d):
    return p_dual - (p_m + p_d) / 2

def csd_loss(p_dual_t, p_m_t, p_d_t, p_dual_s, p_m_s, p_d_s,
             variant="smoothl1_norm", tau_csd=0.5, huber_beta=1.0, eps=1e-6, scale=None):
    """Complementarity-Shift Distillation.

    NORMALIZATION (corrected): the scale `s` is a FIXED GLOBAL constant estimated once from the
    frozen teacher over the training split, not `mean(|delta_T|)` of the current batch. A per-batch
    divisor would amplify batches whose teacher shift happens to be small and shrink batches whose
    shift is large -- that is not a pure rescaling, it silently re-weights samples relative to each
    other. A single fixed scalar fixes the gradient magnitude problem while leaving the relative
    magnitude structure across samples and batches untouched.

    Variants
      smoothl1_norm                -- DEFAULT, globally-scaled signed Huber on delta
      smoothl1                     -- unscaled (rev2 formulation), ablation only
      magnitude_weighted_direction -- magnitude-weighted cosine + scaled magnitude term
      kl_softmax                   -- v1 formulation, negative control (destroys magnitude info)
    """
    dt = _delta(p_dual_t.detach(), p_m_t.detach(), p_d_t.detach())
    ds = _delta(p_dual_s, p_m_s, p_d_s)
    s_glob = float(scale) if scale is not None else float(globals().get("CSD_GLOBAL_SCALE", 1.0))
    s_glob = max(s_glob, 1e-3)

    if variant == "smoothl1_norm":
        return F.smooth_l1_loss(ds / s_glob, dt / s_glob, beta=huber_beta)
    if variant == "smoothl1":
        return F.smooth_l1_loss(ds, dt, beta=huber_beta)
    if variant == "magnitude_weighted_direction":
        mag = dt.norm(dim=1)
        w = (mag / mag.median().clamp_min(eps)).clamp(max=1.0)
        l_dir = ((1 - F.cosine_similarity(ds, dt, dim=1, eps=eps)) * w).sum() / w.sum().clamp_min(eps)
        return 0.5 * l_dir + 0.5 * F.smooth_l1_loss(ds / s_glob, dt / s_glob, beta=huber_beta)
    if variant == "kl_softmax":
        return F.kl_div(F.log_softmax(ds / tau_csd, dim=1), F.softmax(dt / tau_csd, dim=1), reduction="batchmean")
    raise ValueError(f"unknown csd_variant: {variant}")


@torch.no_grad()
def compute_global_delta_scale(teacher, loader, device, counterfactual=False):
    """E_train[|delta_T|] from the FROZEN teacher -- computed once, then held fixed for all training."""
    teacher.eval(); tot, n = 0.0, 0
    for b in loader:
        m, d = b["macula"].to(device), b["disc"].to(device)
        if counterfactual:
            o = teacher.counterfactual_forward(m, d)
            dt = _delta(o["p_dual"], o["p_macula_cf"], o["p_disc_cf"])
        else:
            o = teacher(m, d)
            dt = _delta(o["p_dual"], o["p_macula"], o["p_disc"])
        tot += float(dt.abs().sum()); n += dt.numel()
    return max(tot / max(n, 1), 1e-3)


def feature_kd_loss(z_t, z_s, projector):
    """Representation-level control.

    CORRECTED DIRECTION: the projector maps STUDENT -> TEACHER space and the teacher features are
    detached, so the regression target is FIXED. Projecting teacher->student with a trainable
    projector (the earlier form) lets the target drift as the projector learns, which makes this a
    weaker control than it appears -- and this is the primary control for CSD's novelty claim, so it
    has to be clean.
    """
    return F.mse_loss(projector(z_s), z_t.detach())


def get_student_output(student, macula, disc, view_mode):
    if view_mode == "dual":        return student(macula, disc)
    if view_mode == "macula_only": return student.forward_single(macula, "macula")
    if view_mode == "disc_only":   return student.forward_single(disc, "disc")
    raise ValueError(view_mode)

def ordinal_violation_rate(p):
    return float((p[:, 1:] - p[:, :-1] > 0).float().mean())

def combined_student_loss(teacher_out, student_out, labels, view_mode, alpha=0.0, beta=0.0,
                          lambda_aux=0.5, tau_kd=2.0, csd_variant="smoothl1_norm", tau_csd=0.5,
                          pos_weight=None, use_counterfactual_csd=False, teacher_cf_out=None,
                          student_cf_out=None, gamma_feat=0.0, feat_projector=None, huber_beta=1.0,
                          csd_scale=None):
    task_logit = student_out["logit_dual"] if view_mode == "dual" else student_out["logit"]
    l_task = coral_loss(task_logit, labels, pos_weight=pos_weight)
    total, log, comps = l_task, {"L_task": l_task.detach().item()}, {"task": l_task}

    if view_mode == "dual":
        l_aux = aux_loss(student_out, labels, pos_weight=pos_weight)
        total = total + lambda_aux * l_aux
        # `.detach().item()` rather than `float(tensor)`: the weighted terms still carry grad_fn, and
        # float() on a requires_grad tensor emits "Converting a tensor with requires_grad=True to a
        # scalar may lead to unexpected behavior" on every batch. Logging must never touch autograd.
        log["L_aux"] = l_aux.detach().item(); log["W_aux"] = (lambda_aux * l_aux).detach().item()
        comps["aux"] = lambda_aux * l_aux
        if alpha > 0:
            l_kd = logit_kd_loss(teacher_out["logit_dual"], student_out["logit_dual"], tau_kd)
            total = total + alpha * l_kd
            log["L_logit_KD"] = l_kd.detach().item(); log["W_logit_KD"] = (alpha * l_kd).detach().item()
            comps["logit_kd"] = alpha * l_kd
        if beta > 0:
            if use_counterfactual_csd:
                l_csd = csd_loss(teacher_cf_out["p_dual"], teacher_cf_out["p_macula_cf"], teacher_cf_out["p_disc_cf"],
                                 student_cf_out["p_dual"], student_cf_out["p_macula_cf"], student_cf_out["p_disc_cf"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta,
                                 scale=csd_scale if csd_scale is not None else globals().get("CSD_GLOBAL_SCALE_CF"))
            else:
                l_csd = csd_loss(teacher_out["p_dual"], teacher_out["p_macula"], teacher_out["p_disc"],
                                 student_out["p_dual"], student_out["p_macula"], student_out["p_disc"],
                                 variant=csd_variant, tau_csd=tau_csd, huber_beta=huber_beta,
                                 scale=csd_scale)
            total = total + beta * l_csd
            log["L_CSD"] = l_csd.detach().item(); log["W_CSD"] = (beta * l_csd).detach().item()
            comps["csd"] = beta * l_csd
        if gamma_feat > 0 and feat_projector is not None:
            l_f = feature_kd_loss(teacher_out["z_fused"], student_out["z_fused"], feat_projector)
            total = total + gamma_feat * l_f
            log["L_feat_KD"] = l_f.detach().item(); log["W_feat_KD"] = (gamma_feat * l_f).detach().item()
            comps["feat_kd"] = gamma_feat * l_f

    log["L_total"] = total.detach().item()
    return total, log, comps

def component_grad_norms(components, params, prefix="gnorm"):
    """Per-component gradient norms w.r.t. a chosen parameter group.

    Loss VALUES alone cannot establish that a term influences learning -- rev2 logged values only,
    which is why its dead CSD term stayed invisible until the logs were re-read by hand after the
    entire run had finished."""
    out, params = {}, [p for p in params if p.requires_grad]
    if not params: return out
    for name, t in components.items():
        if t is None or not getattr(t, "requires_grad", False): continue
        g = torch.autograd.grad(t, params, retain_graph=True, allow_unused=True)
        out[f"{prefix}_{name}"] = sum(float(x.pow(2).sum()) for x in g if x is not None) ** 0.5
    if out.get(f"{prefix}_task", 0) > 0:
        for k in ("csd", "aux", "logit_kd", "feat_kd"):
            if f"{prefix}_{k}" in out:
                out[f"{prefix}_ratio_{k}_over_task"] = out[f"{prefix}_{k}"] / out[f"{prefix}_task"]
    return out

def gradient_probe_groups(student):
    """Two parameter groups, because ONE group cannot measure every loss term fairly.

    The earlier probe used `fusion + main_head` only. The auxiliary heads do not route through
    either, so `gnorm_aux` was structurally 0.0 -- the first preflight duly printed
    `gnorm_aux: 0.0`, which reads as "the auxiliary loss does nothing" when it actually means "the
    probe cannot see it". The SHARED BACKBONE is the one parameter set every term back-propagates
    into, so it is the apples-to-apples probe; the fusion/main-head group is kept as the
    decision-path view.
    """
    return {"SharedBackbone": list(student.backbone.parameters()),
            "FusionMain": list(student.fusion.parameters()) + list(student.main_head.parameters())}

def probe_all_groups(components, student):
    out = {}
    for gname, params in gradient_probe_groups(student).items():
        out.update(component_grad_norms(components, params, prefix=f"gnorm{gname}"))
    # Back-compatible aliases so existing figures/columns keep resolving; the shared-backbone group
    # is the one that is comparable across ALL loss terms, so it supplies them.
    for k, v in list(out.items()):
        if k.startswith("gnormSharedBackbone_"):
            out["gnorm_" + k[len("gnormSharedBackbone_"):]] = v
        if k.startswith("gnormSharedBackbone_ratio_"):
            out["gnorm_ratio_" + k[len("gnormSharedBackbone_ratio_"):]] = v
    if out.get("gnorm_task", 0) > 0 and "gnorm_csd" in out:
        out["gnorm_ratio_csd_over_task"] = out["gnorm_csd"] / out["gnorm_task"]
    return out

print("Losses defined.")

## 11 — Complete metrics library

**QWK is the single primary metric** (DR grades are ordinal; a 4-grade error is not the same as a
1-grade error). Everything else is reported as secondary/supplementary — comprehensive coverage,
but explicitly not "everything is primary", which would read as metric fishing.

- *Ordinal*: QWK, MAE, Severe-Error Rate, Ordinal Violation Rate
- *Categorical*: Accuracy, Balanced Accuracy, macro/weighted Precision·Recall·F1
- *Per grade*: Precision, Recall (sensitivity), F1, Specificity (one-vs-rest), Support
- *Calibration*: Brier, ECE
- *Mechanism*: ShiftMAE, CosAgree, BenefitCorr, internal & external dual-view gain

In [ ]:
from sklearn.metrics import (cohen_kappa_score, f1_score, precision_score, recall_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix)

def fast_qwk(y_true, y_pred, K=NUM_CLASSES):
    """Vectorized QWK -- the bootstrap calls this ~10,000x per comparison."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    O = np.zeros((K, K)); np.add.at(O, (y_true, y_pred), 1)
    w = (np.arange(K)[:, None] - np.arange(K)[None, :]) ** 2 / (K - 1) ** 2
    ht, hp = np.bincount(y_true, minlength=K), np.bincount(y_pred, minlength=K)
    E = np.outer(ht, hp) / max(len(y_true), 1)
    den = (w * E).sum()
    return 1.0 - (w * O).sum() / den if den > 0 else 0.0

def test_fast_qwk_matches_sklearn(n_trials=100, seed=0):
    """`fast_qwk` is a hand-written vectorized implementation and it is the PRIMARY metric of this
    study, so it is checked against sklearn's `cohen_kappa_score(weights="quadratic")` on random
    inputs plus the edge cases that break naive implementations.

    `labels=range(K)` matters: without it sklearn infers the label set from the data, so a model that
    never predicts grade 4 would be scored on a 4x4 matrix while fast_qwk always uses 5x5.
    """
    from sklearn.metrics import cohen_kappa_score
    rng = np.random.default_rng(seed)
    labels = list(range(NUM_CLASSES))
    worst, cases = 0.0, 0

    def _cmp(yt, yp, name):
        nonlocal worst, cases
        a = fast_qwk(yt, yp)
        b = cohen_kappa_score(yt, yp, weights="quadratic", labels=labels)
        if not np.isfinite(b):          # sklearn returns nan for a degenerate agreement matrix
            return
        d = abs(a - b); worst = max(worst, d); cases += 1
        assert d < 1e-8, f"fast_qwk disagrees with sklearn on {name}: {a} vs {b} (diff {d})"

    for t in range(n_trials):
        n = int(rng.integers(30, 400))
        yt = rng.integers(0, NUM_CLASSES, n)
        yp = np.clip(yt + rng.integers(-2, 3, n), 0, NUM_CLASSES - 1)
        _cmp(yt, yp, f"random trial {t}")

    y = rng.integers(0, NUM_CLASSES, 200)
    _cmp(y, y.copy(), "perfect predictions")
    _cmp(y, (NUM_CLASSES - 1) - y, "reversed predictions")
    _cmp(y, np.zeros_like(y), "single-class predictions")
    _cmp(y, np.clip(y, 0, NUM_CLASSES - 2), "grade 4 never predicted")
    _cmp(np.array([0, 0, 4, 4]), np.array([0, 4, 0, 4]), "tiny balanced case")

    print(f"  QWK reference-equivalence PASSED on {cases} cases (max |fast_qwk - sklearn| = {worst:.2e})")
    return worst

def compute_all_metrics(y_true, y_pred, p_cum=None, K=NUM_CLASSES):
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    labels = list(range(K))
    m = {
        "QWK": fast_qwk(y_true, y_pred, K),
        "MAE": float(np.mean(np.abs(y_true - y_pred))),
        "SevereErrorRate": float(np.mean(np.abs(y_true - y_pred) >= 2)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "BalancedAccuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    for avg in ("macro", "weighted"):
        tag = "Macro" if avg == "macro" else "Weighted"
        m[f"{tag}Precision"] = float(precision_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}Recall"]    = float(recall_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))
        m[f"{tag}F1"]        = float(f1_score(y_true, y_pred, average=avg, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    prec = precision_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    total = cm.sum()
    for g in labels:
        tp = cm[g, g]; fn = cm[g, :].sum() - tp; fp = cm[:, g].sum() - tp; tn = total - tp - fn - fp
        m[f"Precision_Grade{g}"]   = float(prec[g])
        m[f"Recall_Grade{g}"]      = float(rec[g])          # sensitivity
        m[f"Sensitivity_Grade{g}"] = float(rec[g])          # alias kept for continuity with rev2/rev3
        m[f"F1_Grade{g}"]          = float(f1[g])
        m[f"Specificity_Grade{g}"] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        m[f"Support_Grade{g}"]     = int(cm[g, :].sum())
        m[f"Predicted_Grade{g}"]   = int(cm[:, g].sum())
    if p_cum is not None:
        m["OrdinalViolationRate"] = ordinal_violation_rate(p_cum)
        m.update(compute_calibration(p_cum, y_true))
    return m

def compute_calibration(p_cum, y_true, n_bins=10):
    """Pooled over the K-1 cumulative thresholds. CSD is framed as distilling a shift in
    CONFIDENCE, so calibration is load-bearing for the claim (judge.md Flag 4)."""
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    brier = float(((pf - tf) ** 2).mean())
    ece, n, edges = 0.0, pf.numel(), torch.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        if msk.sum() == 0: continue
        ece += float(msk.sum()) / n * abs(float(pf[msk].mean()) - float(tf[msk].mean()))
    # Named precisely: these pool the K-1 CUMULATIVE THRESHOLDS, which is not conventional
    # multiclass ECE/Brier. Per-threshold values are reported alongside.
    out = {"OrdinalThreshold_Brier": brier, "OrdinalThreshold_ECE": ece,
           "Brier": brier, "ECE": ece}   # aliases kept so existing table/figure code still resolves
    # Per-threshold diagnostics, named for what they actually are. The earlier `ECE_threshold{k}`
    # was mean|p - y| -- a mean absolute probability error, NOT an Expected Calibration Error, which
    # requires binning. Both quantities are now reported, each under its correct name.
    for k in range(p.shape[1]):
        pk, tk = p[:, k], t[:, k]
        out[f"Threshold{k}_MAEProb"] = float((pk - tk).abs().mean())
        e_k, n_k = 0.0, pk.numel()
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            msk = (pk > lo) & (pk <= hi) if i > 0 else (pk >= lo) & (pk <= hi)
            if msk.sum() == 0: continue
            e_k += float(msk.sum()) / n_k * abs(float(pk[msk].mean()) - float(tk[msk].mean()))
        out[f"Threshold{k}_ECE"] = e_k
    return out

def reliability_curve(p_cum, y_true, n_bins=10):
    p = p_cum.detach().cpu().float()
    y = torch.as_tensor(np.asarray(y_true, int)).long()
    levels = torch.arange(p.shape[1]).unsqueeze(0)
    t = (y.unsqueeze(1) > levels).float()
    pf, tf = p.flatten(), t.flatten()
    edges = torch.linspace(0, 1, n_bins + 1); rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        msk = (pf > lo) & (pf <= hi) if i > 0 else (pf >= lo) & (pf <= hi)
        rows.append({"bin_lo": float(lo), "bin_hi": float(hi), "count": int(msk.sum()),
                     "mean_predicted": float(pf[msk].mean()) if msk.sum() else float("nan"),
                     "observed_frequency": float(tf[msk].mean()) if msk.sum() else float("nan")})
    return pd.DataFrame(rows)

def check_prediction_collapse(y_true, y_pred, K=NUM_CLASSES, recall_floor=0.05):
    """rev2's core pathology only became visible through per-grade recall -- Grades 1-3 were
    essentially never predicted while overall QWK still looked plausible. This makes that failure
    mode impossible to miss again."""
    y_true = np.asarray(y_true, int); y_pred = np.asarray(y_pred, int)
    warns = []
    for g in range(K):
        if (y_pred == g).sum() == 0:
            warns.append(f"Grade {g} NEVER predicted")
        support = (y_true == g).sum()
        if support > 0:
            r = (y_pred[y_true == g] == g).mean()
            if r < recall_floor:
                warns.append(f"Grade {g} recall {r:.3f} < {recall_floor}")
    n_distinct = len(np.unique(y_pred))
    if n_distinct <= 2:
        warns.append(f"prediction distribution COLLAPSED to {n_distinct} distinct grade(s)")
    return warns

_qwk_max_dev = test_fast_qwk_matches_sklearn()
record_gate("Gate_QWK_ReferenceEquivalence", True,
            f"fast_qwk matches sklearn cohen_kappa_score(weights='quadratic', labels=0..4); "
            f"max deviation {_qwk_max_dev:.2e} over 105 cases incl. perfect/reversed/single-class/"
            "missing-grade edge cases")
print("Metrics library defined.")

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device, view_mode="dual", return_clusters=False):
    model.eval()
    preds, targets, ps, cids = [], [], [], []
    for batch in loader:
        macula, disc = batch["macula"].to(device), batch["disc"].to(device)
        out = get_student_output(model, macula, disc, view_mode)
        p = out["p_dual" if view_mode == "dual" else "p"]
        preds.extend((p > 0.5).sum(dim=1).cpu().tolist())
        targets.extend(batch["label"].tolist())
        ps.append(p.cpu())
        if return_clusters: cids.extend(batch["cluster_id"].tolist())
    y_true, y_pred, p_cum = np.array(targets), np.array(preds), torch.cat(ps, 0)
    return (y_true, y_pred, p_cum, np.array(cids)) if return_clusters else (y_true, y_pred, p_cum)

@torch.no_grad()
def quick_val_qwk(model, loader, device, view_mode="dual"):
    y, yp, _ = get_predictions(model, loader, device, view_mode)
    return fast_qwk(y, yp)

@torch.no_grad()
def compute_dual_view_gain(model, loader, device):
    """INTERNAL gain: dual head vs this model's OWN auxiliary heads (shared, jointly-trained
    backbone). Must not be conflated with external gain -- judge.md Flag 8."""
    y, pd_, _ = get_predictions(model, loader, device, "dual")
    _, pm, _  = get_predictions(model, loader, device, "macula_only")
    _, pdd, _ = get_predictions(model, loader, device, "disc_only")
    qd, qm, qdd = fast_qwk(y, pd_), fast_qwk(y, pm), fast_qwk(y, pdd)
    return {"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
            "DualViewGain_G_internal": qd - max(qm, qdd)}

def compute_external_gain(qwk_dual, qwk_indep_macula, qwk_indep_disc):
    """EXTERNAL gain: vs INDEPENDENTLY trained single-view students."""
    return qwk_dual - max(qwk_indep_macula, qwk_indep_disc)

def _sample_ordinal_nll(p_cum, y, K=NUM_THRESHOLDS):
    levels = torch.arange(K, device=p_cum.device).unsqueeze(0)
    y_k = (y.unsqueeze(1) > levels).float()
    p = p_cum.clamp(1e-6, 1 - 1e-6)
    return -(y_k * torch.log(p) + (1 - y_k) * torch.log(1 - p)).sum(dim=1)

from scipy import stats as sps

@torch.no_grad()
def compute_shift_fidelity(teacher, student, loader, device, counterfactual=False, prefix=""):
    """Did CSD transfer the PATTERN, independently of whether QWK moved? (judge.md Flag 10)

    BenefitCorr is the strongest of the three: it correlates the teacher's and student's per-sample
    fusion benefit B_i = NLL(p_agg) - NLL(p_dual). If complementarity really transferred, the
    student should benefit from dual-view on the SAME samples the teacher does."""
    teacher.eval(); student.eval()
    smae, cos, bt, bs = [], [], [], []
    km, kd = ("p_macula_cf", "p_disc_cf") if counterfactual else ("p_macula", "p_disc")
    for batch in loader:
        m, d = batch["macula"].to(device), batch["disc"].to(device)
        y = batch["label"].to(device)
        if counterfactual:
            # Same-head counterfactual space: dual / macula-only / disc-only all pass through the
            # SAME main_head. The abl_csd_counterfactual model is TRAINED on this delta, so scoring
            # it only in three-head space would measure it in an objective it never optimised.
            to, so = teacher.counterfactual_forward(m, d), student.counterfactual_forward(m, d)
        else:
            to, so = teacher(m, d), student(m, d)
        dt = _delta(to["p_dual"], to[km], to[kd])
        ds = _delta(so["p_dual"], so[km], so[kd])
        smae.append((ds - dt).abs().sum(1).cpu())
        cos.append(F.cosine_similarity(ds, dt, dim=1, eps=1e-6).cpu())
        for out, sink in ((to, bt), (so, bs)):
            p_agg = (out[km] + out[kd]) / 2
            sink.append((_sample_ordinal_nll(p_agg, y) - _sample_ordinal_nll(out["p_dual"], y)).cpu())
    smae, cos = torch.cat(smae), torch.cat(cos)
    a, b = torch.cat(bt).numpy(), torch.cat(bs).numpy()
    # Named precisely: ShiftL1 is the mean L1 NORM of the shift difference; ShiftMAE is the mean
    # absolute error PER THRESHOLD. The earlier code reported the L1 norm under the name "ShiftMAE".
    if a.std() > 1e-8 and b.std() > 1e-8:
        pear = float(sps.pearsonr(a, b)[0]); spear = float(sps.spearmanr(a, b).statistic)
    else:
        pear = spear = float("nan")
    pfx = prefix or ("CF_" if counterfactual else "")
    return {f"{pfx}ShiftL1": float(smae.mean()), f"{pfx}ShiftMAE": float(smae.mean() / NUM_THRESHOLDS),
            f"{pfx}CosAgree": float(cos.mean()), f"{pfx}BenefitCorr": pear,
            f"{pfx}BenefitCorrSpearman": spear,
            f"{pfx}TeacherBenefitPositiveFrac": float((a > 0).mean()),
            f"{pfx}StudentBenefitPositiveFrac": float((b > 0).mean())}

print("Evaluation helpers defined.")

## 12 — Efficiency benchmark suite (standardized protocol)

Latency is always measured on a **CPU copy** of the model — there is no code path that can time a
CUDA model and label it CPU. Protocol is fixed and recorded with every measurement:
`batch=1, warmup=50, runs=500, threads=1`.

Size comparisons use **equivalent deployment artifacts** (FP32 export vs INT8 export), never an
FP32 training checkpoint against an INT8 state_dict.

In [ ]:
try:
    import psutil; _HAS_PSUTIL = True
except Exception:
    _HAS_PSUTIL = False

def param_count(model):
    return int(sum(p.numel() for p in model.parameters()))

def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2) if path and os.path.exists(path) else float("nan")

def benchmark_latency(model, macula, disc, view_mode="dual", bench=BENCH, on_cpu=True, label=""):
    """Median-of-medians over independent repetitions.

    Colab's CPU is shared, so one 500-inference block can be biased by whatever else is running on
    the host. `repeats` independent blocks are timed and the reported median is the median of the
    per-block medians; the IQR *between* blocks quantifies how stable the measurement itself was.
    """
    torch.set_num_threads(bench["threads"])
    if on_cpu:
        m = copy.deepcopy(model).to("cpu").eval()
        mac, dsc = macula[:bench["batch_size"]].cpu(), disc[:bench["batch_size"]].cpu()
    else:
        m = model.eval(); mac, dsc = macula[:bench["batch_size"]], disc[:bench["batch_size"]]

    if view_mode == "dual":
        fn = lambda: m(mac, dsc)
    else:
        which = "macula" if "macula" in view_mode else "disc"
        img = mac if which == "macula" else dsc
        fn = lambda: m.forward_single(img, which=which)

    reps = int(bench.get("repeats", 1))
    block_medians, all_ts = [], []
    with torch.no_grad():
        for _ in range(bench["warmup"]): fn()
        for _r in range(reps):
            ts = []
            for _ in range(bench["runs"]):
                t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1000)
            ts = np.array(ts); all_ts.append(ts); block_medians.append(float(np.median(ts)))
    ts = np.concatenate(all_ts)
    bm = np.array(block_medians)
    med = float(np.median(bm))                      # median-of-medians
    return {"Latency_mean_ms": float(ts.mean()), "Latency_median_ms": med,
            "Latency_median_of_medians_ms": med,
            "Latency_block_median_IQR_ms": float(np.percentile(bm, 75) - np.percentile(bm, 25)),
            "Latency_block_median_min_ms": float(bm.min()), "Latency_block_median_max_ms": float(bm.max()),
            "Latency_sd_ms": float(ts.std()), "Latency_p95_ms": float(np.percentile(ts, 95)),
            "Latency_p99_ms": float(np.percentile(ts, 99)),
            # One inference call processes one EYE = one (macula, disc) PAIR = TWO fundus images.
            "Throughput_pairs_per_s": float(1000.0 / med) if med > 0 else float("nan"),
            "Throughput_images_per_s": float(2000.0 / med) if med > 0 else float("nan"),
            "bench_device": "cpu" if on_cpu else "cuda", "bench_threads": bench["threads"],
            "bench_warmup": bench["warmup"], "bench_runs": bench["runs"],
            "bench_repeats": reps, "bench_total_inferences": int(reps * bench["runs"]),
            "bench_batch_size": bench["batch_size"]}

def measure_memory(build_fn, macula, disc, n_inference=20):
    """Process RSS around model construction and inference, with a genuinely polled peak.

    The earlier version called the RSS reading taken *after* one inference `PeakRSS_MB`, which is a
    final sample, not a peak. Names now say what each number is, and a sampler thread polls RSS
    during a short inference loop so `PeakRSS_during_inference_MB` really is a maximum.

    Caveat kept in the name: this is PROCESS-level RSS, so it includes everything else the notebook
    holds. It is a deployment-relevant order-of-magnitude, not an isolated model footprint.
    """
    if not _HAS_PSUTIL:
        return {"ModelLoadRSSDelta_MB": float("nan"), "InferenceRSSDelta_MB": float("nan"),
                "RSS_AfterInference_MB": float("nan"), "PeakRSS_during_inference_MB": float("nan")}
    import threading
    proc = psutil.Process(os.getpid())
    base = proc.memory_info().rss / 1024 ** 2
    m = build_fn()
    loaded = proc.memory_info().rss / 1024 ** 2

    peak = [loaded]; stop = threading.Event()
    def _poll():
        while not stop.is_set():
            try: peak[0] = max(peak[0], proc.memory_info().rss / 1024 ** 2)
            except Exception: pass
            time.sleep(0.005)
    th = threading.Thread(target=_poll, daemon=True); th.start()
    try:
        with torch.no_grad():
            for _ in range(n_inference):
                m(macula[:1].cpu(), disc[:1].cpu())
    finally:
        stop.set(); th.join(timeout=1.0)
    after = proc.memory_info().rss / 1024 ** 2
    del m
    return {"ModelLoadRSSDelta_MB": max(loaded - base, 0.0),
            "InferenceRSSDelta_MB": max(after - loaded, 0.0),
            "RSS_AfterInference_MB": after,
            "PeakRSS_during_inference_MB": max(peak[0], after),
            "RSS_measurement_scope": "process-level (includes the notebook), not an isolated model"}

def efficiency_derived(size_mb, ref_size_mb, latency_ms, ref_latency_ms):
    out = {}
    if ref_size_mb and size_mb and not math.isnan(size_mb) and not math.isnan(ref_size_mb) and size_mb > 0:
        out["CompressionRatio_vs_ref"] = ref_size_mb / size_mb
        out["SizeReduction_pct"] = (1 - size_mb / ref_size_mb) * 100
    if ref_latency_ms and latency_ms and latency_ms > 0:
        out["Speedup_vs_ref"] = ref_latency_ms / latency_ms
    return out

def retention_metrics(m_comp, m_ref, keys=("QWK", "Accuracy", "MacroF1", "MacroRecall")):
    """Retention for score-type metrics; for error-type metrics report the DELTA instead, since a
    ratio of errors is not interpretable in the same direction."""
    out = {}
    for k in keys:
        if k in m_comp and k in m_ref and m_ref[k] not in (0, None) and not math.isnan(m_ref[k]):
            out[f"{k}_retention_pct"] = 100.0 * m_comp[k] / m_ref[k]
    for k in ("MAE", "SevereErrorRate", "ECE", "Brier"):
        if k in m_comp and k in m_ref:
            out[f"delta_{k}"] = m_comp[k] - m_ref[k]
    return out

BENCH_EFF = BENCH_PREFLIGHT if PREFLIGHT else BENCH
CPU_INFO = platform.processor() or "unknown"
try:
    CPU_INFO = subprocess.check_output("lscpu | grep 'Model name' | head -1", shell=True, text=True).split(":")[-1].strip() or CPU_INFO
except Exception:
    pass
print("Benchmark CPU:", CPU_INFO, "| protocol:", BENCH)

In [ ]:
# ---- Shared evaluation helpers ----
# These are pure helpers with no dependency on any results table, and several sections need them:
# the VALIDATION-side RQ2 table (which now runs BEFORE the test evaluation, because the deployment
# decision may not see test data), the test evaluation itself, the statistics, and the figures.
# They are defined once here so no section can be reordered into a forward reference.

@torch.no_grad()
def _predict_cpu_all(model, loader):
    """ONE forward pass yields the dual head AND both auxiliary heads.

    The earlier version called `model.forward_single(...)` for the auxiliary views. A model produced
    by `torch.export` is a GraphModule carrying only the traced `forward()` -- it has no
    `forward_single` attribute at all, so the internal dual-view gain would have raised
    AttributeError the moment the PT2E path succeeded. `forward()` already returns p_macula and
    p_disc, so reading them from the same pass is both export-safe and three times cheaper.
    """
    yt, cids, pd_, pm_, pdd_ = [], [], [], [], []
    aux_ok = True
    for b in loader:
        o = model(b["macula"].cpu(), b["disc"].cpu())
        if isinstance(o, dict):
            pd_.append(o["p_dual"])
            if "p_macula" in o and "p_disc" in o:
                pm_.append(o["p_macula"]); pdd_.append(o["p_disc"])
            else:
                aux_ok = False
        else:
            pd_.append(o); aux_ok = False
        yt.extend(b["label"].tolist()); cids.extend(b["cluster_id"].tolist())
    p_dual = torch.cat(pd_, 0)
    out = {"y_true": np.array(yt), "cluster_ids": np.array(cids), "p_dual": p_dual,
           "y_pred_dual": (p_dual > 0.5).sum(1).numpy()}
    if aux_ok:
        pm, pdd = torch.cat(pm_, 0), torch.cat(pdd_, 0)
        out.update({"p_macula": pm, "p_disc": pdd,
                    "y_pred_macula": (pm > 0.5).sum(1).numpy(),
                    "y_pred_disc": (pdd > 0.5).sum(1).numpy()})
    return out

def _predict_cpu(model, loader, view_mode="dual"):
    """Thin compatibility wrapper over _predict_cpu_all."""
    r = _predict_cpu_all(model, loader)
    key = {"dual": "dual", "macula_only": "macula", "disc_only": "disc"}[view_mode]
    if f"y_pred_{key}" not in r:
        raise RuntimeError(f"model exposes no head output for view_mode={view_mode}")
    return r["y_true"], r[f"y_pred_{key}"], r[f"p_{key}"], r["cluster_ids"]

def serialized_state_dict_size_mb(model, tag):
    """Apples-to-apples model size: ALWAYS a freshly serialized state_dict, FP32 and INT8 alike.

    Previously FP32 rows reported `file_size_mb(training_checkpoint)` -- a file that also carries
    epoch, val_qwk, seed and config -- while INT8 rows reported a pure state_dict. The headline
    compression ratio was therefore computed between two different kinds of file.
    """
    path = f"{CKPT_DIR}/size_probe/{tag}_state_dict.pt"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    try:
        robust_torch_save(model.state_dict(), path)
        return path, file_size_mb(path)
    except Exception as e:
        print(f"  could not serialize {tag} state_dict: {e!r}")
        return None, float("nan")

_int8_state_dict_size_mb = serialized_state_dict_size_mb   # legacy alias

def _metric_fns():
    # SevereErrorRate is a bootstrap metric, not just a table column. The deployment rule says a
    # quantized model must not "credibly worsen" severe error, and that clause previously read the
    # QWK confidence interval because no SER interval existed. A clinical-safety criterion has to be
    # tested on the clinical-safety metric.
    return {"QWK": lambda t, p: fast_qwk(t, p),
            "Accuracy": lambda t, p: float((t == p).mean()),
            "MacroF1": lambda t, p: float(f1_score(t, p, average="macro",
                                                    labels=list(range(NUM_CLASSES)), zero_division=0)),
            "MAE": lambda t, p: float(np.mean(np.abs(t - p))),
            "SevereErrorRate": lambda t, p: float(np.mean(np.abs(t - p) >= 2))}

# Which serialized precision each condition's predictions are stored under.
QUANT_OF = {"ptq_int8": "PTQ_INT8", "qat_int8": "QAT_INT8", "ft_ptq_int8": "FT_PTQ_INT8",
            "ptq_int8_pt2e": "PT2E_INT8"}

DISPLAY_ORDER = ["teacher", "macula_only", "disc_only", "dual_no_distill", "dual_logitkd",
                 "dual_featkd", "dual_csd", "abl_csd_raw_smoothl1", "abl_csd_kl_softmax",
                 "abl_csd_counterfactual", "best_fp32", "fp32_ft_control", "fp32_ft_plain",
                 "ptq_int8", "ft_ptq_int8", "qat_int8", "ptq_int8_pt2e"]

# PRECISE METHOD NAMES. `dual_featkd` and `dual_csd` BOTH sit on top of the tuned logit-KD baseline
# (each adds one extra term to it), so labelling them "Feature-KD" and "CSD" would misdescribe the
# comparison. The controlled ladder actually being tested is:
#     no distillation  ->  logit-KD  ->  logit-KD + feature-KD  ->  logit-KD + CSD
# The paper must say "CSD augmentation and feature-distillation augmentation over a standard logit-KD
# baseline", never a bare "CSD vs Feature-KD".
METHOD_LABELS = {
    "teacher": "Teacher (ResNet-50, dual-view)",
    "macula_only": "Student, macula only",
    "disc_only": "Student, optic-disc only",
    "dual_no_distill": "Dual-view, no distillation",
    "dual_logitkd": "Logit-KD",
    "dual_featkd": "Logit-KD + Feature-KD",
    "dual_csd": "Logit-KD + CSD (proposed)",
    "abl_csd_raw_smoothl1": "CSD ablation: unscaled Huber",
    "abl_csd_kl_softmax": "CSD ablation: KL-softmax (negative control)",
    "abl_csd_counterfactual": "CSD ablation: same-head counterfactual",
    "best_fp32": "M* (FP32)",
    "fp32_ft_control": "FP32 fine-tune control (matched graph)",
    "fp32_ft_plain": "FP32 fine-tune control (plain, secondary)",
    "ft_ptq_int8": "FP32 fine-tune -> PTQ INT8",
    "ptq_int8": "PTQ INT8",
    "qat_int8": "QAT INT8",
    "ptq_int8_pt2e": "PT2E PTQ INT8 (supplementary)",
}
def pretty(c):
    return METHOD_LABELS.get(c, c)

pd.DataFrame([{"condition": k, "paper_label": v} for k, v in METHOD_LABELS.items()]).to_csv(
    f"{TABLES_DIR}/table_condition_labels.csv", index=False)
print('Shared evaluation helpers defined.')

## 13 — FP32 smoke tests (must pass before any training)

In [ ]:
from tqdm.auto import tqdm

def smoke_test():
    ds = DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform)
    batch = next(iter(make_loader(ds, 8, True, workers=0)))
    m, d, y = batch["macula"].to(DEVICE), batch["disc"].to(DEVICE), batch["label"].to(DEVICE)

    teacher = DualViewResNetTeacher(init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student = DualViewLightStudent(init_thresholds=INIT_THRESHOLDS).to(DEVICE)

    n_s, n_sb, n_t = param_count(student), param_count(student.backbone), param_count(teacher)
    print(f"Student {n_s:,} params (backbone {n_sb:,}) | Teacher {n_t:,} | compression {n_t/n_s:.0f}x")
    assert n_s > 150_000, f"student too small ({n_s:,}) -- rev2's 34K student was capacity-capped"

    t_out = teacher(m, d)
    ovr = ordinal_violation_rate(t_out["p_dual"])
    assert ovr == 0.0, f"CORAL monotonicity broken: OVR={ovr}"

    _ = teacher.forward_single(m, "macula")
    cf = teacher.counterfactual_forward(m, d)

    for vm in ["dual", "macula_only", "disc_only"]:
        s_out = get_student_output(student, m, d, vm)
        loss, log, comps = combined_student_loss(t_out, s_out, y, vm, alpha=0.5, beta=1.0,
                                                 pos_weight=POS_WEIGHT.to(DEVICE))
        loss.backward(); student.zero_grad()
        print(f"  [{vm}] loss={loss.item():.4f}")

    s_cf, s_dual = student.counterfactual_forward(m, d), student(m, d)
    l_cf, _, _ = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                       use_counterfactual_csd=True, teacher_cf_out=cf,
                                       student_cf_out=s_cf, pos_weight=POS_WEIGHT.to(DEVICE))
    l_cf.backward(); student.zero_grad()
    print(f"  [dual + counterfactual CSD] loss={l_cf.item():.4f}")

    # Gate 4 precursor: CSD must produce a real gradient, not decoration.
    s_dual = student(m, d)
    _, _, comps = combined_student_loss(t_out, s_dual, y, "dual", alpha=0.5, beta=1.0,
                                        pos_weight=POS_WEIGHT.to(DEVICE))
    gn = probe_all_groups(comps, student)
    student.zero_grad()
    ratio = gn.get("gnorm_ratio_csd_over_task", 0.0)
    print("  gradient norms:", {k: round(v, 4) for k, v in gn.items()})
    assert ratio > 0.01, f"CSD gradient ratio {ratio:.5f} negligible -- rev2's failure mode"
    print(f"  CSD/task gradient ratio at beta=1.0 (probe value): {ratio:.3f}")
    if ratio > 3.0:
        print("  NOTE: at beta=1.0 CSD would dominate the task loss. That is the OPPOSITE of rev2's")
        print("        failure and equally harmful, which is exactly why the Section 22 grid searches")
        print("        beta in [0.1, 0.5] and lets validation choose rather than fixing beta by hand.")

    student.eval(); student.fuse_model()
    print("  fuse_model() OK")
    print("\nSMOKE TEST PASSED.")
    return {"student_params": n_s, "teacher_params": n_t, "csd_grad_ratio": ratio}

SMOKE = smoke_test()
record_gate("Gate_SmokeTest_FP32", True,
            f"student={SMOKE['student_params']:,} teacher={SMOKE['teacher_params']:,} "
            f"csd/task grad ratio={SMOKE['csd_grad_ratio']:.3f}")

## 14 — APTOS backbone pretraining

In [ ]:
def build_backbone(kind):
    if kind == "resnet50":
        m = tv.resnet50(weights=tv.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Identity(); return m, 2048
    if kind == "lightweight":
        m = LightweightBackbone(); return m, m.out_dim
    raise ValueError(kind)

# APTOS has its own label distribution, so its ordinal head must start from APTOS marginals --
# initializing it with DRTiD thresholds would place the pretraining head at the wrong operating point.
APTOS_INIT_THRESHOLDS = compute_init_thresholds(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis")
print("APTOS init thresholds:", [round(t, 4) for t in APTOS_INIT_THRESHOLDS])
print("DRTiD init thresholds:", [round(t, 4) for t in INIT_THRESHOLDS])

def pretrain_backbone(kind, epochs, lr, batch_size, seed=PRIMARY_SEED, force=False):
    out = f"{CKPT_DIR}/pretrained_backbones/aptos_{kind}_backbone.pt"
    epochs = 1 if PREFLIGHT else epochs          # rehearsal: prove the path runs, not convergence
    set_seed(seed)                      # BEFORE construction so random init is actually controlled
    backbone, feat_dim = build_backbone(kind)
    if not force and checkpoint_is_compatible(out, backbone, unwrap_key=None):
        print(f"{kind}: compatible checkpoint exists, skipping."); return out
    pw = compute_pos_weights(f"{APTOS_ROOT}/train_1.csv", grade_col="diagnosis").to(DEVICE)
    tl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/train_1.csv", f"{APTOS_ROOT}/train_images/train_images",
                                            aptos_train_transform), batch_size, True, seed)
    vl = make_loader(APTOSSingleViewDataset(f"{APTOS_ROOT}/valid.csv", f"{APTOS_ROOT}/val_images/val_images",
                                            aptos_eval_transform), batch_size, False)
    head = CORALHead(feat_dim, NUM_CLASSES, APTOS_INIT_THRESHOLDS)
    backbone, head = backbone.to(DEVICE), head.to(DEVICE)
    opt = torch.optim.AdamW(list(backbone.parameters()) + list(head.parameters()), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)

    best, hist = -1.0, []
    for ep in range(epochs):
        backbone.train(); head.train()
        for b in tqdm(tl, desc=f"[pretrain-{kind}] ep{ep}", leave=False):
            img, y = b["image"].to(DEVICE), b["label"].to(DEVICE)
            loss = coral_loss(head(backbone(img))[0], y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        backbone.eval(); head.eval()
        pr, tg = [], []
        with torch.no_grad():
            for b in vl:
                _, p = head(backbone(b["image"].to(DEVICE)))
                pr.extend((p > 0.5).sum(1).cpu().tolist()); tg.extend(b["label"].tolist())
        q = fast_qwk(tg, pr); hist.append({"epoch": ep, "val_qwk": q})
        print(f"  ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best = q; robust_torch_save(backbone.state_dict(), out)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/pretrain_{kind}_history.csv", index=False)
    assert best > 0.0, f"pretrain {kind} QWK<=0 -- worse than majority baseline"
    print(f"{kind} pretrain done. best val QWK={best:.4f}")
    return out

RESNET50_BACKBONE_CKPT   = pretrain_backbone("resnet50",   epochs=20, lr=1e-4, batch_size=32)
LIGHTWEIGHT_BACKBONE_CKPT = pretrain_backbone("lightweight", epochs=30, lr=1e-3, batch_size=32)

## 15 — Teacher training & Gate 2

In [ ]:
def train_teacher(freeze_epochs=5, finetune_epochs=20, patience=8, lambda_aux=0.3,
                  seed=PRIMARY_SEED, batch_size=16, freeze_lr=3e-4, finetune_lr=1e-5, force=False):
    out = f"{CKPT_DIR}/teacher/teacher_final.pt"
    if PREFLIGHT:
        freeze_epochs, finetune_epochs, patience = 1, 1, 1
    set_seed(seed)                      # BEFORE construction
    model = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, model, "model_state"):
        print("Teacher checkpoint compatible, skipping."); return out
    model = model.to(DEVICE)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    model.backbone.load_state_dict(robust_torch_load(RESNET50_BACKBONE_CKPT, map_location=DEVICE))

    best, hist, holder = -1.0, [], {"state": None}
    def run(epochs, lr, best):
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 0.02)
        bad = 0
        for _ in range(epochs):
            ge = len(hist); model.train()
            for b in tqdm(tl, desc=f"[teacher] ep{ge}", leave=False):
                m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
                o = model(m, d)
                loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + lambda_aux * aux_loss(o, y, pos_weight=pw)
                opt.zero_grad(); loss.backward(); opt.step()
            sch.step()
            q = quick_val_qwk(model, vl, DEVICE, "dual")
            hist.append({"epoch": ge, "val_qwk": q, "lr": sch.get_last_lr()[0]})
            print(f"  ep{ge}: val_QWK={q:.4f}")
            if q > best:
                best, bad = q, 0
                holder["state"] = copy.deepcopy(model.state_dict())
                robust_torch_save({"model_state": holder["state"], "epoch": ge, "val_qwk": q}, out)
            else:
                bad += 1
                if bad >= patience: print(f"  early stop @ep{ge}"); break
        return best

    for p in model.backbone.parameters(): p.requires_grad = False
    best = run(freeze_epochs, freeze_lr, best)
    # Reload best freeze-phase weights from MEMORY, not Drive -- a write-then-immediate-read of the
    # same path can hit Drive's FUSE sync lag and raise FileNotFoundError mid-run.
    if holder["state"] is not None: model.load_state_dict(holder["state"])
    for p in model.backbone.parameters(): p.requires_grad = True
    best = run(finetune_epochs, finetune_lr, best)

    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/teacher_history.csv", index=False)
    print(f"Teacher done. best val QWK={best:.4f}")
    return out

TEACHER_CKPT = train_teacher()

_t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
_t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
_gain = compute_dual_view_gain(_t, VAL_LOADER, DEVICE)
GATE2_PASSED = _gain["QWK_dual"] > max(_gain["QWK_aux_macula"], _gain["QWK_aux_disc"])
print("Gate 2 (validation):", {k: round(v, 4) for k, v in _gain.items()})
if not GATE2_PASSED:
    print("*** Teacher shows no dual-view advantage. CSD's Delta is only meaningful if it does.  ***")
    print("*** Levers: lower lambda_aux, lower freeze_lr, more epochs, or a different seed.      ***")
# BLOCKING in the real run: every CSD result downstream is uninterpretable without this.
# Non-blocking during PREFLIGHT, where the teacher is deliberately undertrained.
record_gate("Gate2_Teacher", GATE2_PASSED,
            f"QWK_dual={_gain['QWK_dual']:.4f} vs max(aux)={max(_gain['QWK_aux_macula'], _gain['QWK_aux_disc']):.4f} "
            f"G_internal={_gain['DualViewGain_G_internal']:+.4f}",
            blocking=not PREFLIGHT)

# ---- Fixed global CSD normalization, from the FROZEN teacher over TRAIN (P0-6) ----
_scale_loader = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform), 16, False)
CSD_GLOBAL_SCALE    = compute_global_delta_scale(_t, _scale_loader, DEVICE, counterfactual=False)
CSD_GLOBAL_SCALE_CF = compute_global_delta_scale(_t, _scale_loader, DEVICE, counterfactual=True)
print(f"CSD global scale  E_train[|delta_T|]        = {CSD_GLOBAL_SCALE:.6f}")
print(f"CSD global scale (counterfactual delta)     = {CSD_GLOBAL_SCALE_CF:.6f}")
save_json({"csd_global_scale": CSD_GLOBAL_SCALE, "csd_global_scale_cf": CSD_GLOBAL_SCALE_CF},
          f"{CONFIG_DIR}/csd_global_scale.json")
del _t, _scale_loader

## 16 — Generic student trainer

One function drives every student condition, so training logic cannot drift between conditions.
Logs per-epoch loss values, **weighted contributions**, and **per-component gradient norms** to
`gradient_contributions_<condition>_<seed>.csv`.

In [ ]:
_teacher_cache = None
def get_teacher():
    global _teacher_cache
    if _teacher_cache is None:
        t = DualViewResNetTeacher(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
        t.load_state_dict(robust_torch_load(TEACHER_CKPT, map_location=DEVICE)["model_state"])
        t.eval()
        for p in t.parameters(): p.requires_grad = False
        _teacher_cache = t
    return _teacher_cache

def train_student_condition(run_name, seed, view_mode, alpha=0.0, beta=0.0, lambda_aux=0.5,
                            csd_variant="smoothl1_norm", tau_kd=2.0, tau_csd=0.5,
                            use_counterfactual_csd=False, epochs=40, patience=8, lr=1e-3,
                            batch_size=16, gamma_feat=0.0, huber_beta=1.0, weight_decay=1e-4,
                            force=False):
    if PREFLIGHT:
        # 2 epochs (not 1) so best-epoch selection and the patience path are still exercised.
        epochs, patience = 2, 2
    ck_dir = f"{CKPT_DIR}/student/{run_name}"; os.makedirs(ck_dir, exist_ok=True)
    out = f"{ck_dir}/best_seed{seed}.pt"
    set_seed(seed)                      # BEFORE construction
    probe = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, probe, "model_state"):
        print(f"{run_name}|seed{seed}: compatible checkpoint exists, skipping.")
        return out, None
    del probe

    cfg = dict(run_name=run_name, seed=seed, view_mode=view_mode, alpha=alpha, beta=beta,
               lambda_aux=lambda_aux, csd_variant=csd_variant, tau_kd=tau_kd, tau_csd=tau_csd,
               use_counterfactual_csd=use_counterfactual_csd, epochs=epochs, patience=patience,
               lr=lr, batch_size=batch_size, gamma_feat=gamma_feat, huber_beta=huber_beta,
               weight_decay=weight_decay, pos_weight_mode=POS_WEIGHT_MODE)
    save_json(cfg, f"{CONFIG_DIR}/{run_name}_seed{seed}.json")

    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)

    teacher = get_teacher()
    student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    student.backbone.load_state_dict(robust_torch_load(LIGHTWEIGHT_BACKBONE_CKPT, map_location=DEVICE))

    feat_proj, trainable = None, list(student.parameters())
    if gamma_feat > 0:
        feat_proj = nn.Linear(student.fusion.out_dim, teacher.fusion.out_dim).to(DEVICE)
        trainable += list(feat_proj.parameters())

    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.02)
    # Probes are taken on the SHARED BACKBONE (every loss term reaches it) and on fusion+main_head
    # (the decision path). See gradient_probe_groups() for why one group alone is not enough.
    best, bad, hist = -1.0, 0, []

    for ep in range(epochs):
        student.train(); logs, gnorms = [], {}
        for bi, b in enumerate(tqdm(tl, desc=f"[{run_name}|s{seed}] ep{ep}", leave=False)):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            with torch.no_grad():
                t_out = teacher(m, d)
                t_cf = teacher.counterfactual_forward(m, d) if use_counterfactual_csd else None
            s_out = get_student_output(student, m, d, view_mode)
            s_cf = student.counterfactual_forward(m, d) if (use_counterfactual_csd and view_mode == "dual") else None

            loss, log, comps = combined_student_loss(
                t_out, s_out, y, view_mode, alpha=alpha, beta=beta, lambda_aux=lambda_aux,
                tau_kd=tau_kd, csd_variant=csd_variant, tau_csd=tau_csd, pos_weight=pw,
                use_counterfactual_csd=use_counterfactual_csd, teacher_cf_out=t_cf,
                student_cf_out=s_cf, gamma_feat=gamma_feat, feat_projector=feat_proj,
                huber_beta=huber_beta)
            if bi == 0 and view_mode == "dual":
                gnorms = probe_all_groups(comps, student); student.zero_grad(set_to_none=True)
            opt.zero_grad(); loss.backward(); opt.step()
            logs.append(log)
        sch.step()

        q = quick_val_qwk(student, vl, DEVICE, view_mode)
        row = {k: float(np.mean([l[k] for l in logs if k in l])) for k in logs[0]}
        row.update({"epoch": ep, "val_qwk": q, "lr": sch.get_last_lr()[0], **gnorms})
        hist.append(row)
        ls = {k: round(v, 4) for k, v in row.items() if k.startswith("L_")}
        gs = {k.replace("gnorm_", ""): round(v, 3) for k, v in gnorms.items()}
        print(f"  ep{ep}: val_QWK={q:.4f} losses={ls}" + (f" grad={gs}" if gs else ""))

        if q > best:
            best, bad = q, 0
            robust_torch_save({"model_state": student.state_dict(), "epoch": ep, "val_qwk": q,
                               "seed": seed, "config": cfg}, out)
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break

    hdf = pd.DataFrame(hist)
    hdf.to_csv(f"{LOGS_DIR}/{run_name}_seed{seed}_history.csv", index=False)
    gcols = [c for c in hdf.columns if c.startswith("gnorm_") or c.startswith("W_") or c.startswith("L_")]
    if gcols:
        hdf[["epoch"] + gcols].to_csv(f"{LOGS_DIR}/gradient_contributions_{run_name}_{seed}.csv", index=False)
    print(f"[{run_name}|s{seed}] best val QWK={best:.4f}")
    return out, hist

print("Student trainer defined.")

## 17–20 — Baselines with matched hyperparameter budgets

**Hyperparameter fairness.** Every distillation method gets its own small, pre-registered validation
grid on the inferential seed, then the winner is run across all core seeds. Giving CSD a grid while
fixing `alpha=0.5, tau=2` for logit-KD and `gamma=1.0` for feature-KD would make any CSD win
attributable to tuning budget rather than to the method — the first thing a reviewer would ask.

Grids are declared here, before any result is seen, and selection uses validation QWK only.

In [ ]:
# Single-view baselines. Also the reference for EXTERNAL dual-view gain, since they are the only
# models that never see two views.
for s_ in SEEDS_BASELINE:
    train_student_condition("macula_only", seed=s_, view_mode="macula_only")
    train_student_condition("disc_only",   seed=s_, view_mode="disc_only")

In [ ]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_no_distill", seed=s_, view_mode="dual", alpha=0.0, beta=0.0, lambda_aux=0.5)

In [ ]:
# ---- Pre-registered grids (fixed before any result is inspected) ----
GRID_LOGITKD  = [{"alpha": 0.25, "tau_kd": 2.0}, {"alpha": 0.5, "tau_kd": 2.0},
                 {"alpha": 0.5,  "tau_kd": 4.0}, {"alpha": 1.0, "tau_kd": 2.0}]
GRID_FEATKD   = [{"gamma_feat": 0.1}, {"gamma_feat": 0.5}, {"gamma_feat": 1.0}, {"gamma_feat": 2.0}]
GRID_CSD      = [{"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.1},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.2},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.5,  "beta": 0.5},
                 {"csd_variant": "smoothl1_norm", "alpha": 0.25, "beta": 0.2},
                 {"csd_variant": "magnitude_weighted_direction", "alpha": 0.5, "beta": 0.2}]
if PREFLIGHT:
    GRID_LOGITKD, GRID_FEATKD, GRID_CSD = GRID_LOGITKD[:2], GRID_FEATKD[:2], GRID_CSD[:2]
save_json({"logitkd": GRID_LOGITKD, "featkd": GRID_FEATKD, "csd": GRID_CSD},
          f"{CONFIG_DIR}/preregistered_grids.json")

def run_grid(tag, grid, base_kwargs, seeds=None):
    """Trains each configuration on EVERY tuning seed and selects on the MEAN validation QWK.

    The earlier version tuned on a single inferential seed and then ran the winner multi-seed. That
    makes the chosen hyperparameter a draw from the seed lottery: on 200 validation eyes, seed-to-seed
    QWK spread is comparable to the spread between neighbouring grid points, so "best on seed 42" is
    not evidence that a configuration is better. Selection stays VALIDATION-ONLY either way.
    """
    seeds = seeds or SEEDS_TUNING
    per_seed, rows = [], []
    for combo in grid:
        name = tag + "_" + "_".join(f"{k}{v}" for k, v in sorted(combo.items()))
        qs, f1s, sers, maes = [], [], [], []
        for sd in seeds:
            ck, _ = train_student_condition(name, seed=sd, view_mode="dual",
                                            **{**base_kwargs, **combo})
            st = robust_torch_load(ck, map_location=DEVICE)
            mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
            mdl.load_state_dict(st["model_state"])
            y, yp, pc = get_predictions(mdl, VAL_LOADER, DEVICE, "dual")
            m = compute_all_metrics(y, yp, pc)
            qs.append(m["QWK"]); f1s.append(m["MacroF1"])
            sers.append(m["SevereErrorRate"]); maes.append(m["MAE"])
            per_seed.append({**combo, "run_name": name, "seed": sd, "val_QWK": m["QWK"],
                             "val_MacroF1": m["MacroF1"], "val_SevereErrorRate": m["SevereErrorRate"],
                             "val_MAE": m["MAE"]})
            del mdl
        rows.append({**combo, "run_name": name, "n_seeds": len(seeds),
                     "val_QWK_mean": float(np.mean(qs)), "val_QWK_sd": float(np.std(qs, ddof=1)) if len(qs) > 1 else 0.0,
                     "val_MacroF1_mean": float(np.mean(f1s)),
                     "val_SevereErrorRate_mean": float(np.mean(sers)),
                     "val_MAE_mean": float(np.mean(maes)),
                     # kept so downstream code that reads `val_QWK` still resolves
                     "val_QWK": float(np.mean(qs)), "val_MacroF1": float(np.mean(f1s)),
                     "val_SevereErrorRate": float(np.mean(sers)), "val_MAE": float(np.mean(maes))})
    df = (pd.DataFrame(rows)
            .sort_values(["val_QWK_mean", "val_MacroF1_mean", "val_SevereErrorRate_mean", "val_MAE_mean"],
                         ascending=[False, False, True, True])
            .reset_index(drop=True))
    df.to_csv(f"{TABLES_DIR}/table_01_grid_{tag}.csv", index=False)
    pd.DataFrame(per_seed).to_csv(f"{TABLES_DIR}/table_01_grid_{tag}_per_seed.csv", index=False)
    print(f"--- {tag} grid: mean validation QWK over seeds {seeds} ---")
    print(df.drop(columns=[c for c in ("val_QWK", "val_MacroF1", "val_SevereErrorRate", "val_MAE")
                           if c in df.columns]).to_string(index=False))
    return df

GRID_DF_LOGITKD = run_grid("grid_logitkd", GRID_LOGITKD, {"beta": 0.0, "lambda_aux": 0.5})
BEST_KD = {"alpha": float(GRID_DF_LOGITKD.iloc[0]["alpha"]), "tau_kd": float(GRID_DF_LOGITKD.iloc[0]["tau_kd"])}
print("selected logit-KD:", BEST_KD, f"(mean val QWK={GRID_DF_LOGITKD.iloc[0]['val_QWK_mean']:.4f})")

In [ ]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_logitkd", seed=s_, view_mode="dual", beta=0.0, lambda_aux=0.5, **BEST_KD)

In [ ]:
GRID_DF_FEATKD = run_grid("grid_featkd", GRID_FEATKD, {"beta": 0.0, "lambda_aux": 0.5, **BEST_KD})
BEST_FEAT = {"gamma_feat": float(GRID_DF_FEATKD.iloc[0]["gamma_feat"])}
print("selected feature-KD:", BEST_FEAT, f"(mean val QWK={GRID_DF_FEATKD.iloc[0]['val_QWK_mean']:.4f})")

for s_ in SEEDS_CORE:
    train_student_condition("dual_featkd", seed=s_, view_mode="dual", beta=0.0, lambda_aux=0.5,
                            **BEST_KD, **BEST_FEAT)

## 21–23 — Gate 4 (CSD signal), grid search, final CSD training

In [ ]:
@torch.no_grad()
def gate4_csd_signal(teacher, loader, device):
    """Is there a non-trivial complementarity shift to distil at all?"""
    teacher.eval(); norms = []
    for b in loader:
        o = teacher(b["macula"].to(device), b["disc"].to(device))
        norms.append(_delta(o["p_dual"], o["p_macula"], o["p_disc"]).abs().sum(1).cpu())
    n = torch.cat(norms)
    stats = {"mean_L1": float(n.mean()), "median_L1": float(n.median()),
             "q25": float(n.quantile(.25)), "q75": float(n.quantile(.75)),
             "frac_gt_0.02": float((n > 0.02).float().mean())}
    pd.DataFrame({"delta_L1_norm": n.numpy()}).to_csv(f"{METRICS_DIR}/gate4_teacher_delta_distribution.csv", index=False)
    ok = stats["mean_L1"] >= 1e-3
    record_gate("Gate4_CSD_Signal", ok, ", ".join(f"{k}={v:.4f}" for k, v in stats.items()),
                blocking=not PREFLIGHT)
    return stats

DELTA_STATS = gate4_csd_signal(get_teacher(), VAL_LOADER, DEVICE)

# ---- Gate 4b: FINAL CSD gradient balance ----
# The Section 13 smoke test measured the CSD/task gradient ratio with a randomly initialised teacher
# and CSD_GLOBAL_SCALE=1.0, because neither existed yet. What actually trains is the FROZEN final
# teacher with the measured global scale, so the balance has to be re-measured here. rev2 died of a
# CSD term contributing <0.5% of the objective; this gate makes that failure impossible to miss.
def gate4b_final_csd_gradient(teacher, loader, device, betas=(0.1, 0.2, 0.5, 1.0)):
    set_seed(PRIMARY_SEED)
    probe_student = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(device)
    # Start from the SAME weights a real student starts from -- an APTOS-pretrained backbone -- so the
    # measured balance describes the actual optimisation, not a randomly initialised stand-in.
    probe_student.backbone.load_state_dict(robust_torch_load(LIGHTWEIGHT_BACKBONE_CKPT, map_location=device))
    b = next(iter(loader))
    m, d, y = b["macula"].to(device), b["disc"].to(device), b["label"].to(device)
    with torch.no_grad():
        t_out = teacher(m, d)
    rows = {}
    for beta in betas:
        s_out = probe_student(m, d)
        _, _, comps = combined_student_loss(t_out, s_out, y, "dual", alpha=0.5, beta=beta,
                                            lambda_aux=0.5, pos_weight=POS_WEIGHT.to(device),
                                            csd_scale=CSD_GLOBAL_SCALE)
        g = probe_all_groups(comps, probe_student)
        probe_student.zero_grad(set_to_none=True)
        rows[str(beta)] = {k: round(v, 6) for k, v in g.items()}
    diag = {"csd_global_scale": CSD_GLOBAL_SCALE, "teacher_ckpt": TEACHER_CKPT,
            "probe_groups": ["SharedBackbone", "FusionMain"], "by_beta": rows,
            "note": ("ratios are measured on the SHARED BACKBONE, the only parameter set every loss "
                     "term back-propagates into")}
    save_json(diag, f"{METRICS_DIR}/csd_final_gradient_diagnostic.json")
    ratios = {float(k): v.get("gnormSharedBackbone_ratio_csd_over_task", 0.0) for k, v in rows.items()}
    print("  CSD/task gradient ratio on the shared backbone, by beta:",
          {k: round(v, 4) for k, v in ratios.items()})
    # The grid searches beta in [0.1, 0.5]; SOME beta in the searched range must give CSD a real but
    # non-dominant share. Both failure modes are caught: silent (<1%) and overwhelming (>10x).
    usable = {b_: r for b_, r in ratios.items() if 0.01 <= r <= 10.0}
    ok = len(usable) > 0
    record_gate("Gate4b_CSD_GradientBalance", ok,
                f"scale={CSD_GLOBAL_SCALE:.6f}; ratios={ {k: round(v, 4) for k, v in ratios.items()} }; "
                f"{len(usable)}/{len(ratios)} beta values land in the usable band [0.01, 10]",
                blocking=not PREFLIGHT)
    del probe_student
    return diag

CSD_FINAL_GRAD = gate4b_final_csd_gradient(get_teacher(), VAL_LOADER, DEVICE)

In [ ]:
# CSD gets exactly the same treatment as the other methods: pre-registered grid, inferential seed,
# validation-only selection. Beta range is set from the MEASURED gradient ratio (beta=1.0 puts CSD
# several times above the task gradient), so the grid brackets subordinate-to-balanced.
GRID_DF_CSD = run_grid("grid_csd", GRID_CSD, {"lambda_aux": 0.5})
BEST_CSD_VARIANT = GRID_DF_CSD.iloc[0]["csd_variant"]
BEST_ALPHA       = float(GRID_DF_CSD.iloc[0]["alpha"])
BEST_BETA        = float(GRID_DF_CSD.iloc[0]["beta"])
grid_df = GRID_DF_CSD
print(f"selected CSD: variant={BEST_CSD_VARIANT} alpha={BEST_ALPHA} beta={BEST_BETA} "
      f"(mean val QWK={GRID_DF_CSD.iloc[0]['val_QWK_mean']:.4f} over seeds {SEEDS_TUNING})")
save_json({"csd_variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA,
           "logitkd": BEST_KD, "featkd": BEST_FEAT}, f"{CONFIG_DIR}/selected_hyperparameters.json")

In [ ]:
for s_ in SEEDS_CORE:
    train_student_condition("dual_csd", seed=s_, view_mode="dual", alpha=BEST_ALPHA, beta=BEST_BETA,
                            lambda_aux=0.5, csd_variant=BEST_CSD_VARIANT,
                            tau_kd=BEST_KD["tau_kd"], tau_csd=0.5)

In [ ]:
# ---- CSD ablations (spec 13): formulation controls, at the selected alpha/beta ----
for s_ in SEEDS_BASELINE:
    train_student_condition("abl_csd_raw_smoothl1", seed=s_, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.7, lambda_aux=0.5, csd_variant="smoothl1")          # rev2 formulation
for s_ in SEEDS_BASELINE:
    train_student_condition("abl_csd_kl_softmax", seed=s_, view_mode="dual", alpha=BEST_ALPHA,
                            beta=0.5, lambda_aux=0.5, csd_variant="kl_softmax")        # v1 negative control
train_student_condition("abl_csd_counterfactual", seed=PRIMARY_SEED, view_mode="dual",
                        alpha=BEST_ALPHA, beta=BEST_BETA, lambda_aux=0.5,
                        csd_variant=BEST_CSD_VARIANT, use_counterfactual_csd=True)     # judge.md Flag 1/3

## 24–25 — Model selection (validation only) & best FP32 deployment candidate

**Two-stage selection**, because picking the single best `(condition, seed)` row out of 20 runs is a
maximum over noisy estimates — a seed lottery whose winner also carries an upward-biased validation
score (winner's curse).

```
STAGE 1 (statistical)  method M* = argmax_m  mean_s QWK_val(m, s)      <- compares METHODS
STAGE 2 (operational)  checkpoint s* = argmax_s QWK_val(M*, s)         <- picks a deployable file
```

Ties within 0.005 resolve by Macro-F1 ↑ → severe-error ↓ → MAE ↓ → simpler method.
**The test set is not consulted at either stage.**

Every core seed of the selected method becomes an RQ2 base model, so `FP32_s → PTQ_s / QAT_s /
FP32FT_s / FT-PTQ_s` is an exactly matched set per seed.

In [ ]:
CORE_CONDITIONS = ["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]

def collect_val_scores():
    rows = []
    for cond in CORE_CONDITIONS:
        for s in SEEDS_CORE:
            ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
            if not os.path.exists(ck): continue
            st = robust_torch_load(ck, map_location="cpu")
            mdl = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
            mdl.load_state_dict(st["model_state"])
            y, yp, pc = get_predictions(mdl, VAL_LOADER, DEVICE, "dual")
            m = compute_all_metrics(y, yp, pc)
            rows.append({"condition": cond, "seed": s, "checkpoint": ck, **{k: m[k] for k in
                         ("QWK", "MacroF1", "SevereErrorRate", "MAE", "Accuracy")}})
            del mdl
    return pd.DataFrame(rows)

VAL_SCORES = collect_val_scores()
VAL_SCORES.to_csv(f"{TABLES_DIR}/table_02_validation_scores.csv", index=False)
print(VAL_SCORES.groupby("condition")[["QWK", "MacroF1", "SevereErrorRate", "MAE"]].agg(["mean", "std"]).round(4).to_string())

def select_method(df):
    """STAGE 1 -- pick the METHOD by mean validation QWK across seeds.

    The earlier rule took the single highest-QWK (condition, seed) row out of every run. With 4
    methods x 5 seeds that is a 20-way maximum over noisy estimates, so the winner is partly a seed
    lottery and its validation QWK is upward-biased (winner's curse). Averaging over seeds first
    makes the comparison between METHODS, which is what RQ1 actually asks.
    """
    agg = (df.groupby("condition")
             .agg(QWK_mean=("QWK", "mean"), QWK_sd=("QWK", "std"), n_seeds=("QWK", "size"),
                  MacroF1_mean=("MacroF1", "mean"), SER_mean=("SevereErrorRate", "mean"),
                  MAE_mean=("MAE", "mean"))
             .reset_index())
    agg["method_rank_simplicity"] = agg["condition"].map(
        {c: i for i, c in enumerate(CORE_CONDITIONS)}).fillna(99)
    agg = agg.sort_values("QWK_mean", ascending=False).reset_index(drop=True)
    top = agg.iloc[0]["QWK_mean"]
    tied = agg[agg["QWK_mean"] >= top - SELECTION_TIE_EPS]
    if len(tied) > 1:
        print(f"  {len(tied)} methods within {SELECTION_TIE_EPS} mean QWK -- applying tie-break chain "
              "(Macro-F1 up, severe-error down, MAE down, simpler method)")
        tied = tied.sort_values(["MacroF1_mean", "SER_mean", "MAE_mean", "method_rank_simplicity"],
                                ascending=[False, True, True, True])
    return tied.iloc[0], agg

def select_checkpoint(df, condition):
    """STAGE 2 -- within the chosen method, pick the deployable checkpoint by validation QWK."""
    d = df[df.condition == condition].sort_values("QWK", ascending=False).reset_index(drop=True)
    top = d.iloc[0]["QWK"]
    tied = d[d["QWK"] >= top - SELECTION_TIE_EPS]
    if len(tied) > 1:
        tied = tied.sort_values(["MacroF1", "SevereErrorRate", "MAE"], ascending=[False, True, True])
    return tied.iloc[0]

METHOD_ROW, METHOD_TABLE = select_method(VAL_SCORES)
METHOD_TABLE.to_csv(f"{TABLES_DIR}/table_02b_method_selection.csv", index=False)
print("Stage 1 -- method selection on MEAN validation QWK:")
print(METHOD_TABLE.round(4).to_string(index=False))

BEST_CONDITION = METHOD_ROW["condition"]
BEST_ROW = select_checkpoint(VAL_SCORES, BEST_CONDITION)
BEST_SEED = int(BEST_ROW["seed"])
BEST_FP32_CKPT = BEST_ROW["checkpoint"]
print(f"\nStage 1: method M* = {BEST_CONDITION} "
      f"(mean val QWK={METHOD_ROW['QWK_mean']:.4f} +/- {METHOD_ROW['QWK_sd'] if METHOD_ROW['QWK_sd'] == METHOD_ROW['QWK_sd'] else 0:.4f} over {int(METHOD_ROW['n_seeds'])} seeds)")
print(f"Stage 2: deployment checkpoint = seed {BEST_SEED} (val QWK={BEST_ROW['QWK']:.4f})")

# Every core seed of the SELECTED method is an RQ2 base model, so FP32_s / PTQ_s / QAT_s / FP32FT_s
# form an exactly matched set per seed (see the quantization section).
RQ2_BASE_CKPTS = {}
for _s in SEEDS_CORE:
    _ck = f"{CKPT_DIR}/student/{BEST_CONDITION}/best_seed{_s}.pt"
    if os.path.exists(_ck): RQ2_BASE_CKPTS[_s] = _ck
print(f"RQ2 base checkpoints (method {BEST_CONDITION}): seeds {sorted(RQ2_BASE_CKPTS)}")

# The best CSD model is tracked separately: even if M* is not CSD, the paper still needs the best
# CSD artifact for the RQ1 mechanism analysis.
_csd = VAL_SCORES[VAL_SCORES.condition == "dual_csd"]
BEST_CSD_SEED = int(select_checkpoint(VAL_SCORES, "dual_csd")["seed"]) if len(_csd) else PRIMARY_SEED
BEST_CSD_CKPT = f"{CKPT_DIR}/student/dual_csd/best_seed{BEST_CSD_SEED}.pt"
print(f"Best CSD artifact        = dual_csd seed {BEST_CSD_SEED}")
save_json({"best_condition": BEST_CONDITION, "best_seed": BEST_SEED, "best_ckpt": BEST_FP32_CKPT,
           "best_val_qwk": float(BEST_ROW["QWK"]),
           "method_mean_val_qwk": float(METHOD_ROW["QWK_mean"]),
           "method_n_seeds": int(METHOD_ROW["n_seeds"]),
           "best_csd_seed": BEST_CSD_SEED, "rq2_base_seeds": sorted(RQ2_BASE_CKPTS),
           "selection_rule": ("STAGE 1: argmax mean validation QWK over seeds (method); "
                              f"STAGE 2: argmax validation QWK within that method (checkpoint); "
                              f"ties<{SELECTION_TIE_EPS} -> {SELECTION_TIEBREAK}"),
           "selection_data": "DRTiD validation split only -- the test set is not consulted"},
          f"{CONFIG_DIR}/model_selection.json")

def load_student(ckpt):
    m = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS).to(DEVICE)
    m.load_state_dict(robust_torch_load(ckpt, map_location=DEVICE)["model_state"])
    return m.eval()

# One fixed example pair reused for export tracing, latency benchmarking and parity checks.
_ex_batch = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 8, False, workers=0)))
_sample_pair = (_ex_batch["macula"], _ex_batch["disc"])

## 26–29 — Quantization: PTQ INT8, QAT INT8, and matched controls

**Matched scope comes first.** RQ2 asks *PTQ vs QAT*. That question is only answerable if both
quantize the same operators, so **both use eager backbone-only quantization**: the same
`QuantizableBackbone` wrapper, the same fused Conv-BN-ReLU set, the same `fbgemm` qconfig family.
They differ in exactly one thing — static calibration versus quantization-aware fine-tuning.

**Why PT2E is not the RQ2 path.** `prepare_pt2e` with `Quantizer.set_global()` quantizes every
eligible operator in the exported graph, which is a *different* operator set from eager
backbone-only. Comparing PT2E PTQ against eager QAT would confound "PTQ vs QAT" with "different
quantization scope". PT2E is still exercised — as a **supplementary deployment-path demonstration**
(`ptq_int8_pt2e`), reported in its own row and excluded from every RQ2 statistic. The paper must
describe it exactly this way:

```
RQ2 pair      : eager static PTQ (backbone-only)  vs  eager QAT (backbone-only)   [identical scope]
supplementary : PT2E static PTQ, reported separately, never mixed into the comparison
```

**Scope is stated honestly.** Only the CNN backbone is quantized: `torch.cat`, LayerNorm and CORAL's
cumsum/softplus/sigmoid have no eager INT8 kernels. The artifact is an **INT8-quantized backbone with
FP32 fusion and ordinal heads** (mixed-precision deployment).

**`quantization_coverage_pct` is an integrity check, not a headline.** For eager it counts converted
modules; for a PT2E graph it counts quantization *nodes*, which do not map one-to-one onto
conv/linear operators. Gates require `quantized_ops > 0`; the percentage never becomes a claim.

**Everything in RQ2 is matched per seed.** Every quantization variant is derived from the SAME
base checkpoint `FP32_s`, for every core seed `s`:

```
                    ┌── PTQ_s               (static calibration, no gradient)
FP32_s ─────────────┼── QAT_s               (quantization-aware fine-tuning)
                    ├── FP32FT_s            (identical prepared graph, fake-quant OFF)
                    └── FP32FT-plain_s ──► FT-PTQ_s   (ordinary FP32 fine-tune, THEN calibration)
```

So the paired bootstrap always compares seed *s* against seed *s* and never needs a positional
fallback. Previously `best_fp32` and PTQ existed on one seed while QAT had three, which made
`QAT_s ↔ FP32_s` impossible at the training-seed level.

**The four controls answer four different questions.**

| Comparison | Question it answers |
|---|---|
| `PTQ vs FP32` | what does static INT8 cost? |
| `QAT vs FP32` | what does quantization-aware training recover? |
| `QAT vs FP32-FT` | is the QAT gain more than just extra fine-tuning? |
| `QAT vs FT-PTQ` | is *adapting to quantization noise* better than fine-tuning first and quantizing after? |

* `fp32_ft_control` — the *identical* fused, QAT-prepared graph with fake quantization and observers
  switched off. Same architecture, same fusion, same optimizer, same epochs, same early stopping;
  the only remaining difference is whether fake quantization is active.
* `fp32_ft_plain` — an ordinary FP32 fine-tune of the plain (unfused) student, same budget, run on
  every RQ2 seed.
* `ft_ptq_int8` — `fp32_ft_plain` statically calibrated. This isolates *adaptation to quantization
  noise* from *extra gradient steps before quantizing*. It is deliberately built from the **plain**
  fine-tune, not from the prepared-graph control: recovering plain weights out of a fused QAT graph
  loses every BatchNorm parameter (Conv-BN fusion renames them), which was verified to transfer only
  27 of 95 tensors and would have produced a hybrid rather than a control.

**PTQ calibration is deterministic.** The calibration loader is `shuffle=False` over the **entire**
training split (all 800 eyes), and the exact image list is written to `ptq_calibration_manifest.csv`
with a SHA-256. The earlier version drew a shuffled 512-eye subset, so re-running PTQ could produce a
different INT8 model.

**Operator matching is exact.** `15 == 15` does not prove the same 15 operators were quantized, so
`Gate6c_QuantScopeMatched` compares the sorted *module paths* of the quantized operators and is
**blocking** on the final run.

In [ ]:
from torch.ao.quantization import (prepare, convert, prepare_qat, get_default_qconfig,
                                   get_default_qat_qconfig, QuantStub, DeQuantStub,
                                   disable_fake_quant, disable_observer)

# LOCKED QUANTIZATION SCOPE. The RQ2 pair (PTQ vs QAT) must quantize the same operators, so both
# take the eager backbone-only path. PT2E runs separately, as a supplementary demonstration that the
# modern export path works, and never enters an RQ2 statistic.
QUANT_SCOPE = "eager_backbone_only"
QUANT_COVERAGE_CAVEAT = ("quantization_coverage_pct is an INTEGRITY CHECK, not a headline metric: "
                         "for PT2E it counts quantization graph NODES, which do not map one-to-one "
                         "onto conv/linear operators. Gates require quantized_ops > 0 only.")

QUANT_ENGINE = "fbgemm" if "fbgemm" in torch.backends.quantized.supported_engines                else torch.backends.quantized.supported_engines[0]
torch.backends.quantized.engine = QUANT_ENGINE
print("quantization engine:", QUANT_ENGINE)

# PT2E moved. torch 2.11 removed `torch.ao.quantization.quantize_pt2e` and PT2E now lives in
# torchao (`torchao.quantization.pt2e.quantize_pt2e`). The first preflight's PT2E gate failed purely
# because of that relocation. Both locations are tried, newest first, and whichever resolved is
# recorded -- PT2E remains SUPPLEMENTARY and never enters an RQ2 statistic either way.
PT2E_AVAILABLE, PT2E_IMPORT_ERROR, PT2E_SOURCE = False, None, None
_pt2e_attempts = []
for _src, _mod_q, _mod_z in [
        ("torchao", "torchao.quantization.pt2e.quantize_pt2e", "torchao.quantization.pt2e.quantizer"),
        ("torch.ao", "torch.ao.quantization.quantize_pt2e", "torch.ao.quantization.quantizer")]:
    try:
        _q = __import__(_mod_q, fromlist=["prepare_pt2e"])
        prepare_pt2e, convert_pt2e = _q.prepare_pt2e, _q.convert_pt2e
        prepare_qat_pt2e = getattr(_q, "prepare_qat_pt2e", None)   # probe only; eager QAT is used
        _Quantizer = _qcfg = None
        for _qp, _cls, _cfg in [
                (f"{_mod_z}.x86_inductor_quantizer", "X86InductorQuantizer",
                 "get_default_x86_inductor_quantization_config"),
                (f"{_mod_z}.xnnpack_quantizer", "XNNPACKQuantizer", "get_symmetric_quantization_config")]:
            try:
                _m = __import__(_qp, fromlist=[_cls])
                _Quantizer, _qcfg = getattr(_m, _cls), getattr(_m, _cfg)
                break
            except Exception as _qe:
                _pt2e_attempts.append(f"{_qp}: {_qe!r}")
        if _Quantizer is None:
            raise ImportError("no PT2E quantizer class found: " + " | ".join(_pt2e_attempts[-2:]))
        PT2E_AVAILABLE, PT2E_SOURCE = True, _src
        break
    except Exception as _e:
        _pt2e_attempts.append(f"{_mod_q}: {_e!r}")
        PT2E_IMPORT_ERROR = " | ".join(_pt2e_attempts)
print("PT2E available:", PT2E_AVAILABLE,
      f"(via {PT2E_SOURCE})" if PT2E_AVAILABLE else f"({PT2E_IMPORT_ERROR})")

class QuantizableBackbone(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.quant = QuantStub(); self.backbone = backbone; self.dequant = DeQuantStub()
    def forward(self, x):
        return self.dequant(self.backbone(self.quant(x)))

def count_quantized_graph_nodes(model):
    """PT2E returns a GraphModule whose INT8 ops are call_function nodes
    (torch.ops.quantized_decomposed.*), not nn.Module instances -- counting modules alone reports 0
    for a perfectly good PT2E conversion, which would fail Gate 6 on a working model."""
    g = getattr(model, "graph", None)
    if g is None: return 0
    n = 0
    for node in g.nodes:
        t = str(getattr(node, "target", ""))
        if "quantize_per_tensor" in t or "quantize_per_channel" in t or "quantized_decomposed" in t:
            n += 1
    return n

def count_quantized_modules(model):
    """Eager mode: fused quantized modules live under torch.ao.nn.intrinsic.quantized.* -- the class
    NAME (e.g. ConvReLU2d) need not contain 'Quantized', so check the module PATH."""
    return sum(1 for m in model.modules() if "quantized" in type(m).__module__.lower())

def count_quantized_ops(model):
    """Mode-agnostic: eager modules OR PT2E graph nodes."""
    return max(count_quantized_modules(model), count_quantized_graph_nodes(model))

def is_quantized_tree(model):
    return count_quantized_ops(model) > 0

def quantized_module_paths(model):
    """The sorted NAMES of quantized modules -- an exact operator set, not just a count.

    `PTQ ops == QAT ops == 15` does not prove the same 15 operators were quantized. Comparing the
    module paths does, and that is what makes the RQ2 pair genuinely apples-to-apples.
    """
    return sorted(name for name, m in model.named_modules()
                  if "quantized" in type(m).__module__.lower())

def count_active_fake_quant(model):
    """FakeQuantize modules whose fake-quant is actually ENABLED (a disabled one is a pass-through)."""
    n = 0
    for m in model.modules():
        if hasattr(m, "fake_quant_enabled"):
            try:
                n += int(m.fake_quant_enabled.sum().item() > 0)
            except Exception:
                n += 1
    return n

def count_eligible_ops(model):
    return sum(1 for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear)))

def quantization_coverage(quantized_model, fp32_reference):
    """Fraction of eligible conv/linear ops actually converted. Works for eager and PT2E."""
    elig = count_eligible_ops(fp32_reference)
    got = count_quantized_ops(quantized_model)
    return {"eligible_ops": elig, "quantized_ops": got,
            "quantization_coverage_pct": 100.0 * min(got, elig) / max(elig, 1),
            "quantized_op_count_raw": got}

def make_calib_loader(batch_size=None):
    """DETERMINISTIC calibration: shuffle=False over the FULL training split, eval transforms only.

    The earlier loader used shuffle=True and stopped after 64 batches, so PTQ was calibrated on a
    random ~512-eye subset -- re-running the same cell produced a different INT8 model. Labels are
    irrelevant here; only the input distribution matters, and the whole split is cheap.
    """
    bs = batch_size or PTQ_CFG["batch_size"]
    return make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, eval_transform), bs, shuffle=False)

def write_calibration_manifest():
    """Records exactly which images calibrated PTQ, and hashes the list."""
    df = pd.read_csv(DRTID_TRAIN_CSV)[["record_id", "macula_path", "disc_path", "grade"]]
    path = f"{CONFIG_DIR}/ptq_calibration_manifest.csv"
    df.to_csv(path, index=False)
    h = sha256_file(path)
    with open(f"{CONFIG_DIR}/ptq_calibration_manifest_sha256.txt", "w") as f:
        f.write(h)
    print(f"PTQ calibration set: {len(df)} eyes ({2*len(df)} images), full training split, "
          f"shuffle=False, sha256={h[:16]}...")
    return path, h, len(df)

PTQ_CALIB_MANIFEST, PTQ_CALIB_SHA256, PTQ_CALIB_N = write_calibration_manifest()

def _fresh_student_from(ckpt):
    m = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    m.load_state_dict(robust_torch_load(ckpt, map_location="cpu")["model_state"])
    return m.to("cpu").eval()


def _calibrate(prepared, calib_batches):
    cl = make_calib_loader()
    n = 0
    with torch.no_grad():
        for i, b in enumerate(cl):
            prepared(b["macula"], b["disc"]); n = i + 1
            if n >= calib_batches: break
    return n

def run_ptq_eager(fp32_ckpt, calib_batches=None):
    """PRIMARY PTQ path -- eager, backbone-only, byte-for-byte the same scope QAT uses.

    calib_batches=None means "the entire training split", which is the locked protocol; PREFLIGHT
    truncates it purely so the rehearsal is fast.
    """
    if calib_batches is None:
        calib_batches = 4 if PREFLIGHT else 10 ** 9      # 1e9 -> exhaust the loader
    ref = _fresh_student_from(fp32_ckpt)
    model = _fresh_student_from(fp32_ckpt)
    model.eval()                       # fuse_conv_bn_eval asserts eval mode
    model.backbone.fuse_model(qat=False)
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prepared = prepare(model, inplace=False)
    n = _calibrate(prepared, calib_batches)
    q = convert(prepared, inplace=False)
    if count_quantized_ops(q) == 0:
        raise RuntimeError("eager PTQ convert produced no quantized modules")
    return q, {"path": "eager_static_ptq", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE,
               "calib_batches": n, "calib_batch_size": PTQ_CFG["batch_size"],
               "calib_shuffle": False, "calib_manifest_sha256": PTQ_CALIB_SHA256,
               "calib_eyes": PTQ_CALIB_N if not PREFLIGHT else n * PTQ_CFG["batch_size"],
               "is_graph_module": False, "used_in_RQ2": True,
               "quantized_module_paths": quantized_module_paths(q),
               **quantization_coverage(q, ref)}

def run_ptq_pt2e(fp32_ckpt, calib_batches=64):
    """SUPPLEMENTARY PT2E path (torch.export + prepare_pt2e/convert_pt2e).

    Reported on its own row and never compared against eager QAT: set_global() quantizes every
    eligible operator in the exported graph, a different scope from eager backbone-only, and mixing
    the two would confound "PTQ vs QAT" with "different scope".
    """
    if not PT2E_AVAILABLE:
        raise RuntimeError(f"PT2E unavailable: {PT2E_IMPORT_ERROR}")
    calib_batches = 4 if PREFLIGHT else calib_batches
    ref = _fresh_student_from(fp32_ckpt)
    model = _fresh_student_from(fp32_ckpt)
    # Export with a DYNAMIC batch dimension: exporting from a batch-1 example specializes the graph
    # to batch 1 and calibration then trips "Guard failed: macula.size()[0] == 1".
    ex = (_sample_pair[0][:2].cpu(), _sample_pair[1][:2].cpu())
    try:
        from torch.export import Dim
        _bd = Dim("batch", min=1, max=256)
        exported = torch.export.export(model, ex, dynamic_shapes=({0: _bd}, {0: _bd})).module()
    except Exception:
        exported = torch.export.export(model, ex).module()
    prepared = prepare_pt2e(exported, _Quantizer().set_global(_qcfg()))
    n = _calibrate(prepared, calib_batches)
    q = convert_pt2e(prepared)
    cov = quantization_coverage(q, ref)
    if cov["quantized_ops"] == 0:
        raise RuntimeError("PT2E convert produced no quantized ops")
    return q, {"path": "pt2e_static_ptq", "scope": "pt2e_global_exported_graph",
               "pt2e_source": PT2E_SOURCE, "engine": QUANT_ENGINE, "calib_batches": n,
               "is_graph_module": True, "used_in_RQ2": False, **cov}

In [ ]:
# ---- PTQ smoke check on ONE seed ----
# The real PTQ models are built per seed in the RQ2 section below; this only verifies that the
# eager path produces a usable INT8 model at all, so a structural failure surfaces before the loop
# spends time on five seeds.
_ptq_probe_ok, _ptq_probe_err = False, None
try:
    _pm, _pi = run_ptq_eager(BEST_FP32_CKPT, calib_batches=2)
    with torch.no_grad():
        _b = next(iter(make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), 4, False)))
        _o = _pm(_b["macula"], _b["disc"])
        _p = _o["p_dual"] if isinstance(_o, dict) else _o
    assert _p.shape[1] == NUM_THRESHOLDS, f"PTQ output has {_p.shape[1]} thresholds, expected {NUM_THRESHOLDS}"
    _ptq_probe_ok = is_quantized_tree(_pm)
    print(f"  PTQ smoke check OK: {_pi['quantized_ops']} quantized ops, scope={_pi['scope']}")
    del _pm
except Exception as e:
    _ptq_probe_err = repr(e); print(f"  PTQ smoke check FAILED: {e!r}")
record_gate("Gate6a_PTQ_SmokeCheck", _ptq_probe_ok,
            "eager INT8 path produces a usable model" if _ptq_probe_ok else f"FAILED: {_ptq_probe_err}",
            blocking=not PREFLIGHT)

# ---- SUPPLEMENTARY PT2E PTQ: modern export path, reported, never mixed into RQ2 ----
PTQ_PT2E_MODEL, PTQ_PT2E_INFO, PTQ_PT2E_OK = None, {}, False
try:
    PTQ_PT2E_MODEL, PTQ_PT2E_INFO = run_ptq_pt2e(BEST_FP32_CKPT, calib_batches=8)
    PTQ_PT2E_OK = is_quantized_tree(PTQ_PT2E_MODEL)
    print(f"  PT2E supplementary PTQ OK via {PT2E_SOURCE}: "
          f"{PTQ_PT2E_INFO['quantized_ops']} quantization graph nodes")
except Exception as e:
    PTQ_PT2E_INFO = {"path": "pt2e_static_ptq", "pt2e_available": PT2E_AVAILABLE,
                     "pt2e_source": PT2E_SOURCE, "error": repr(e), "used_in_RQ2": False}
    print(f"  PT2E supplementary PTQ unavailable ({e!r}). RQ2 is unaffected -- it never used PT2E.")
record_gate("Gate6b_PT2E_Supplementary", PTQ_PT2E_OK,
            f"modern torch.export quantization path demonstrated via {PT2E_SOURCE}" if PTQ_PT2E_OK else
            f"not demonstrated ({PTQ_PT2E_INFO.get('error', 'unavailable')}); supplementary only, non-blocking")

In [ ]:
def _finetune_loop(model, tag, epochs, lr, batch_size, patience, seed, fake_quant=False):
    """Shared fine-tuning loop for QAT and its FP32 control, so the ONLY difference between them is
    whether fake quantization is active."""
    set_seed(seed)
    pw = POS_WEIGHT.to(DEVICE)
    tl = make_loader(DRTiDDualViewDataset(DRTID_TRAIN_CSV, paired_transform=paired_train_transform), batch_size, True, seed)
    vl = make_loader(DRTiDDualViewDataset(DRTID_VAL_CSV, eval_transform), batch_size, False)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best, bad, best_state, hist = -1.0, 0, None, []
    for ep in range(epochs):
        model.train()
        for b in tqdm(tl, desc=f"[{tag}] ep{ep}", leave=False):
            m, d, y = b["macula"].to(DEVICE), b["disc"].to(DEVICE), b["label"].to(DEVICE)
            o = model(m, d)
            loss = coral_loss(o["logit_dual"], y, pos_weight=pw) + 0.5 * aux_loss(o, y, pos_weight=pw)
            opt.zero_grad(); loss.backward(); opt.step()
        q = quick_val_qwk(model, vl, DEVICE, "dual")
        hist.append({"epoch": ep, "val_qwk": q}); print(f"  {tag} ep{ep}: val_QWK={q:.4f}")
        if q > best:
            best, bad = q, 0; best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: print(f"  early stop @ep{ep}"); break
    if best_state is not None: model.load_state_dict(best_state)
    pd.DataFrame(hist).to_csv(f"{LOGS_DIR}/{tag}_history.csv", index=False)
    return model, best, len(hist)

def count_fake_quant(model):
    return sum(1 for m in model.modules() if "fakequantize" in type(m).__name__.lower()
               or "FakeQuant" in type(m).__name__)

def _build_qat_prepared(fp32_ckpt, load_fp32=True):
    model = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if load_fp32:
        model.load_state_dict(robust_torch_load(fp32_ckpt, map_location="cpu")["model_state"])
    model = model.to("cpu").train()
    model.backbone.fuse_model(qat=True)
    model.backbone = QuantizableBackbone(model.backbone)
    model.backbone.qconfig = get_default_qat_qconfig(QUANT_ENGINE)
    prepare_qat(model, inplace=True)
    return model

def run_qat(fp32_ckpt, epochs=None, lr=None, batch_size=None, patience=None, seed=PRIMARY_SEED,
            force=False, tag_extra=""):
    """QAT from the best FP32 weights; validation-selected; then converted.
    Resumable: the fine-tuned pre-conversion weights are cached so a re-run skips the epochs."""
    epochs = 1 if PREFLIGHT else (epochs if epochs is not None else QAT_CFG["epochs"])
    lr = lr if lr is not None else globals().get("QAT_BEST_LR", QAT_CFG["lr_grid"][1])
    batch_size = batch_size or QAT_CFG["batch_size"]
    patience = 1 if PREFLIGHT else (patience if patience is not None else QAT_CFG["patience"])
    ck = f"{CKPT_DIR}/student/qat/qat_prepared_seed{seed}{tag_extra}.pt"
    os.makedirs(os.path.dirname(ck), exist_ok=True)

    if not force and prepared_checkpoint_reusable(ck):
        try:
            saved = robust_torch_load(ck, map_location="cpu")
            model = _build_qat_prepared(fp32_ckpt, load_fp32=False)
            model.load_state_dict(saved["model_state"])
            q = convert(model.to("cpu").eval(), inplace=False)
            info = dict(saved.get("info", {}))
            info.update({"resumed_from_checkpoint": True, "scope": QUANT_SCOPE, "used_in_RQ2": True})
            info.update(quantization_coverage(q, _fresh_student_from(fp32_ckpt)))
            info["quantized_module_paths"] = quantized_module_paths(q)
            print(f"QAT: reused cached fine-tuned weights (val QWK={info.get('best_val_qwk', float('nan')):.4f})")
            return q, info
        except Exception as e:
            print(f"QAT cache unusable ({e!r}) -- re-running fine-tuning.")

    model = _build_qat_prepared(fp32_ckpt, load_fp32=True)
    # ASSERT fake quantization is genuinely active DURING training -- counting quantized modules
    # after conversion proves nothing about what happened while training.
    n_fq = count_fake_quant(model)
    assert n_fq > 0, "QAT prepare produced no FakeQuantize modules -- fake quantization is NOT active"
    print(f"  QAT: {n_fq} FakeQuantize modules active before fine-tuning starts")

    model, best, n_ep = _finetune_loop(model, f"QAT_seed{seed}{tag_extra}", epochs, lr, batch_size, patience, seed)
    info = {"path": "eager_qat", "scope": QUANT_SCOPE, "engine": QUANT_ENGINE,
            "best_val_qwk": best, "epochs_run": n_ep, "lr": lr, "qat_training_seed": seed,
            "fake_quant_modules_during_training": n_fq, "used_in_RQ2": True,
            "resumed_from_checkpoint": False}
    _q_probe = convert(copy.deepcopy(model).to("cpu").eval(), inplace=False)
    info["quantized_module_paths"] = quantized_module_paths(_q_probe)
    del _q_probe
    robust_torch_save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                       "info": info, "epochs_run": n_ep}, ck)
    q = convert(model.to("cpu").eval(), inplace=False)
    info.update(quantization_coverage(q, _fresh_student_from(fp32_ckpt)))
    return q, info

def run_fp32_ft_control(fp32_ckpt, epochs=None, lr=None, batch_size=None, patience=None,
                        seed=PRIMARY_SEED, force=False):
    """PRIMARY QAT control: the SAME fused, QAT-prepared graph with fake quantization and observers
    DISABLED.

    The earlier control fine-tuned an ordinary unfused FP32 student, so it differed from QAT in
    fusion AND graph structure as well as in fake quantization -- it could not isolate the single
    variable it was named after. Returns the live model, because a prepared graph's state_dict
    cannot be loaded back into a plain DualViewLightStudent.
    """
    # EXACTLY the QAT recipe -- same epochs, same LR, same batch size, same patience. If these ever
    # diverge, the control stops isolating fake quantization and starts measuring a different budget.
    epochs = 1 if PREFLIGHT else (epochs if epochs is not None else QAT_CFG["epochs"])
    lr = lr if lr is not None else globals().get("QAT_BEST_LR", QAT_CFG["lr_grid"][1])
    batch_size = batch_size or QAT_CFG["batch_size"]
    patience = 1 if PREFLIGHT else (patience if patience is not None else QAT_CFG["patience"])
    out = f"{CKPT_DIR}/student/fp32_ft_control/prepared_seed{seed}.pt"
    os.makedirs(os.path.dirname(out), exist_ok=True)

    def _fresh_control():
        m = _build_qat_prepared(fp32_ckpt, load_fp32=True)
        m.apply(disable_fake_quant); m.apply(disable_observer)
        n_active = count_active_fake_quant(m)
        assert n_active == 0, f"control still has {n_active} ACTIVE fake-quant modules -- not a control"
        return m

    if not force and prepared_checkpoint_reusable(out):
        try:
            saved = robust_torch_load(out, map_location="cpu")
            m = _fresh_control()
            m.load_state_dict(saved["model_state"])
            m.apply(disable_fake_quant); m.apply(disable_observer)
            v = float(saved.get("val_qwk", float("nan")))
            print(f"fp32_ft_control seed{seed}: reused cached weights (val QWK={v:.4f})")
            return m.eval(), out, v
        except Exception as e:
            print(f"fp32_ft_control cache unusable ({e!r}) -- re-running fine-tuning.")

    model = _fresh_control()
    model, best, n_ep = _finetune_loop(model, f"FP32FT_seed{seed}", epochs, lr, batch_size, patience, seed)
    model.apply(disable_fake_quant); model.apply(disable_observer)   # belt and braces after training
    assert count_active_fake_quant(model) == 0, "fake quantization re-enabled during control training"
    robust_torch_save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                       "val_qwk": best, "seed": seed, "epochs_run": n_ep,
                       "role": "qat_matched_fp32_control_fake_quant_disabled"}, out)
    return model.eval(), out, best

def run_fp32_ft_plain(fp32_ckpt, epochs=None, lr=None, batch_size=None, patience=None,
                      seed=PRIMARY_SEED, force=False):
    """SECONDARY control: ordinary FP32 fine-tuning of the unfused student, same budget."""
    epochs = 1 if PREFLIGHT else (epochs if epochs is not None else QAT_CFG["epochs"])
    lr = lr if lr is not None else globals().get("QAT_BEST_LR", QAT_CFG["lr_grid"][1])
    batch_size = batch_size or QAT_CFG["batch_size"]
    patience = 1 if PREFLIGHT else (patience if patience is not None else QAT_CFG["patience"])
    out = f"{CKPT_DIR}/student/fp32_ft_plain/best_seed{seed}.pt"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    probe = DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)
    if not force and checkpoint_is_compatible(out, probe, "model_state"):
        print(f"fp32_ft_plain seed{seed}: cached, skipping."); return out
    del probe
    model = _fresh_student_from(fp32_ckpt)
    model, best, n_ep = _finetune_loop(model, f"FP32FTplain_seed{seed}", epochs, lr, batch_size,
                                       patience, seed)
    robust_torch_save({"model_state": model.state_dict(), "val_qwk": best, "seed": seed,
                       "epochs_run": n_ep, "role": "secondary_plain_fp32_finetuning_control"}, out)
    return out

In [ ]:
# =====================================================================================
# STEP 1 -- QAT learning rate, chosen on VALIDATION over the tuning seeds.
# QAT was previously run at a hand-fixed lr=3e-5. Since QAT is half of RQ2 it gets the same
# treatment every other method gets: a small pre-registered grid, scored by mean validation QWK,
# never by test performance. The FP32 control then inherits the identical setting.
# =====================================================================================
QAT_LR_GRID = QAT_CFG["lr_grid"][:2] if PREFLIGHT else QAT_CFG["lr_grid"]
_qat_grid_rows = []
for _lr in QAT_LR_GRID:
    _qs = []
    for _sd in SEEDS_TUNING:
        try:
            _m, _i = run_qat(RQ2_BASE_CKPTS.get(_sd, BEST_FP32_CKPT), lr=_lr, seed=_sd,
                             tag_extra=f"_lrgrid{_lr:g}")
            _qs.append(float(_i.get("best_val_qwk", float("nan"))))
            del _m
        except Exception as e:
            print(f"  QAT lr={_lr:g} seed={_sd} FAILED: {e!r}")
    if _qs:
        _qat_grid_rows.append({"lr": _lr, "n_seeds": len(_qs),
                               "val_QWK_mean": float(np.nanmean(_qs)),
                               "val_QWK_sd": float(np.nanstd(_qs, ddof=1)) if len(_qs) > 1 else 0.0})
QAT_GRID_DF = pd.DataFrame(_qat_grid_rows).sort_values("val_QWK_mean", ascending=False).reset_index(drop=True)
QAT_GRID_DF.to_csv(f"{TABLES_DIR}/table_01_grid_qat_lr.csv", index=False)
QAT_BEST_LR = float(QAT_GRID_DF.iloc[0]["lr"]) if len(QAT_GRID_DF) else QAT_CFG["lr_grid"][1]
print("--- QAT learning-rate grid (validation only) ---")
print(QAT_GRID_DF.round(4).to_string(index=False) if len(QAT_GRID_DF) else "  (no successful runs)")
print(f"selected QAT lr = {QAT_BEST_LR:g}  (the FP32 fine-tuning control uses the same value)")
save_json({"grid": QAT_LR_GRID, "selected_lr": QAT_BEST_LR, "seeds": SEEDS_TUNING,
           "selected_on": "mean DRTiD validation QWK"}, f"{CONFIG_DIR}/qat_lr_selection.json")

In [ ]:
# =====================================================================================
# STEP 2 -- one matched variant set per core seed.
# For every seed s: FP32_s -> {PTQ_s, QAT_s, FP32FT_s, FT-PTQ_s}. Every RQ2 comparison is then a
# genuine seed-to-seed pairing and the bootstrap never falls back to positional pairing.
# =====================================================================================
PTQ_MODELS, PTQ_INFOS       = {}, {}
QAT_MODELS, QAT_INFOS       = {}, {}
FP32_FT_MODELS, FP32_FT_CKPTS, FP32_FT_VALQWK = {}, {}, {}
FP32_FT_PLAIN_CKPTS         = {}
FT_PTQ_MODELS, FT_PTQ_INFOS = {}, {}
RQ2_SEEDS = [sd for sd in SEEDS_CORE if sd in RQ2_BASE_CKPTS]

for sd in RQ2_SEEDS:
    base = RQ2_BASE_CKPTS[sd]
    print(f"\n===== RQ2 seed {sd} | base = {os.path.basename(base)} =====")

    # (a) PTQ_s -- static calibration only, no gradient steps
    try:
        PTQ_MODELS[sd], PTQ_INFOS[sd] = run_ptq_eager(base)
        print(f"  PTQ_{sd}: {PTQ_INFOS[sd]['quantized_ops']} quantized ops, "
              f"{PTQ_INFOS[sd]['calib_batches']} calibration batches")
    except Exception as e:
        print(f"  PTQ seed {sd} FAILED: {e!r}")

    # (b) FP32FT_s -- identical prepared graph, fake quantization OFF
    try:
        _m, _p, _v = run_fp32_ft_control(base, seed=sd)
        FP32_FT_MODELS[sd], FP32_FT_CKPTS[sd], FP32_FT_VALQWK[sd] = _m, _p, _v
    except Exception as e:
        print(f"  FP32-FT control seed {sd} FAILED: {e!r}")

    # (c) FP32FT-plain_s -- an ORDINARY FP32 fine-tune of the plain student, same budget as QAT.
    #     This is the model FT-PTQ_s is built from, so no weight surgery is involved: fine-tuning and
    #     quantization both happen on the same plain module tree PTQ_s uses.
    try:
        FP32_FT_PLAIN_CKPTS[sd] = run_fp32_ft_plain(base, seed=sd)
    except Exception as e:
        print(f"  fp32_ft_plain seed {sd} FAILED: {e!r}")

    # (d) FT-PTQ_s -- extra fine-tuning FIRST, static calibration AFTER. Together with QAT_s this
    #     isolates "adapting to quantization noise" from "more gradient steps before quantizing".
    if sd in FP32_FT_PLAIN_CKPTS:
        try:
            FT_PTQ_MODELS[sd], FT_PTQ_INFOS[sd] = run_ptq_eager(FP32_FT_PLAIN_CKPTS[sd])
            FT_PTQ_INFOS[sd]["derived_from"] = "fp32_ft_plain"
            print(f"  FT-PTQ_{sd}: {FT_PTQ_INFOS[sd]['quantized_ops']} quantized ops")
        except Exception as e:
            print(f"  FT-PTQ seed {sd} unavailable: {e!r}")

    # (e) QAT_s -- quantization-aware fine-tuning at the validation-selected LR
    try:
        QAT_MODELS[sd], QAT_INFOS[sd] = run_qat(base, lr=QAT_BEST_LR, seed=sd)
    except Exception as e:
        print(f"  QAT seed {sd} FAILED: {e!r}")

# Representative for the exports/registry; the per-seed set above is what the statistics use.
FP32_FT_PLAIN_CKPT = FP32_FT_PLAIN_CKPTS.get(PRIMARY_SEED,
                                             next(iter(FP32_FT_PLAIN_CKPTS.values()), ""))

# Deployment representatives, always chosen on VALIDATION.
PTQ_MODEL = PTQ_MODELS.get(BEST_SEED, PTQ_MODELS.get(min(PTQ_MODELS), None) if PTQ_MODELS else None)
PTQ_INFO  = PTQ_INFOS.get(BEST_SEED, PTQ_INFOS.get(min(PTQ_INFOS), {}) if PTQ_INFOS else {})
PTQ_OK    = bool(PTQ_MODELS) and all(is_quantized_tree(m) for m in PTQ_MODELS.values())

def _qat_val(sd):
    try:
        v = float(QAT_INFOS[sd].get("best_val_qwk", float("nan")))
    except Exception:
        v = float("nan")
    return v if v == v else float("-inf")

QAT_DEPLOY_SEED = max(QAT_MODELS, key=_qat_val) if QAT_MODELS else None
QAT_MODEL = QAT_MODELS[QAT_DEPLOY_SEED] if QAT_DEPLOY_SEED is not None else None
QAT_INFO  = QAT_INFOS.get(QAT_DEPLOY_SEED, {})

record_gate("Gate6_PTQ_Integrity", PTQ_OK and len(PTQ_MODELS) == len(RQ2_SEEDS),
            f"{len(PTQ_MODELS)}/{len(RQ2_SEEDS)} seeds quantized; scope={PTQ_INFO.get('scope')}; "
            f"quantized_ops={PTQ_INFO.get('quantized_ops')}/{PTQ_INFO.get('eligible_ops')} eligible; "
            f"calibration={PTQ_INFO.get('calib_eyes')} eyes shuffle=False "
            f"sha256={str(PTQ_CALIB_SHA256)[:12]}...",
            blocking=not PREFLIGHT)

QAT_OK = (len(QAT_MODELS) == len(RQ2_SEEDS)
          and all(is_quantized_tree(m) for m in QAT_MODELS.values())
          and all((QAT_INFOS[sd].get("fake_quant_modules_during_training", 0) or 0) > 0
                  or QAT_INFOS[sd].get("resumed_from_checkpoint", False) for sd in QAT_MODELS))
record_gate("Gate7_QAT_Integrity", QAT_OK,
            f"{len(QAT_MODELS)}/{len(RQ2_SEEDS)} seeds quantized; lr={QAT_BEST_LR:g}; "
            f"deploy seed={QAT_DEPLOY_SEED} (val QWK={_qat_val(QAT_DEPLOY_SEED) if QAT_DEPLOY_SEED is not None else float('nan'):.4f}); "
            f"fake-quant during training={QAT_INFO.get('fake_quant_modules_during_training', 'cached')}",
            blocking=not PREFLIGHT)

# EXACT operator-set matching. Counting alone cannot show that the same operators were quantized.
_ptq_paths = PTQ_INFO.get("quantized_module_paths", [])
_qat_paths = QAT_INFO.get("quantized_module_paths", [])
save_json({"ptq": _ptq_paths, "n": len(_ptq_paths)}, f"{CONFIG_DIR}/ptq_quantized_modules.json")
save_json({"qat": _qat_paths, "n": len(_qat_paths)}, f"{CONFIG_DIR}/qat_quantized_modules.json")
_only_ptq = sorted(set(_ptq_paths) - set(_qat_paths))
_only_qat = sorted(set(_qat_paths) - set(_ptq_paths))
_scope_ok = bool(_ptq_paths and _qat_paths and not _only_ptq and not _only_qat
                 and PTQ_INFO.get("scope") == QAT_INFO.get("scope"))
record_gate("Gate6c_QuantScopeMatched", _scope_ok,
            f"{len(_ptq_paths)} PTQ vs {len(_qat_paths)} QAT quantized modules; "
            f"identical module set={not (_only_ptq or _only_qat)}"
            + (f"; PTQ-only={_only_ptq[:3]}; QAT-only={_only_qat[:3]}" if (_only_ptq or _only_qat) else ""),
            blocking=not PREFLIGHT)

save_json({"locked_scope": QUANT_SCOPE,
           "rq2_design": "per-seed matched: FP32_s -> {PTQ_s, QAT_s, FP32FT_s, FT-PTQ_s}",
           "rq2_seeds": RQ2_SEEDS, "rq2_base_condition": BEST_CONDITION,
           "coverage_caveat": QUANT_COVERAGE_CAVEAT,
           "scope_match_verified": _scope_ok,
           "quantized_module_paths_identical": not (_only_ptq or _only_qat),
           "ptq_calibration": {"manifest": PTQ_CALIB_MANIFEST, "sha256": PTQ_CALIB_SHA256,
                               "eyes": PTQ_CALIB_N, "shuffle": False,
                               "batch_size": PTQ_CFG["batch_size"]},
           "qat_lr_selected": QAT_BEST_LR, "qat_lr_grid": QAT_LR_GRID,
           "ptq": {str(k): v for k, v in PTQ_INFOS.items()},
           "qat": {str(k): v for k, v in QAT_INFOS.items()},
           "ft_ptq": {str(k): v for k, v in FT_PTQ_INFOS.items()},
           "qat_deploy_seed": QAT_DEPLOY_SEED,
           "qat_deploy_seed_selected_on": "DRTiD validation QWK",
           "fp32_ft_control": {"kind": "qat_prepared_graph_with_fake_quant_disabled",
                               "val_qwk": {str(k): v for k, v in FP32_FT_VALQWK.items()}},
           "engine": QUANT_ENGINE, "pt2e_available": PT2E_AVAILABLE, "pt2e_source": PT2E_SOURCE,
           "pt2e_import_error": PT2E_IMPORT_ERROR,
           "artifact": "INT8-quantized backbone with FP32 fusion and ordinal heads (mixed precision)"},
          f"{CONFIG_DIR}/quantization_info.json")

## 29b — RQ2 on VALIDATION, and the frozen deployment decision

**This is the last decision made before the test set is touched.** The earlier version chose the
deployment model from `RAW`, which is the DRTiD *test* evaluation — so the test set was, in effect,
a model-selection set. The order is now the only defensible one:

```
validation  ->  deployment selection  ->  FREEZE  ->  DRTiD test  ->  DeepDRiD
```

Every RQ2 variant is scored on validation here, the pre-registered rule picks one, and
`DEPLOY_CHOICE_FROZEN` is set. The test-side cell later only *asserts* that flag.

**The severe-error clause is evaluated on severe error.** `severe_error_must_not_credibly_worsen`
previously consulted the QWK confidence interval — a rule about clinical safety adjudicated by an
agreement metric. `SevereErrorRate` is now a bootstrap metric, and an INT8 candidate is rejected when
the 95% CI of `ΔSER = SER_INT8 − SER_FP32` lies entirely above zero.

In [ ]:
# ---- Pre-registered DEPLOYMENT selection rule ----
DEPLOY_RULE = {"min_qwk_retention_pct": 95.0,
               "severe_error_must_not_credibly_worsen": True,
               "severe_error_test": "95% CI of dSER = SER_int8 - SER_fp32 must not lie entirely above 0",
               "tiebreak": "lowest median CPU latency",
               "fallback": "best_fp32",
               "selection_data": "DRTiD VALIDATION only"}
save_json(DEPLOY_RULE, f"{CONFIG_DIR}/deployment_selection_rule.json")

@torch.no_grad()
def _val_predictions(model, on_cpu=False):
    if on_cpu:
        r = _predict_cpu_all(model, VAL_LOADER)
        return r["y_true"], r["y_pred_dual"], r["p_dual"], r["cluster_ids"]
    return get_predictions(model, VAL_LOADER, DEVICE, "dual", return_clusters=True)

# Per-seed validation metrics for every RQ2 variant, plus the efficiency numbers the rule needs.
VAL_PRED_STORE, _rq2_val_rows = {}, []

def _score_val(condition, seed, model, quant, on_cpu, ckpt=None):
    yt, yp, pc, cid = _val_predictions(model, on_cpu=on_cpu)
    m = compute_all_metrics(yt, yp, pc)
    VAL_PRED_STORE[(condition, seed, quant)] = {"y_true": yt, "y_pred": yp, "cluster_ids": cid}
    lat = benchmark_latency(model, _sample_pair[0], _sample_pair[1], "dual",
                            bench=BENCH_EFF, on_cpu=True, label=condition)
    _sdp, _sds = serialized_state_dict_size_mb(model, f"valsize_{condition}_seed{seed}")
    _rq2_val_rows.append({"condition": condition, "seed": seed, "quantization": quant,
                          "val_QWK": m["QWK"], "val_MacroF1": m["MacroF1"],
                          "val_Accuracy": m["Accuracy"], "val_MAE": m["MAE"],
                          "val_SevereErrorRate": m["SevereErrorRate"],
                          "Latency_median_ms": lat["Latency_median_ms"],
                          "CheckpointSize_MB": _sds, "checkpoint": ckpt or ""})

for sd in RQ2_SEEDS:
    _m = load_student(RQ2_BASE_CKPTS[sd])
    _score_val("best_fp32", sd, _m, "FP32", False, RQ2_BASE_CKPTS[sd]); del _m
for sd, mdl in FP32_FT_MODELS.items():
    _score_val("fp32_ft_control", sd, mdl.to(DEVICE), "FP32", False, FP32_FT_CKPTS[sd])
for sd, pm in PTQ_MODELS.items():
    _score_val("ptq_int8", sd, pm, "PTQ_INT8", True)
for sd, ck in FP32_FT_PLAIN_CKPTS.items():
    _mp = load_student(ck); _score_val("fp32_ft_plain", sd, _mp, "FP32", False, ck); del _mp
for sd, fm in FT_PTQ_MODELS.items():
    _score_val("ft_ptq_int8", sd, fm, "FT_PTQ_INT8", True)
for sd, qm in QAT_MODELS.items():
    _score_val("qat_int8", sd, qm, "QAT_INT8", True)

RQ2_VALIDATION_RESULTS = pd.DataFrame(_rq2_val_rows)
RQ2_VALIDATION_SUMMARY = (RQ2_VALIDATION_RESULTS
    .groupby("condition")
    .agg(n_seeds=("seed", "size"), val_QWK=("val_QWK", "mean"), val_QWK_sd=("val_QWK", "std"),
         val_MacroF1=("val_MacroF1", "mean"), val_SER=("val_SevereErrorRate", "mean"),
         val_MAE=("val_MAE", "mean"), Latency_median_ms=("Latency_median_ms", "mean"),
         CheckpointSize_MB=("CheckpointSize_MB", "mean"))
    .reindex([c for c in DISPLAY_ORDER if c in set(RQ2_VALIDATION_RESULTS.condition)])
    .reset_index())
RQ2_VALIDATION_RESULTS.to_csv(f"{TABLES_DIR}/table_04a_rq2_validation_per_seed.csv", index=False)
RQ2_VALIDATION_SUMMARY.to_csv(f"{TABLES_DIR}/table_04_rq2_validation.csv", index=False)
print("RQ2 on VALIDATION (the only data the deployment decision may use):")
print(RQ2_VALIDATION_SUMMARY.round(4).to_string(index=False))

In [ ]:
# ---- Paired dSER on VALIDATION, used by the safety clause of the deployment rule ----
def _val_paired_delta(cond_int8, metric="SevereErrorRate", B=None, rng_seed=7):
    """Cluster+seed bootstrap of metric(INT8_s) - metric(FP32_s) on the VALIDATION split."""
    B = B or (300 if PREFLIGHT else 2000)
    fn = _metric_fns()[metric]
    seeds = [sd for sd in RQ2_SEEDS
             if (cond_int8, sd, QUANT_OF.get(cond_int8, "FP32")) in VAL_PRED_STORE
             and ("best_fp32", sd, "FP32") in VAL_PRED_STORE]
    if not seeds: return None
    q = QUANT_OF.get(cond_int8, "FP32")
    yt = VAL_PRED_STORE[("best_fp32", seeds[0], "FP32")]["y_true"]
    cl = VAL_PRED_STORE[("best_fp32", seeds[0], "FP32")]["cluster_ids"]
    uniq = np.unique(cl); idx_by = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed); diffs = np.empty(B)
    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by[c] for c in pick])
        sel = rng.integers(0, len(seeds), size=len(seeds))
        diffs[b] = float(np.mean([
            fn(yt[idx], VAL_PRED_STORE[(cond_int8, seeds[j], q)]["y_pred"][idx])
            - fn(yt[idx], VAL_PRED_STORE[("best_fp32", seeds[j], "FP32")]["y_pred"][idx])
            for j in sel]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"condition": cond_int8, "metric": metric, "n_seeds": len(seeds),
            "mean_diff": float(diffs.mean()), "ci_low": float(lo), "ci_high": float(hi),
            "credibly_worse": bool(lo > 0)}

VAL_DELTA_ROWS = []
for _c in ["ptq_int8", "ft_ptq_int8", "qat_int8"]:
    for _met in ["SevereErrorRate", "QWK"]:
        _r = _val_paired_delta(_c, _met)
        if _r: VAL_DELTA_ROWS.append(_r)
VAL_DELTAS = pd.DataFrame(VAL_DELTA_ROWS)
if len(VAL_DELTAS):
    VAL_DELTAS.to_csv(f"{TABLES_DIR}/table_04b_rq2_validation_deltas.csv", index=False)
    print("\nPaired validation deltas vs FP32 (used by the deployment rule):")
    print(VAL_DELTAS.round(4).to_string(index=False))

def choose_deployment_model(val_summary, val_deltas):
    """Pre-registered rule, evaluated on VALIDATION ONLY.

    A candidate must (a) retain >= min_qwk_retention_pct of FP32 validation QWK and (b) not be
    credibly worse on SEVERE ERROR -- judged by the CI of dSER, not by the QWK CI as before.
    Among survivors, lowest median CPU latency wins; if none survive, FP32 is deployed.
    """
    base = val_summary[val_summary.condition == "best_fp32"]
    if not len(base): return "best_fp32", "no FP32 validation reference available"
    q0 = float(base["val_QWK"].iloc[0])
    cands = []
    for c in ["ptq_int8", "ft_ptq_int8", "qat_int8"]:
        sub = val_summary[val_summary.condition == c]
        if not len(sub): continue
        ret = 100.0 * float(sub["val_QWK"].iloc[0]) / q0 if q0 else float("nan")
        d = val_deltas[(val_deltas.condition == c) & (val_deltas.metric == "SevereErrorRate")]
        ser_worse = bool(len(d) and d.iloc[0]["credibly_worse"])
        ok = (ret >= DEPLOY_RULE["min_qwk_retention_pct"]) and not ser_worse
        print(f"  {c:14s} QWK retention {ret:6.1f}%  severe-error credibly worse: {ser_worse}  -> "
              f"{'ELIGIBLE' if ok else 'rejected'}")
        if ok:
            cands.append((float(sub["Latency_median_ms"].iloc[0]), c, ret))
    if not cands:
        return "best_fp32", (f"no INT8 variant met >={DEPLOY_RULE['min_qwk_retention_pct']}% validation "
                             "QWK retention without credibly worsening severe error")
    cands.sort()
    return cands[0][1], (f"validation QWK retention {cands[0][2]:.1f}%, severe error not credibly worse, "
                         f"lowest median CPU latency {cands[0][0]:.2f} ms")

print("\nApplying the pre-registered deployment rule to VALIDATION results:")
DEPLOY_CHOICE, DEPLOY_REASON = choose_deployment_model(RQ2_VALIDATION_SUMMARY, VAL_DELTAS)
DEPLOY_CHOICE_SEED = {"best_fp32": BEST_SEED, "ptq_int8": BEST_SEED,
                      "ft_ptq_int8": BEST_SEED, "qat_int8": QAT_DEPLOY_SEED}.get(DEPLOY_CHOICE, BEST_SEED)
DEPLOY_QUANTIZATION = QUANT_OF.get(DEPLOY_CHOICE, "FP32")
DEPLOY_CHOICE_FROZEN = True     # nothing after this point may change the deployment model
print(f"\nDEPLOYMENT MODEL = {DEPLOY_CHOICE} (seed {DEPLOY_CHOICE_SEED}) -- {DEPLOY_REASON}")
print("FROZEN. The DRTiD test set and DeepDRiD are evaluated after this line and cannot change it.")
save_json({"chosen": DEPLOY_CHOICE, "seed": DEPLOY_CHOICE_SEED,
           "quantization": DEPLOY_QUANTIZATION, "reason": DEPLOY_REASON,
           "frozen": DEPLOY_CHOICE_FROZEN, "rule": DEPLOY_RULE,
           "decided_on": "DRTiD validation split",
           "validation_summary": RQ2_VALIDATION_SUMMARY.to_dict(orient="records"),
           "validation_deltas": VAL_DELTAS.to_dict(orient="records") if len(VAL_DELTAS) else []},
          f"{RESULTS_DIR}/deployment_choice.json")

## 30–33 — Full internal test evaluation

**Honest scope statement.** Within this locked run the DRTiD official test set is not consulted for
any selection decision — every choice (hyperparameters, `M*`, deployment candidate) is made on
validation. But this project has a history: outcomes from an earlier development run (rev2) were
inspected and did motivate the revisions that produced this pipeline. So the correct wording for the
paper is *"the official test set was not used for model selection within the final locked run;
outcomes from earlier development runs had previously been inspected"* — **not** "touched once ever".
**DeepDRiD is the genuinely frozen confirmatory evaluation.**

Every condition × seed is evaluated on the held-out DRTiD test set. **Per-sample predictions are
written to `predictions/<condition>_<seed>.csv`** so any future metric can be recomputed without
re-running inference or touching the models again.

In [ ]:
TEST_DS = DRTiDDualViewDataset(DRTID_TEST_CSV, eval_transform)
TEST_LOADER = make_loader(TEST_DS, 16, False)
_sample = next(iter(TEST_LOADER))
TEACHER = get_teacher()

CLUSTER_LEVEL = {"DRTiD": "record_eye", "DeepDRiD": "patient"}

def save_predictions(condition, seed, y_true, y_pred, p_cum, cluster_ids, quantization="FP32",
                     dataset="DRTiD", split="test", latency_ms=np.nan):
    df = pd.DataFrame({"dataset": dataset, "split": split,
                       "cluster_id": cluster_ids,
                       "cluster_level": CLUSTER_LEVEL.get(dataset, "unknown"),
                       "sample_id": [f"{dataset}_{p}" for p in cluster_ids],
                       "true_grade": y_true, "pred_grade": y_pred,
                       "condition": condition, "seed": seed, "quantization": quantization,
                       "latency_ms": latency_ms})
    for k in range(p_cum.shape[1]):
        df[f"p_threshold_{k}"] = p_cum[:, k].cpu().numpy()
    path = f"{PREDS_DIR}/{dataset}_{split}_{condition}_seed{seed}_{quantization}.csv"
    df.to_csv(path, index=False)
    return path

def save_confusion(condition, seed, y_true, y_pred, tag=""):
    labels = list(range(NUM_CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    idx = [f"true_{g}" for g in labels]; col = [f"pred_{g}" for g in labels]
    suffix = f"{condition}_seed{seed}{tag}"
    pd.DataFrame(cm, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_raw_{suffix}.csv")
    pd.DataFrame(cmn, index=idx, columns=col).to_csv(f"{METRICS_DIR}/confusion_matrix_normalized_{suffix}.csv")
    return cm, cmn

PRED_STORE = {}   # (condition, seed, quantization) -> dict(y_true, y_pred, cluster_ids)
rows, collapse_report = [], []



def evaluate_condition(model, condition, seed, view_mode="dual", ckpt=None, quantization="FP32",
                       loader=None, on_cpu_model=False, with_shift=True, with_latency=True,
                       fp32_reference_model=None, size_tag=None, with_memory=False):
    loader = loader or TEST_LOADER
    cpu_pred = None
    if on_cpu_model:
        cpu_pred = _predict_cpu_all(model, loader)
        y_true, y_pred, p_cum, cids = (cpu_pred["y_true"], cpu_pred["y_pred_dual"],
                                       cpu_pred["p_dual"], cpu_pred["cluster_ids"])
    else:
        y_true, y_pred, p_cum, cids = get_predictions(model, loader, DEVICE, view_mode, return_clusters=True)

    m = compute_all_metrics(y_true, y_pred, p_cum)
    row = {"condition": condition, "seed": seed, "quantization": quantization,
           "view_mode": view_mode, **m}

    if view_mode == "dual":
        if on_cpu_model:
            # Quantized models are CPU-only, but they still need the dual-view gain -- this is the
            # exact RQ2 sub-question ("does INT8 preserve the dual-view advantage?"). Taken from the
            # SAME forward pass, so it works for an exported GraphModule too.
            if "y_pred_macula" in cpu_pred:
                qd = fast_qwk(y_true, y_pred)
                qm = fast_qwk(y_true, cpu_pred["y_pred_macula"])
                qdd = fast_qwk(y_true, cpu_pred["y_pred_disc"])
                row.update({"QWK_dual": qd, "QWK_aux_macula": qm, "QWK_aux_disc": qdd,
                            "DualViewGain_G_internal": qd - max(qm, qdd)})
            else:
                print(f"  {condition}: no auxiliary head outputs -- internal dual-view gain unavailable")
                row.update({"QWK_dual": fast_qwk(y_true, y_pred), "QWK_aux_macula": float("nan"),
                            "QWK_aux_disc": float("nan"), "DualViewGain_G_internal": float("nan")})
        else:
            row.update(compute_dual_view_gain(model, loader, DEVICE))
            if with_shift:
                row.update(compute_shift_fidelity(TEACHER, model, loader, DEVICE))
                # Every dual-view student is ALSO scored in same-head counterfactual space, so the
                # counterfactual ablation is compared against the others in the objective it was
                # actually trained on rather than only in three-head space.
                row.update(compute_shift_fidelity(TEACHER, model, loader, DEVICE, counterfactual=True))

    # Quantization does NOT change the architectural parameter count -- packed INT8 weights simply
    # stop appearing as ordinary Parameters. Report the FP32 architecture's count and let SIZE carry
    # the compression story.
    ref_for_params = fp32_reference_model if fp32_reference_model is not None else model
    row["ParamCount"] = param_count(ref_for_params)
    row["ParamCount_is_architectural"] = True

    # SIZE: always a freshly serialized state_dict, for every condition and every precision, so
    # FP32-vs-INT8 compression is measured between two artifacts of the same kind.
    _sdp, _sds = serialized_state_dict_size_mb(model, size_tag or f"{condition}_seed{seed}_{quantization}")
    row["CheckpointSize_MB"] = _sds          # the comparable number used by every ratio
    row["StateDictSize_MB"] = _sds
    row["state_dict_path"] = _sdp or ""
    row["TrainingCheckpointSize_MB"] = (file_size_mb(ckpt) if (ckpt and os.path.exists(ckpt))
                                        else float("nan"))

    if with_latency:
        row.update(benchmark_latency(model, _sample["macula"], _sample["disc"], view_mode,
                                     bench=BENCH_EFF, on_cpu=True, label=condition))
    if with_memory:
        row.update(measure_memory(lambda: copy.deepcopy(model).to("cpu").eval(),
                                  _sample["macula"], _sample["disc"]))

    warns = check_prediction_collapse(y_true, y_pred)
    if warns:
        collapse_report.append({"condition": condition, "seed": seed, "quantization": quantization,
                                "warnings": "; ".join(warns)})
        print(f"  !! {condition}|s{seed}|{quantization}: " + "; ".join(warns))
    save_predictions(condition, seed, y_true, y_pred, p_cum, cids, quantization)
    save_confusion(condition, seed, y_true, y_pred, tag=f"_{quantization}")
    PRED_STORE[(condition, seed, quantization)] = {"y_true": y_true, "y_pred": y_pred, "cluster_ids": cids}
    rows.append(row)
    return row

# ---- Teacher ----
evaluate_condition(TEACHER, "teacher", "-", "dual", TEACHER_CKPT, with_shift=False)

# ---- Single-view baselines (also the EXTERNAL gain reference) ----
for cond, vm in [("macula_only", "macula_only"), ("disc_only", "disc_only")]:
    for s in SEEDS_BASELINE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, vm, ck, with_shift=False); del mdl

# ---- Core dual-view conditions ----
for cond in CORE_CONDITIONS:
    for s in SEEDS_CORE:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

# ---- Ablations ----
for cond, seeds in [("abl_csd_raw_smoothl1", SEEDS_BASELINE), ("abl_csd_kl_softmax", SEEDS_BASELINE),
                    ("abl_csd_counterfactual", [PRIMARY_SEED])]:
    for s in seeds:
        ck = f"{CKPT_DIR}/student/{cond}/best_seed{s}.pt"
        if not os.path.exists(ck): continue
        mdl = load_student(ck); evaluate_condition(mdl, cond, s, "dual", ck); del mdl

RAW = pd.DataFrame(rows)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)
print(f"\nEvaluated {len(RAW)} model-runs on the internal test set.")

In [ ]:
# ---- External dual-view gain, now that the independent single-view references exist ----
_indep_mac = RAW[(RAW.condition == "macula_only")]["QWK"].mean()
_indep_dsc = RAW[(RAW.condition == "disc_only")]["QWK"].mean()
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
print(f"External gain reference: independent macula QWK={_indep_mac:.4f}, disc QWK={_indep_dsc:.4f}")

# ---- RQ2: every variant, on every seed, all derived from the SAME FP32_s ----
# Each row below is one cell of the matched design
#     FP32_s -> {PTQ_s, QAT_s, FP32FT_s, FT-PTQ_s}
# so every pre-registered RQ2 comparison is a true seed-to-seed pairing. `best_fp32` is the base
# model of the SELECTED method at seed s -- the same checkpoint every variant on that row came from.
BEST_FP32_MODEL = load_student(BEST_FP32_CKPT)          # deployment representative (validation-chosen)

for sd in RQ2_SEEDS:
    _m = load_student(RQ2_BASE_CKPTS[sd])
    evaluate_condition(_m, "best_fp32", sd, "dual", RQ2_BASE_CKPTS[sd], quantization="FP32",
                       with_memory=(sd == BEST_SEED), size_tag=f"best_fp32_seed{sd}")
    del _m

# PRIMARY QAT control: the matched QAT-prepared graph with fake quantization disabled. Evaluated
# from the live object because its state_dict belongs to the prepared graph and cannot be loaded
# into a plain DualViewLightStudent.
for sd, mdl in FP32_FT_MODELS.items():
    evaluate_condition(mdl.to(DEVICE), "fp32_ft_control", sd, "dual", FP32_FT_CKPTS[sd], "FP32",
                       with_shift=False, size_tag=f"fp32_ft_control_seed{sd}")

# SECONDARY control, now per seed: ordinary FP32 fine-tune of the unfused student. This is also the
# model FT-PTQ_s is quantized from, so reporting it makes the FT-PTQ chain fully traceable.
for sd, ck in FP32_FT_PLAIN_CKPTS.items():
    _mp = load_student(ck)
    evaluate_condition(_mp, "fp32_ft_plain", sd, "dual", ck, "FP32",
                       with_shift=False, size_tag=f"fp32_ft_plain_seed{sd}")
    del _mp

for sd, pm in PTQ_MODELS.items():
    evaluate_condition(pm, "ptq_int8", sd, "dual", None, "PTQ_INT8", on_cpu_model=True,
                       fp32_reference_model=BEST_FP32_MODEL, size_tag=f"ptq_int8_seed{sd}",
                       with_memory=(sd == BEST_SEED))

for sd, fm in FT_PTQ_MODELS.items():
    evaluate_condition(fm, "ft_ptq_int8", sd, "dual", None, "FT_PTQ_INT8", on_cpu_model=True,
                       fp32_reference_model=BEST_FP32_MODEL, size_tag=f"ft_ptq_int8_seed{sd}")

for sd, qm in QAT_MODELS.items():
    evaluate_condition(qm, "qat_int8", sd, "dual", None, "QAT_INT8", on_cpu_model=True,
                       fp32_reference_model=BEST_FP32_MODEL, size_tag=f"qat_int8_seed{sd}",
                       with_memory=(sd == QAT_DEPLOY_SEED))

# SUPPLEMENTARY PT2E row -- reported for completeness of the deployment story, never entered into an
# RQ2 comparison, because its quantization scope differs from the eager PTQ/QAT pair.
if PTQ_PT2E_OK:
    try:
        evaluate_condition(PTQ_PT2E_MODEL, "ptq_int8_pt2e", BEST_SEED, "dual", None, "PT2E_INT8",
                           on_cpu_model=True, fp32_reference_model=BEST_FP32_MODEL,
                           size_tag="ptq_int8_pt2e")
    except Exception as e:
        print(f"  supplementary PT2E evaluation failed ({e!r}) -- the RQ2 pair is unaffected.")

RAW = pd.DataFrame(rows)
RAW["DualViewGain_G_external"] = RAW.apply(
    lambda r: compute_external_gain(r["QWK"], _indep_mac, _indep_dsc)
    if r["view_mode"] == "dual" else np.nan, axis=1)
RAW.to_csv(f"{METRICS_DIR}/all_conditions_raw.csv", index=False)

if collapse_report:
    pd.DataFrame(collapse_report).to_csv(f"{METRICS_DIR}/prediction_collapse_warnings.csv", index=False)
    print(f"\n{len(collapse_report)} collapse warning(s) written -- see prediction_collapse_warnings.csv")
else:
    print("\nNo prediction-collapse warnings: every model predicts >2 distinct grades with recall above floor.")

# The run must not be able to "succeed" while silently leaving RQ1/RQ2 incomplete.
# Conditions are checked against an EXPECTED list, not against whatever happens to be in RAW. The
# earlier version derived the list from RAW itself, so a total PTQ failure simply removed PTQ from
# the question and the gate still passed without ever answering "FP32 vs PTQ vs QAT".
EXPECTED_RQ1_CONDITIONS = list(CORE_CONDITIONS)
EXPECTED_RQ2_CONDITIONS = ["best_fp32", "fp32_ft_control", "ft_ptq_int8", "ptq_int8", "qat_int8"]
REQUIRED_RQ1 = ["QWK", "MacroF1", "ShiftL1", "CosAgree", "BenefitCorr"]
REQUIRED_RQ2 = ["QWK", "Accuracy", "MacroF1", "CheckpointSize_MB", "Latency_median_ms",
                "DualViewGain_G_internal"]
_present_conditions = set(RAW.condition)
_missing_conditions = {"RQ1": [c for c in EXPECTED_RQ1_CONDITIONS if c not in _present_conditions],
                       "RQ2": [c for c in EXPECTED_RQ2_CONDITIONS if c not in _present_conditions]}

def _missing_columns(conds, required):
    out = {}
    for c in conds:
        if c not in _present_conditions: continue
        sub = RAW[RAW.condition == c]
        for col in required:
            if col not in sub.columns or sub[col].isna().all():
                out.setdefault(c, []).append(col)
    return out

_missing_rq1 = _missing_columns(EXPECTED_RQ1_CONDITIONS, REQUIRED_RQ1)
_missing_rq2 = _missing_columns(EXPECTED_RQ2_CONDITIONS, REQUIRED_RQ2)
_rq_complete = not (_missing_rq1 or _missing_rq2
                    or _missing_conditions["RQ1"] or _missing_conditions["RQ2"])
record_gate("Gate_RQ_Completeness", _rq_complete,
            f"missing conditions={_missing_conditions} | RQ1 missing columns={_missing_rq1 or 'none'} "
            f"| RQ2 missing columns={_missing_rq2 or 'none'}",
            blocking=not PREFLIGHT)

# Gate 3 is split, because the earlier single gate failed the ENTIRE study whenever ANY condition
# failed to predict a rare grade -- including deliberately weak ablations, single-view baselines and
# quantized variants, whose collapse is a finding rather than a defect. The preflight duly reported
# "15 condition(s) flagged" and marked the run failed.
#
#   Gate3A  core dual-view students + the deployed model            -> BLOCKING
#   Gate3B  everything else (baselines, ablations, INT8, external)  -> reported, never blocking
GATE3A_CONDITIONS = [c for c in CORE_CONDITIONS + ["best_fp32"] if c in set(RAW.condition)]
_core_flags = [c for c in collapse_report if c["condition"] in GATE3A_CONDITIONS]
_core_never = [c for c in _core_flags if "NEVER predicted" in c["warnings"]]

# Finiteness is part of viability: a NaN QWK or a NaN loss is not a "weak model", it is a broken run.
_core_rows = RAW[RAW.condition.isin(GATE3A_CONDITIONS)]
_nonfinite = [f"{r['condition']}|s{r['seed']}" for _, r in _core_rows.iterrows()
              if not (np.isfinite(r.get("QWK", np.nan)) and np.isfinite(r.get("MacroF1", np.nan)))]
_distinct_ok = []
for _, r in _core_rows.iterrows():
    rec = PRED_STORE.get((r["condition"], r["seed"], r["quantization"]))
    if rec is not None and len(np.unique(rec["y_pred"])) <= 2:
        _distinct_ok.append(f"{r['condition']}|s{r['seed']}")

_gate3a_ok = not (_core_never or _nonfinite or _distinct_ok)
record_gate("Gate3A_CoreStudentViability", _gate3a_ok,
            f"core conditions {GATE3A_CONDITIONS}: {len(_core_never)} with an unreachable grade, "
            f"{len(_distinct_ok)} collapsed to <=2 distinct grades {_distinct_ok[:3]}, "
            f"{len(_nonfinite)} with non-finite QWK/Macro-F1 {_nonfinite[:3]}",
            blocking=not PREFLIGHT)

_diag = [c for c in collapse_report if c["condition"] not in GATE3A_CONDITIONS]
record_gate("Gate3B_DiagnosticCollapseWarnings", True,
            f"{len(_diag)} non-core condition(s) flagged (baselines/ablations/INT8) -- reported as "
            "findings, never blocking: "
            + ", ".join(sorted({c['condition'] for c in _diag})[:6]))

In [ ]:
# ---- Aggregate across seeds (mean/SD + median/IQR) ----
NUMERIC = [c for c in RAW.columns if RAW[c].dtype.kind in "fc" and c not in ("seed",)]
agg_mean = RAW.groupby("condition")[NUMERIC].agg(["mean", "std", "median",
                                                   lambda x: x.quantile(.75) - x.quantile(.25)])
agg_mean.columns = ["_".join([a, b if b != "<lambda_0>" else "iqr"]) for a, b in agg_mean.columns]
agg_mean.to_csv(f"{METRICS_DIR}/all_conditions_aggregated.csv")

summary_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "MAE", "SevereErrorRate"]
tbl = RAW.groupby("condition")[summary_cols].agg(["mean", "std"]).round(4)
print(tbl.to_string())

## 34 — Statistical analysis

**Hierarchical paired cluster bootstrap over MATCHED seeds.** Each replicate resamples eye-level
clusters with replacement, resamples the **seed pairs** with replacement, and averages the paired
per-seed difference

```
delta_b = (1/S) * sum_s [ M(A_s, b) − M(B_s, b) ]        with seed s the SAME on both sides
```

Two earlier defects are closed by this: (a) it no longer compares CSD's *best* seed against a
baseline's *first* seed — a mismatch that also imported selection bias, since the best seed was
chosen on validation; (b) it no longer draws the two sides' seed indices independently, which turned
a nominally paired difference into an unpaired one (CSD seed 42 against logit-KD seed 8888). Where
two conditions genuinely have **disjoint** seed lists — a 3-seed QAT set against the single
validation-selected FP32 model — pairing falls back to positional cycling and every affected row is
stamped `matched_seeds=False` plus an explanatory `note`.

**p-values come from a paired cluster permutation test**, not from a bootstrap sign proportion.
Within each cluster the two methods' predictions are exchanged at random; the null is exchangeability
of the method labels. Holm correction is applied to the primary QWK comparisons only.

Clustering is at DRTiD's **record/eye** level (its public metadata exposes no patient key) and at
DeepDRiD's **patient** level (documented). Reported as such — never as "patient-clustered" for DRTiD.

In [ ]:
from scipy import stats as sps


def _stack_seed_preds(condition, seeds, quantization="FP32"):
    """Returns (y_true, clusters, {seed: y_pred}) for the seeds actually present."""
    y_true = clusters = None
    preds = {}
    for sd in seeds:
        rec = PRED_STORE.get((condition, sd, quantization))
        if rec is None: continue
        if y_true is None:
            y_true, clusters = rec["y_true"], rec["cluster_ids"]
        else:
            if not np.array_equal(y_true, rec["y_true"]):
                raise AssertionError(f"{condition} seed {sd} has a different sample order -- pairing invalid")
        preds[sd] = rec["y_pred"]
    return y_true, clusters, preds

def _seed_pairs(pa, pb):
    """MATCHED-seed pairing: seed s of A is compared against seed s of B.

    Only when the two seed lists are disjoint -- e.g. a 3-seed QAT set against the single
    validation-selected FP32 model -- do we fall back to positional cycling, and that fallback is
    recorded on every row so it can never be mistaken for a matched comparison.
    """
    common = sorted(set(pa) & set(pb))
    if common:
        return [(sd, sd) for sd in common], True
    # With the matched per-seed RQ2 design (FP32_s -> PTQ_s / QAT_s / FP32FT_s / FT-PTQ_s) and RQ1's
    # shared SEEDS_CORE, every pre-registered comparison has common seeds, so this branch should be
    # unreachable. It is kept only so an unexpected configuration degrades loudly instead of
    # crashing -- and every row it produces is stamped matched_seeds=False.
    print(f"  !! WARNING: no common seeds between the two conditions ({sorted(pa)} vs {sorted(pb)}). "
          "Falling back to positional pairing -- this is NOT a matched-seed comparison.")
    la, lb = sorted(pa), sorted(pb)
    n = max(len(la), len(lb))
    return [(la[i % len(la)], lb[i % len(lb)]) for i in range(n)], False

def hierarchical_paired_bootstrap(cond_a, cond_b, seeds_a, seeds_b, B=BOOTSTRAP_B,
                                   alpha=BOOTSTRAP_ALPHA, rng_seed=0, quant_a="FP32", quant_b="FP32"):
    """Hierarchical paired cluster bootstrap over MATCHED seeds.

    One replicate: resample eye-level clusters with replacement, resample the SEED PAIRS with
    replacement, and average the paired per-seed difference

        delta_b = (1/S) * sum_over_sampled_pairs [ M(A_s, b) - M(B_s, b) ]

    The earlier version drew the two conditions' seed indices independently (`ia`, `ib`), so a
    replicate could contrast CSD seed 42 against logit-KD seed 8888 and report the result as paired.
    That both loses power and misdescribes what the interval covers.
    """
    yt_a, cl, pa = _stack_seed_preds(cond_a, seeds_a, quant_a)
    yt_b, _,  pb = _stack_seed_preds(cond_b, seeds_b, quant_b)
    if yt_a is None or yt_b is None or not pa or not pb: return None
    if not np.array_equal(yt_a, yt_b):
        raise AssertionError("paired bootstrap requires identical sample order across conditions")

    pairs, matched = _seed_pairs(pa, pb)
    uniq = np.unique(cl)
    idx_by_cluster = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed)
    fns = _metric_fns()
    diffs = {k: np.empty(B) for k in fns}

    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_cluster[c] for c in pick])
        yt = yt_a[idx]
        sel = rng.integers(0, len(pairs), size=len(pairs))     # resample PAIRS, never the two sides apart
        for k, fn in fns.items():
            diffs[k][b] = float(np.mean([fn(yt, pa[pairs[j][0]][idx]) - fn(yt, pb[pairs[j][1]][idx])
                                         for j in sel]))

    out = {}
    for k, d in diffs.items():
        lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        out[k] = {"mean_diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi),
                  "excludes_zero": bool(lo > 0 or hi < 0),
                  "n_seeds_a": len(pa), "n_seeds_b": len(pb),
                  "n_seed_pairs": len(pairs), "matched_seeds": bool(matched)}
    return out

def paired_cluster_permutation_test(cond_a, cond_b, seeds_a, seeds_b, metric="QWK",
                                     P=BOOTSTRAP_B, rng_seed=0, quant_a="FP32", quant_b="FP32"):
    """Exchange the two methods' predictions within each cluster at random. Null = method label is
    exchangeable. Two-sided p with the standard +1 correction so p is never exactly 0. Seeds are
    paired with the SAME rule the bootstrap uses."""
    yt, cl, pa = _stack_seed_preds(cond_a, seeds_a, quant_a)
    _,  _,  pb = _stack_seed_preds(cond_b, seeds_b, quant_b)
    if yt is None or not pa or not pb: return None
    fn = _metric_fns()[metric]

    sp, matched = _seed_pairs(pa, pb)
    pairs = [(pa[a_], pb[b_]) for a_, b_ in sp]
    obs = float(np.mean([fn(yt, xa) - fn(yt, xb) for xa, xb in pairs]))

    uniq = np.unique(cl)
    masks = {c: (cl == c) for c in uniq}
    rng = np.random.default_rng(rng_seed)
    count = 0
    for _ in range(P):
        swap = rng.random(len(uniq)) < 0.5
        sel = np.zeros(len(cl), dtype=bool)
        for c, sw in zip(uniq, swap):
            if sw: sel |= masks[c]
        d = []
        for xa, xb in pairs:
            ya = np.where(sel, xb, xa)
            yb = np.where(sel, xa, xb)
            d.append(fn(yt, ya) - fn(yt, yb))
        if abs(np.mean(d)) >= abs(obs) - 1e-12:
            count += 1
    return {"observed_diff": obs, "p_perm": float((count + 1) / (P + 1)),
            "n_permutations": P, "matched_seeds": bool(matched)}

def holm_correction(pvals, names, alpha=0.05):
    pvals = list(pvals); m = len(pvals)
    order = np.argsort(pvals); adj = [None] * m; running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvals[i])); adj[i] = running
    return {names[i]: {"p_raw": float(pvals[i]), "p_holm": float(adj[i]),
                       "significant_holm": bool(adj[i] < alpha)} for i in range(m)}

In [ ]:
# Permutations are expensive; keep the preflight cheap but exercise the exact same code path.
_B = 300 if PREFLIGHT else BOOTSTRAP_B
_P = 300 if PREFLIGHT else 5000
# Every RQ2 condition now spans the SAME seed list, so _seed_pairs() always finds common seeds and
# the positional-pairing fallback is never exercised.
SEEDS_OF = {"best_fp32": RQ2_SEEDS, "fp32_ft_control": RQ2_SEEDS, "fp32_ft_plain": RQ2_SEEDS,
            "ft_ptq_int8": RQ2_SEEDS, "ptq_int8": RQ2_SEEDS, "qat_int8": RQ2_SEEDS,
            "ptq_int8_pt2e": [BEST_SEED]}

stat_rows, primary_p, primary_names, primary_family = [], [], [], []
for rq, pairs in PREREGISTERED_COMPARISONS.items():
    for a, b in pairs:
        sa = SEEDS_OF.get(a, SEEDS_CORE); sb = SEEDS_OF.get(b, SEEDS_CORE)
        qa, qb = QUANT_OF.get(a, "FP32"), QUANT_OF.get(b, "FP32")
        try:
            res = hierarchical_paired_bootstrap(a, b, sa, sb, B=_B, quant_a=qa, quant_b=qb)
        except AssertionError as e:
            print(f"  skip {rq} {a} vs {b}: {e}"); continue
        if res is None:
            print(f"  skip {rq} {a} vs {b}: predictions unavailable"); continue
        perm = paired_cluster_permutation_test(a, b, sa, sb, "QWK", P=_P, quant_a=qa, quant_b=qb)
        for metric, r in res.items():
            row = {"RQ": rq, "comparison": f"{a}_vs_{b}", "metric": metric,
                   "cluster_level": "record_eye(DRTiD)", **r}
            if not r.get("matched_seeds", True):
                row["note"] = "seed lists disjoint -- positional pairing, NOT a matched-seed comparison"
            if metric == "QWK" and perm:
                row.update({"p_perm": perm["p_perm"], "n_permutations": perm["n_permutations"]})
                primary_p.append(perm["p_perm"]); primary_names.append(f"{rq}:{a}_vs_{b}")
                primary_family.append(rq)
            stat_rows.append(row)
        q = res["QWK"]
        print(f"  {rq:4s} {a} vs {b}: dQWK={q['mean_diff']:+.4f} [{q['ci_low']:+.4f},{q['ci_high']:+.4f}] "
              f"seeds={q['n_seeds_a']}v{q['n_seeds_b']}" + (f" p_perm={perm['p_perm']:.4f}" if perm else ""))

STATS = pd.DataFrame(stat_rows)
if primary_p:
    # Holm is applied WITHIN each pre-registered family, not across both. RQ1 (3 hypotheses about
    # distillation) and RQ2 (5 about quantization) are separate questions declared in advance;
    # pooling them would penalise each for the other's existence. Both families are corrected, so
    # this is not a relaxation of multiplicity control -- just the right partition of it.
    holm = {}
    for fam in sorted(set(primary_family)):
        idx = [i for i, f in enumerate(primary_family) if f == fam]
        fam_res = holm_correction([primary_p[i] for i in idx], [primary_names[i] for i in idx])
        for k, v in fam_res.items():
            holm[k] = {**v, "family": fam, "family_size": len(idx)}
    key = STATS.apply(lambda r: f"{r['RQ']}:{r['comparison']}", axis=1)
    STATS["p_holm"] = [holm.get(k, {}).get("p_holm", np.nan) if m == "QWK" else np.nan
                       for k, m in zip(key, STATS["metric"])]
    STATS["significant_holm"] = [holm.get(k, {}).get("significant_holm", None) if m == "QWK" else None
                                 for k, m in zip(key, STATS["metric"])]
    STATS["holm_family"] = [holm.get(k, {}).get("family", "") if m == "QWK" else ""
                            for k, m in zip(key, STATS["metric"])]
    STATS["holm_family_size"] = [holm.get(k, {}).get("family_size", np.nan) if m == "QWK" else np.nan
                                 for k, m in zip(key, STATS["metric"])]
    print("Holm correction applied WITHIN each family: "
          + ", ".join(f"{f}={sum(1 for x in primary_family if x == f)} hypotheses"
                      for f in sorted(set(primary_family))))
STATS.to_csv(f"{TABLES_DIR}/table_05_statistical_tests.csv", index=False)
print(str(len(STATS)) + " comparisons saved (bootstrap B=" + str(_B) + ", permutations=" + str(_P) + ", Holm on primary QWK).")
print("A difference whose CI includes zero is NOT a claim, regardless of the point estimate.")

## 35 — DeepDRiD external validation (frozen)

Models are **frozen** before this section: no fine-tuning, no threshold tuning, no model selection,
no method change based on what happens here. A drop versus DRTiD is a domain-shift finding to
report, not something to engineer away.

### Set-C is the confirmatory test set

DeepDRiD ships three regular-fundus partitions. The challenge used Set-A/Set-B during development and
**Set-C (`Online-Challenge1&2-Evaluation`) for final evaluation**, and this copy of the dataset
includes `Challenge1_labels.xlsx`, so Set-C can be scored locally.

| Partition | Role here | Size |
|---|---|---|
| **Set-C** `Online-Challenge1&2-Evaluation` | **PRIMARY — confirmatory, untouched** | 100 patients / 200 eyes |
| Set-B `regular-fundus-validation` | secondary external | 100 patients / 200 eyes |
| Set-A `regular-fundus-training` | supplementary external | 300 patients / 597 usable eyes |
| pooled A+B | supplementary | 399 patients / 797 eyes |

This matters for honesty: Set-B aggregate performance was **already inspected during the first
preflight**, so Set-B is no longer untouched across the project lifecycle. Set-C has never been
looked at. Patient IDs confirm the partitions are disjoint (Set-A 1–330, Set-B 265–433, Set-C
347–500, with **zero** ID overlap between any pair), so Set-C is a genuine held-out cohort.

Set-B and Set-A therefore get the honest label *"external validation"*; only Set-C may be described
as *"confirmatory evaluation on a partition never inspected during development"*.

### Exclusion audit

The first preflight silently produced 597 eyes from Set-A instead of the documented 600. The loader
now writes `deepdrid_exclusion_audit.csv` naming every dropped eye and why. The cause is known and
recorded: patients **77** and **164** have `left_eye_DR_Level` / `right_eye_DR_Level` **swapped**
relative to the image laterality, and patient 164's left eye has only one field. Those eyes are
excluded rather than repaired, because inferring a grade from the opposite column is an assumption,
not data — and Set-A is supplementary anyway.

### Field ordering stays pre-registered

**Documented ambiguity:** DeepDRiD's public CSVs contain no column stating which of `_1`/`_2` is
macula- vs disc-centred (`Field definition` is an image-quality score, not a field-type label).
Both orderings are evaluated and both are reported. The preflight happened to score slightly higher
under the reverse ordering — that is **not** a reason to switch, because choosing an ordering from
observed performance is post-hoc selection. `_1=macula` remains primary; the reverse remains a
sensitivity analysis.

### Patient-clustered confidence intervals

DeepDRiD exposes a real `patient_id`, so external results get a **patient-clustered bootstrap**
(B=10,000) for QWK / Accuracy / Macro-F1 / MAE / Severe-Error Rate, plus paired PTQ−FP32 and
QAT−FP32 intervals on the primary partition.

In [ ]:
class DeepDRiDDualViewDataset(Dataset):
    """One record per EYE, built from the ACTUAL `image_id` / `image_path` values in DeepDRiD's CSV
    rather than from a guessed `pid/pid_l1.jpg` pattern.

    Every eye that does NOT become a record is recorded with a reason in `self.exclusions`, so a
    partition that comes up short is explained rather than noticed. The first preflight produced 597
    of Set-A's documented 600 eyes with no explanation anywhere.
    """

    EXCLUSION_REASONS = ("MISSING_VIEW", "MISSING_GRADE", "INVALID_GRADE", "PATH_NOT_FOUND",
                         "UNEXPECTED_VIEW_COUNT")

    def __init__(self, root, subsets=("regular-fundus-training", "regular-fundus-validation"),
                 transform=None, field_order="_1=macula"):
        self.transform = transform or eval_transform
        self.field_order = field_order
        recs, excl = [], []

        for sub in subsets:
            csv = f"{root}/{sub}/{sub}.csv"
            if not os.path.exists(csv):
                continue
            df = pd.read_csv(csv)
            img_root = f"{root}/{sub}/Images"

            # index every image on disk once, by basename -- robust to layout differences
            disk = {}
            for dp, _, fns in os.walk(img_root):
                for fn in fns:
                    if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                        disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))

            def resolve(image_id, image_path):
                if isinstance(image_path, str):
                    rel = image_path.replace(chr(92), "/").lstrip("/")
                    cand = os.path.join(root, rel)
                    if os.path.exists(cand):
                        return cand
                return disk.get(str(image_id))

            for pid, grp in df.groupby("patient_id"):
                for eye, col in (("l", "left_eye_DR_Level"), ("r", "right_eye_DR_Level")):
                    sub_g = grp[grp["image_id"].astype(str).str.contains(f"_{eye}", regex=False)]
                    ids_all = ",".join(sub_g["image_id"].astype(str)) if len(sub_g) else ""
                    def _drop(reason, grade=None, paths="n/a", excluded=True):
                        # `excluded=False` records an ANOMALY on a record that is still used, so the
                        # audit never claims an eye was dropped when it was not.
                        excl.append({"subset": sub, "patient_id": int(pid), "eye": eye,
                                     "reason": reason, "excluded": bool(excluded),
                                     "n_views": int(len(sub_g)), "image_ids": ids_all,
                                     "grade": grade, "path_status": paths})
                    if len(sub_g) == 0:
                        continue                       # this eye simply is not in the CSV
                    if len(sub_g) < 2:
                        _drop("MISSING_VIEW"); continue
                    lvl = sub_g[col].dropna() if col in sub_g else pd.Series(dtype=float)
                    if lvl.empty:
                        # Known DeepDRiD quirk: for patients 77 and 164 the two level columns are
                        # swapped relative to image laterality, so this eye's own column is empty.
                        # Recovering the grade from the opposite column would be an assumption, not
                        # data, so the eye is excluded and named here instead.
                        _drop("MISSING_GRADE"); continue
                    grade = int(lvl.iloc[0])
                    if not (0 <= grade < NUM_CLASSES):
                        _drop("INVALID_GRADE", grade=grade); continue
                    if len(sub_g) > 2:
                        # Real case in Set-A: patient 164's right eye has three images
                        # (164_r3, 164_r4, 164_r1). The eye is KEPT and the two lowest field indices
                        # are used; the anomaly is recorded so the pairing is never silent.
                        _drop("UNEXPECTED_VIEW_COUNT", grade=grade, excluded=False)
                    # order by the trailing field index in image_id (…_l1 before …_l2)
                    sub_g = sub_g.assign(_k=sub_g["image_id"].astype(str).str[-1]).sort_values("_k")
                    ids = sub_g["image_id"].astype(str).tolist()[:2]
                    paths = [resolve(i, sub_g[sub_g["image_id"].astype(str) == i]["image_path"].iloc[0]
                                     if "image_path" in sub_g else None) for i in ids]
                    if any(pp is None for pp in paths):
                        _drop("PATH_NOT_FOUND", grade=grade,
                              paths=";".join("missing" if p is None else "ok" for p in paths))
                        continue
                    recs.append({"patient_id": int(pid), "eye": eye,
                                 "img_field1": paths[0], "img_field2": paths[1],
                                 "id_field1": ids[0], "id_field2": ids[1], "grade": grade})
        self.df = pd.DataFrame(recs)
        self.exclusions = pd.DataFrame(excl)
        self.unresolved = int((self.exclusions.reason == "PATH_NOT_FOUND").sum()) if len(self.exclusions) else 0
        if len(self.exclusions):
            _dropped = self.exclusions[self.exclusions.excluded]
            print(f"  DeepDRiD[{'+'.join(subsets)}]: {len(_dropped)} eye(s) EXCLUDED "
                  f"{_dropped.reason.value_counts().to_dict() if len(_dropped) else {}}"
                  f" | {len(self.exclusions) - len(_dropped)} anomaly(ies) recorded but kept")

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = ((r["img_field1"], r["img_field2"]) if self.field_order == "_1=macula"
                else (r["img_field2"], r["img_field1"]))
        return {"macula": self.transform(image=_rgb(a))["image"],
                "disc":   self.transform(image=_rgb(b))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["patient_id"])}   # DeepDRiD DOES document patient identity


class DeepDRiDSetCDataset(Dataset):
    """Set-C: DeepDRiD's `Online-Challenge1&2-Evaluation` partition.

    Its layout differs from Set-A/Set-B -- labels arrive as ONE ROW PER IMAGE in
    `Challenge1_labels.xlsx` (`image_id`, `DR_Levels`) with no patient_id or image_path column, and
    the images sit under `Images/<patient>/<image_id>.jpg`. Patient and eye are therefore parsed out
    of the image_id (`347_l1` -> patient 347, eye l, field 1).

    This is the partition the challenge reserved for final evaluation, and nothing in this project
    has looked at it, which is what makes it the confirmatory set.
    """
    SUBDIR = "Online-Challenge1&2-Evaluation"

    def __init__(self, root, transform=None, field_order="_1=macula"):
        self.transform = transform or eval_transform
        self.field_order = field_order
        base = f"{root}/{self.SUBDIR}"
        recs, excl = [], []
        lab = f"{base}/Challenge1_labels.xlsx"
        self.available = os.path.exists(lab)
        if not self.available:
            self.df, self.exclusions, self.unresolved = pd.DataFrame(), pd.DataFrame(), 0
            print(f"  DeepDRiD Set-C labels not found at {lab}")
            return

        df = pd.read_excel(lab)
        gcol = "DR_Levels" if "DR_Levels" in df.columns else df.columns[-1]
        df["image_id"] = df["image_id"].astype(str)
        df["patient_id"] = df["image_id"].str.split("_").str[0]
        df["eye"] = df["image_id"].str.extract(r"_([lr])")[0]
        df["field"] = df["image_id"].str.extract(r"_[lr](\d+)$")[0]

        disk = {}
        for dp, _, fns in os.walk(f"{base}/Images"):
            for fn in fns:
                if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                    disk.setdefault(os.path.splitext(fn)[0], os.path.join(dp, fn))

        for (pid, eye), grp in df.groupby(["patient_id", "eye"]):
            ids_all = ",".join(grp["image_id"])
            def _drop(reason, grade=None, paths="n/a", excluded=True):
                excl.append({"subset": "setC", "patient_id": pid, "eye": eye, "reason": reason,
                             "excluded": bool(excluded), "n_views": int(len(grp)),
                             "image_ids": ids_all, "grade": grade, "path_status": paths})
            if len(grp) < 2:
                _drop("MISSING_VIEW"); continue
            lv = grp[gcol].dropna()
            if lv.empty:
                _drop("MISSING_GRADE"); continue
            if lv.nunique() > 1:
                # The two fields of one eye must agree on that eye's grade.
                _drop("INVALID_GRADE", grade=";".join(map(str, sorted(lv.unique())))); continue
            grade = int(lv.iloc[0])
            if not (0 <= grade < NUM_CLASSES):
                _drop("INVALID_GRADE", grade=grade); continue
            g = grp.sort_values("field")
            ids = g["image_id"].tolist()[:2]
            paths = [disk.get(i) for i in ids]
            if any(p is None for p in paths):
                _drop("PATH_NOT_FOUND", grade=grade,
                      paths=";".join("missing" if p is None else "ok" for p in paths)); continue
            recs.append({"patient_id": int(pid), "eye": eye, "img_field1": paths[0],
                         "img_field2": paths[1], "id_field1": ids[0], "id_field2": ids[1],
                         "grade": grade})
        self.df = pd.DataFrame(recs)
        self.exclusions = pd.DataFrame(excl)
        self.unresolved = int((self.exclusions.reason == "PATH_NOT_FOUND").sum()) if len(self.exclusions) else 0
        if len(self.exclusions):
            _dropped = self.exclusions[self.exclusions.excluded]
            print(f"  DeepDRiD Set-C: {len(_dropped)} eye(s) EXCLUDED "
                  f"{_dropped.reason.value_counts().to_dict() if len(_dropped) else {}}"
                  f" | {len(self.exclusions) - len(_dropped)} anomaly(ies) recorded but kept")

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        a, b = ((r["img_field1"], r["img_field2"]) if self.field_order == "_1=macula"
                else (r["img_field2"], r["img_field1"]))
        return {"macula": self.transform(image=_rgb(a))["image"],
                "disc":   self.transform(image=_rgb(b))["image"],
                "label":  torch.tensor(int(r["grade"]), dtype=torch.long),
                "cluster_id": int(r["patient_id"])}

# DeepDRiD partitions, with their roles pre-registered.
#   setC  -- Online-Challenge1&2-Evaluation. The challenge's own final-evaluation partition, and the
#            only one this project has never inspected. PRIMARY confirmatory result.
#   setB  -- regular-fundus-validation. Aggregate performance was already seen during preflight, so
#            it is honest external validation, not untouched confirmation.
#   setA  -- regular-fundus-training, and A+B pooled. Supplementary.
DEEPDRID_SUBSETS = {"setC":       None,                                   # own loader class
                    "validation": ("regular-fundus-validation",),
                    "training":   ("regular-fundus-training",),
                    "pooled":     ("regular-fundus-training", "regular-fundus-validation")}
DEEPDRID_PRIMARY_SUBSET = "setC"
DEEPDRID_SUBSET_ROLE = {"setC": "PRIMARY_CONFIRMATORY", "validation": "external_validation",
                        "training": "supplementary", "pooled": "supplementary"}
# Documented, from the DeepDRiD release: Set-A 300 patients/600 eyes, Set-B 100/200, Set-C 100/200.
DEEPDRID_EXPECTED = {"setC": (100, 200), "validation": (100, 200), "training": (300, 600)}

EXTERNAL_ROWS, EXT_BOOT_ROWS = [], []
# Per-sample predictions on the PRIMARY external partition, keyed like PRED_STORE. DeepDRiD exposes a
# real patient_id, so external intervals can be clustered at the patient level -- the correct unit,
# since a patient's two eyes are not independent observations.
EXT_PRED_STORE = {}
if DEEPDRID_ROOT is None:
    print("DeepDRiD not found -- external validation SKIPPED (reported, not silently omitted).")
    record_gate("Gate9_ExternalValidation", False, "DeepDRiD dataset not present in this runtime",
                blocking=not PREFLIGHT)
else:
    ext_models = [("teacher", "-", TEACHER, "FP32", False),
                  ("best_fp32", BEST_SEED, BEST_FP32_MODEL, "FP32", False)]
    if BEST_CONDITION != "dual_csd":
        ext_models.append(("best_csd_fp32", BEST_CSD_SEED, load_student(BEST_CSD_CKPT), "FP32", False))
    if PTQ_OK and BEST_SEED in PTQ_MODELS:
        ext_models.append(("ptq_int8", BEST_SEED, PTQ_MODELS[BEST_SEED], "PTQ_INT8", True))
    if FT_PTQ_MODELS:
        _fs = BEST_SEED if BEST_SEED in FT_PTQ_MODELS else sorted(FT_PTQ_MODELS)[0]
        ext_models.append(("ft_ptq_int8", _fs, FT_PTQ_MODELS[_fs], "FT_PTQ_INT8", True))
    # The QAT row carries QAT_DEPLOY_SEED -- the seed that model was actually trained with.
    if QAT_OK and QAT_DEPLOY_SEED is not None:
        ext_models.append(("qat_int8", QAT_DEPLOY_SEED, QAT_MODEL, "QAT_INT8", True))

    _ds_cache = {}
    def _get_ext_ds(name):
        if name not in _ds_cache:
            if name == "setC":
                _ds_cache[name] = DeepDRiDSetCDataset(DEEPDRID_ROOT, transform=eval_transform)
            else:
                _ds_cache[name] = DeepDRiDDualViewDataset(DEEPDRID_ROOT,
                                                          subsets=DEEPDRID_SUBSETS[name],
                                                          transform=eval_transform)
        return _ds_cache[name]

    # ---- exclusion audit: every eye that did NOT become a record, and why ----
    _excl = []
    for _n in DEEPDRID_SUBSETS:
        _d = _get_ext_ds(_n)
        if len(getattr(_d, "exclusions", [])):
            _e = _d.exclusions.copy(); _e["partition"] = _n; _excl.append(_e)
    DEEPDRID_EXCLUSIONS = pd.concat(_excl, ignore_index=True) if _excl else pd.DataFrame(
        columns=["partition", "patient_id", "eye", "reason", "excluded", "n_views", "image_ids",
                 "grade", "path_status"])
    DEEPDRID_EXCLUSIONS.to_csv(f"{METRICS_DIR}/deepdrid_exclusion_audit.csv", index=False)

    _count_rows, _count_ok = [], True
    for _n, (_ep, _ee) in DEEPDRID_EXPECTED.items():
        _d = _get_ext_ds(_n)
        _gp, _ge = (int(_d.df.patient_id.nunique()), len(_d.df)) if len(_d.df) else (0, 0)
        _ok = (_gp == _ep and _ge == _ee)
        _count_ok &= _ok
        _count_rows.append({"partition": _n, "expected_patients": _ep, "got_patients": _gp,
                            "expected_eyes": _ee, "got_eyes": _ge, "matches_published": _ok,
                            "excluded_eyes": int(((DEEPDRID_EXCLUSIONS.partition == _n)
                                                  & DEEPDRID_EXCLUSIONS.excluded).sum())
                                             if len(DEEPDRID_EXCLUSIONS) else 0,
                            "anomalies_kept": int(((DEEPDRID_EXCLUSIONS.partition == _n)
                                                   & ~DEEPDRID_EXCLUSIONS.excluded).sum())
                                              if len(DEEPDRID_EXCLUSIONS) else 0})
    DEEPDRID_COUNTS = pd.DataFrame(_count_rows)
    DEEPDRID_COUNTS.to_csv(f"{TABLES_DIR}/table_06b_deepdrid_partition_counts.csv", index=False)
    print("DeepDRiD partition counts vs the published dataset description:")
    print(DEEPDRID_COUNTS.to_string(index=False))
    if len(DEEPDRID_EXCLUSIONS):
        print("Audit rows by reason:",
              DEEPDRID_EXCLUSIONS.groupby(["reason", "excluded"]).size().to_dict())
        print(DEEPDRID_EXCLUSIONS.head(12).to_string(index=False))
    # The PRIMARY partition must be complete; a short supplementary partition is explained, not fatal.
    _primary_ok = bool(DEEPDRID_COUNTS[DEEPDRID_COUNTS.partition == DEEPDRID_PRIMARY_SUBSET]
                       ["matches_published"].all())
    record_gate("Gate9c_DeepDRiD_ExclusionAudit", _primary_ok,
                f"primary partition '{DEEPDRID_PRIMARY_SUBSET}' complete={_primary_ok}; "
                f"{int(DEEPDRID_EXCLUSIONS.excluded.sum()) if len(DEEPDRID_EXCLUSIONS) else 0} eye(s) "
                f"excluded and {int((~DEEPDRID_EXCLUSIONS.excluded).sum()) if len(DEEPDRID_EXCLUSIONS) else 0} "
                "anomaly(ies) kept -- every one itemised in deepdrid_exclusion_audit.csv",
                blocking=not PREFLIGHT)

    # ---- pooling is only legitimate across patient-disjoint partitions ----
    _pat = {}
    for _n in ("training", "validation", "setC"):
        _d = _get_ext_ds(_n)
        _pat[_n] = set(_d.df.patient_id.tolist()) if len(_d.df) else set()
    _pairs = [("training", "validation"), ("training", "setC"), ("validation", "setC")]
    _ov = {f"{a}&{b}": len(_pat[a] & _pat[b]) for a, b in _pairs}
    _pat_counts = {k: len(v) for k, v in _pat.items()}
    record_gate("Gate9b_DeepDRiD_PartitionDisjoint", sum(_ov.values()) == 0,
                f"patients {_pat_counts} -> overlaps {_ov}"
                + ("" if sum(_ov.values()) == 0 else " -- overlapping rows are NOT independent"))

    for sub_name in DEEPDRID_SUBSETS:
        ds = _get_ext_ds(sub_name)
        if len(ds) == 0:
            print(f"  DeepDRiD[{sub_name}]: 0 usable eye-records"); continue
        for order in DEEPDRID_FIELD_ORDERS:
            ds.field_order = order                      # read inside __getitem__, so this is enough
            ld = make_loader(ds, 16, False)
            is_primary = (order == DEEPDRID_PRIMARY_FIELD_ORDER and sub_name == DEEPDRID_PRIMARY_SUBSET)
            print(f"\nDeepDRiD [{sub_name} | {order}]: {len(ds)} eyes, "
                  f"{ds.df.patient_id.nunique()} patients, grades {sorted(ds.df.grade.unique().tolist())}"
                  + ("   <-- PRE-REGISTERED PRIMARY (confirmatory)" if is_primary else ""))
            for name, seed, mdl, quant, on_cpu in ext_models:
                if on_cpu:
                    r = _predict_cpu_all(mdl, ld)
                    yt, yp, pc, pid = (r["y_true"], r["y_pred_dual"], r["p_dual"], r["cluster_ids"])
                else:
                    yt, yp, pc, pid = get_predictions(mdl, ld, DEVICE, "dual", return_clusters=True)
                m = compute_all_metrics(yt, yp, pc)
                EXTERNAL_ROWS.append({"condition": name, "seed": seed, "quantization": quant,
                                      "subset": sub_name, "field_order": order,
                                      "partition_role": DEEPDRID_SUBSET_ROLE.get(sub_name, "supplementary"),
                                      "role": "PRIMARY" if is_primary else "supplementary",
                                      # A pooled row is only a valid single sample if the partitions
                                      # it merges share no patients; every row carries the verdict so
                                      # a reader never has to assume it.
                                      "patient_disjoint_partitions": bool(sum(_ov.values()) == 0),
                                      "n_eyes": len(ds), "n_patients": int(ds.df.patient_id.nunique()), **m})
                _tag = f"external_{sub_name}_{order.replace('=', '')}"
                save_predictions(name, seed, yt, yp, pc, pid, quant, dataset="DeepDRiD", split=_tag)
                save_confusion(name, seed, yt, yp, tag=f"_DeepDRiD_{sub_name}_{order.replace('=', '')}")
                if is_primary:
                    EXT_PRED_STORE[(name, seed, quant)] = {"y_true": yt, "y_pred": yp, "cluster_ids": pid}
                w = check_prediction_collapse(yt, yp)
                print(f"  {name:14s} QWK={m['QWK']:.4f} Acc={m['Accuracy']:.4f} MacroF1={m['MacroF1']:.4f}"
                      + (f"  !! {'; '.join(w)}" if w else ""))

    if EXTERNAL_ROWS:
        EXT = pd.DataFrame(EXTERNAL_ROWS)
        EXT.to_csv(f"{TABLES_DIR}/table_06_external_validation_deepdrid.csv", index=False)
        record_gate("Gate9_ExternalValidation", True,
                    f"{EXT.condition.nunique()} models x {EXT.subset.nunique()} partitions x "
                    f"{EXT.field_order.nunique()} field orders; PRIMARY = "
                    f"{DEEPDRID_PRIMARY_SUBSET}/{DEEPDRID_PRIMARY_FIELD_ORDER} (confirmatory)",
                    blocking=not PREFLIGHT)
    else:
        record_gate("Gate9_ExternalValidation", False, "no usable DeepDRiD records assembled",
                    blocking=not PREFLIGHT)

EXT_DF = pd.DataFrame(EXTERNAL_ROWS) if EXTERNAL_ROWS else pd.DataFrame()

In [ ]:
# ---- Patient-clustered bootstrap on the PRIMARY external partition (Set-C) ----
# Point estimates on 200 eyes carry real uncertainty, and DeepDRiD gives us the correct clustering
# unit: a patient contributes two eyes that are not independent. Resampling PATIENTS (not eyes)
# propagates that, both for each model on its own and for the paired INT8-vs-FP32 differences.
def patient_cluster_bootstrap(y_true, y_pred, clusters, B=None, alpha=0.05, rng_seed=11):
    B = B or (300 if PREFLIGHT else 10000)
    fns = _metric_fns()
    uniq = np.unique(clusters)
    idx_by = {c: np.where(clusters == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed)
    draws = {k: np.empty(B) for k in fns}
    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by[c] for c in pick])
        for k, fn in fns.items():
            draws[k][b] = fn(y_true[idx], y_pred[idx])
    out = {}
    for k, d in draws.items():
        lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        out[k] = {"point": float(fns[k](y_true, y_pred)), "ci_low": float(lo), "ci_high": float(hi)}
    return out, len(uniq)

def paired_patient_bootstrap(cond_a, key_b, B=None, alpha=0.05, rng_seed=13):
    """Paired A-B on the same patients. Both models saw identical eyes in identical order."""
    B = B or (300 if PREFLIGHT else 10000)
    ka = next((k for k in EXT_PRED_STORE if k[0] == cond_a), None)
    kb = next((k for k in EXT_PRED_STORE if k[0] == key_b), None)
    if ka is None or kb is None: return None
    ra, rb = EXT_PRED_STORE[ka], EXT_PRED_STORE[kb]
    if not np.array_equal(ra["y_true"], rb["y_true"]):
        raise AssertionError("external pairing requires identical sample order")
    yt, cl = ra["y_true"], ra["cluster_ids"]
    uniq = np.unique(cl); idx_by = {c: np.where(cl == c)[0] for c in uniq}
    rng = np.random.default_rng(rng_seed); fns = _metric_fns()
    draws = {k: np.empty(B) for k in fns}
    for b in range(B):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by[c] for c in pick])
        for k, fn in fns.items():
            draws[k][b] = fn(yt[idx], ra["y_pred"][idx]) - fn(yt[idx], rb["y_pred"][idx])
    rows = []
    for k, d in draws.items():
        lo, hi = np.percentile(d, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        rows.append({"comparison": f"{cond_a}_vs_{key_b}", "metric": k,
                     "mean_diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi),
                     "excludes_zero": bool(lo > 0 or hi < 0)})
    return rows

EXT_CI_ROWS, EXT_PAIRED_ROWS = [], []
if EXT_PRED_STORE:
    _B = 300 if PREFLIGHT else 10000
    print(f"Patient-clustered bootstrap on {DEEPDRID_PRIMARY_SUBSET} "
          f"({DEEPDRID_PRIMARY_FIELD_ORDER}), B={_B}:")
    for (cond, seed, quant), rec in EXT_PRED_STORE.items():
        res, n_pat = patient_cluster_bootstrap(rec["y_true"], rec["y_pred"], rec["cluster_ids"], B=_B)
        for metric, r in res.items():
            EXT_CI_ROWS.append({"condition": cond, "seed": seed, "quantization": quant,
                                "subset": DEEPDRID_PRIMARY_SUBSET,
                                "field_order": DEEPDRID_PRIMARY_FIELD_ORDER,
                                "n_patients": n_pat, "metric": metric, **r})
        q = res["QWK"]
        print(f"  {cond:14s} QWK={q['point']:.4f} [{q['ci_low']:.4f}, {q['ci_high']:.4f}] "
              f"({n_pat} patients)")
    for _a, _b in [("ptq_int8", "best_fp32"), ("qat_int8", "best_fp32"),
                   ("ft_ptq_int8", "best_fp32"), ("qat_int8", "ptq_int8")]:
        try:
            _rows = paired_patient_bootstrap(_a, _b, B=_B)
        except AssertionError as e:
            print(f"  skip external {_a} vs {_b}: {e}"); _rows = None
        if _rows:
            EXT_PAIRED_ROWS.extend(_rows)
            _q = next(r for r in _rows if r["metric"] == "QWK")
            print(f"  {_a} vs {_b}: dQWK={_q['mean_diff']:+.4f} "
                  f"[{_q['ci_low']:+.4f}, {_q['ci_high']:+.4f}] credible={_q['excludes_zero']}")
    if EXT_CI_ROWS:
        pd.DataFrame(EXT_CI_ROWS).to_csv(f"{TABLES_DIR}/table_06c_external_patient_clustered_ci.csv",
                                          index=False)
    if EXT_PAIRED_ROWS:
        pd.DataFrame(EXT_PAIRED_ROWS).to_csv(f"{TABLES_DIR}/table_06d_external_paired_deltas.csv",
                                              index=False)
    record_gate("Gate9d_ExternalCIs", bool(EXT_CI_ROWS),
                f"{len(EXT_CI_ROWS)} patient-clustered intervals, "
                f"{len(EXT_PAIRED_ROWS)} paired external comparisons")
else:
    print("No primary-partition predictions stored -- external confidence intervals skipped.")
    record_gate("Gate9d_ExternalCIs", False, "primary external partition unavailable")

## 36–37 — Deployment export, parity checks & inference wrapper (Gate 8)

TorchScript is deprecated and is **not** used as the deployment path. Artifacts:
`checkpoint.pt` (state_dict), `model.pt2` (`torch.export`), `model.onnx`, `metadata.json`, plus
`model_object.pt` for INT8 models only. Export failures are reported as failures — never silently
downgraded to "gate passed".

**Quantized models face the same reload test as FP32.** Every model is rebuilt by a builder and then
loaded from `checkpoint.pt`. For INT8 the builder reconstructs the *quantized skeleton*
(fuse → prepare → convert) and `load_state_dict()` restores the packed INT8 weights with their
scales and zero-points — the supported way to restore a quantized model. Whole-module pickling is
kept only as a fallback, because eager quantized modules do not reliably round-trip through pickle
(reloading one raises `AttributeError: 'ConvReLU2d' object has no attribute '_modules'`).

The artifact on disk is loaded into a fresh object, checked for numeric parity against the in-memory
reference, checked for a valid grade range, and run 100 times. `Gate8d_ArtifactReload` reports
`deployment_verified` per model; the earlier version passed `builder=None` for INT8 and recorded
`reloaded_from_disk=False`, which meant PTQ/QAT were never demonstrated to be deployable at all.

In [ ]:
class DRVergeInference(nn.Module):
    """Export-friendly wrapper: two images in, cumulative threshold scores out."""
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, macula, disc): return self.model(macula, disc)["p_dual"]

def export_model(model, name, on_cpu_model=False, extra_meta=None, save_object=False):
    d = f"{MODELS_DIR}/{name}"; os.makedirs(d, exist_ok=True)
    m = (model if on_cpu_model else copy.deepcopy(model).to("cpu")).eval()
    ex_m, ex_d = _sample["macula"][:1].cpu(), _sample["disc"][:1].cpu()
    status = {"state_dict": False, "torch_export": False, "onnx": False,
              "onnx_parity_max_abs_diff": None, "export_error": None, "onnx_error": None}

    try:
        robust_torch_save(m.state_dict(), f"{d}/checkpoint.pt"); status["state_dict"] = True
    except Exception as e:
        status["export_error"] = f"state_dict: {e!r}"

    # Best-effort full pickled module, saved only for quantized models. It is a FALLBACK, not the
    # primary reload path: eager quantized modules do not reliably round-trip through plain pickle
    # (loading one raises AttributeError: 'ConvReLU2d' object has no attribute '_modules'), which is
    # why the primary path rebuilds the quantized skeleton and loads checkpoint.pt into it.
    status["model_object"] = False
    if save_object:
        try:
            robust_torch_save(m, f"{d}/model_object.pt"); status["model_object"] = True
        except Exception as e:
            status["model_object_error"] = repr(e)

    wrapper = DRVergeInference(m).eval()
    with torch.no_grad():
        ref = wrapper(ex_m, ex_d)

    try:
        ep = torch.export.export(wrapper, (ex_m, ex_d))
        torch.export.save(ep, f"{d}/model.pt2"); status["torch_export"] = True
    except Exception as e:
        status["export_error"] = f"torch.export: {e!r}"
        print(f"  [{name}] torch.export FAILED: {e!r}")

    try:
        torch.onnx.export(wrapper, (ex_m, ex_d), f"{d}/model.onnx",
                          input_names=["macula", "disc"], output_names=["p_cumulative"], dynamo=True)
        status["onnx"] = True
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(f"{d}/model.onnx", providers=["CPUExecutionProvider"])
            got = sess.run(None, {"macula": ex_m.numpy(), "disc": ex_d.numpy()})[0]
            status["onnx_parity_max_abs_diff"] = float(np.max(np.abs(got - ref.numpy())))
        except Exception as e:
            status["onnx_error"] = f"parity: {e!r}"
    except Exception as e:
        status["onnx_error"] = f"export: {e!r}"
        print(f"  [{name}] ONNX export FAILED: {e!r}")

    meta = {"model_name": name, "architecture": type(m).__name__, "dataset": "DRTiD",
            "grade_mapping": list(range(NUM_CLASSES)), "num_thresholds": NUM_THRESHOLDS,
            **PREPROCESSING_META, "torch_version": torch.__version__,
            "torchao_version": ENVIRONMENT.get("torchao"), "git_commit": ENVIRONMENT["git_commit"],
            "artifacts": {k: v for k, v in status.items()}, **(extra_meta or {})}
    save_json(meta, f"{d}/metadata.json")
    ok = status["state_dict"]
    print(f"  [{name}] state_dict={status['state_dict']} pt2={status['torch_export']} "
          f"onnx={status['onnx']} parity={status['onnx_parity_max_abs_diff']}")
    return d, status, meta

EXPORTS = {}
EXPORTS["teacher_fp32"] = export_model(TEACHER, "teacher_fp32",
    extra_meta={"role": "upper_bound_teacher", "training_seed": PRIMARY_SEED})
EXPORTS["best_student_fp32"] = export_model(BEST_FP32_MODEL, "best_student_fp32",
    extra_meta={"role": "deployment_candidate", "condition": BEST_CONDITION,
                "training_seed": BEST_SEED, "best_val_qwk": float(BEST_ROW["QWK"]), "quantization": "FP32"})
if BEST_CONDITION != "dual_csd":
    EXPORTS["best_csd_fp32"] = export_model(load_student(BEST_CSD_CKPT), "best_csd_fp32",
        extra_meta={"role": "best_csd_artifact", "condition": "dual_csd",
                    "training_seed": BEST_CSD_SEED, "quantization": "FP32"})
if PTQ_OK:
    EXPORTS["best_student_ptq_int8"] = export_model(PTQ_MODEL, "best_student_ptq_int8", on_cpu_model=True,
        save_object=True, extra_meta={"role": "deployment_int8", "quantization": "PTQ_INT8", **PTQ_INFO})
if FT_PTQ_MODELS:
    _ftp_seed = BEST_SEED if BEST_SEED in FT_PTQ_MODELS else sorted(FT_PTQ_MODELS)[0]
    EXPORTS["best_student_ft_ptq_int8"] = export_model(
        FT_PTQ_MODELS[_ftp_seed], "best_student_ft_ptq_int8", on_cpu_model=True, save_object=True,
        extra_meta={"role": "deployment_int8", "quantization": "FT_PTQ_INT8",
                    "training_seed": _ftp_seed, **FT_PTQ_INFOS.get(_ftp_seed, {})})
if QAT_OK:
    EXPORTS["best_student_qat_int8"] = export_model(QAT_MODEL, "best_student_qat_int8", on_cpu_model=True,
        save_object=True,
        extra_meta={"role": "deployment_int8", "quantization": "QAT_INT8",
                    "training_seed": QAT_DEPLOY_SEED,
                    "deploy_seed_selected_on": "DRTiD validation QWK", **QAT_INFO})

# Gate 8 is split so that one green "Export PASS" can no longer hide a failed ONNX export or a
# failed torch.export, which is exactly what the first preflight did (every ONNX export failed on a
# missing onnxscript, and both INT8 models failed torch.export, yet the single gate passed).
_state_ok = all(st["state_dict"] for _, st, _ in EXPORTS.values())
_onnx_names = [n for n, (_, st, _) in EXPORTS.items() if st["onnx"]]
_onnx_fp32_expected = [n for n in EXPORTS if "int8" not in n]
_onnx_fp32_ok = all(EXPORTS[n][1]["onnx"] for n in _onnx_fp32_expected)
_onnx_parity = {n: EXPORTS[n][1].get("onnx_parity_max_abs_diff") for n in _onnx_names}
_parity_ok = all(v is not None and v < 1e-3 for v in _onnx_parity.values()) if _onnx_names else False
_pt2_ok = sum(1 for _, st, _ in EXPORTS.values() if st["torch_export"])

record_gate("Gate8a_MandatoryArtifacts", _state_ok,
            f"{len(EXPORTS)} models; every state_dict written={_state_ok}",
            blocking=not PREFLIGHT)

# FP32 ONNX is claimed as a deployment path, so it is blocking for FP32 models. INT8 is NOT: eager
# quantized Conv2d carries packed params that torch.export cannot trace, which the preflight showed
# directly. The quantized deployment path is the skeleton+state_dict reload proven by Gate 8d.
record_gate("Gate8b_FP32_ONNX", bool(_onnx_fp32_ok and _parity_ok),
            f"ONNX exported for {len(_onnx_names)}/{len(EXPORTS)} models "
            f"(FP32 required: {_onnx_fp32_expected}); ONNX Runtime parity {_onnx_parity}",
            blocking=not PREFLIGHT)

record_gate("Gate8c_TorchExport_Optional", _pt2_ok > 0,
            f"torch.export (.pt2) succeeded for {_pt2_ok}/{len(EXPORTS)} models; INT8 failures are "
            "expected (packed quantized params are not exportable) and are not a deployment blocker")

In [ ]:
# ---- Deployment verification (spec 59): reload from disk and confirm it still behaves ----
def verify_deployment(model_dir, builder, on_cpu_reference=None, n_runs=100):
    """Proves the SAVED ARTIFACT works -- not the object still alive in RAM.

    Every model -- FP32 and INT8 alike -- is rebuilt by a `builder()` and then loaded from
    `checkpoint.pt`. For INT8 the builder reconstructs the *quantized skeleton* (fuse -> prepare ->
    convert), which is the supported way to restore a quantized model: `convert()` produces the
    module tree, `load_state_dict()` restores the packed INT8 weights and their qparams. Plain
    pickling of a quantized module is only a fallback here, because it does not reliably round-trip
    (loading one raises AttributeError: 'ConvReLU2d' object has no attribute '_modules').

    Either way the check is the same: load from disk into a fresh object, compare numerically against
    the in-memory reference, confirm the predicted grade is in range, and run repeatedly. The earlier
    version passed `builder=None` for quantized models and recorded `reloaded_from_disk = False`,
    which meant PTQ/QAT were never shown to be deployment-ready at all.
    """
    checks = {}
    ex_m, ex_d = _sample_pair[0][:1].cpu(), _sample_pair[1][:1].cpu()
    ck  = f"{model_dir}/checkpoint.pt"
    obj = f"{model_dir}/model_object.pt"
    checks["checkpoint_exists"] = os.path.exists(ck)
    checks["model_object_exists"] = os.path.exists(obj)

    ref_out = None
    if on_cpu_reference is not None:
        with torch.no_grad():
            o = on_cpu_reference(ex_m, ex_d)
            ref_out = (o["p_dual"] if isinstance(o, dict) else o).cpu()

    fresh, how = None, None
    if builder is not None and checks["checkpoint_exists"]:
        try:
            fresh = builder()                                   # brand new object, no shared state
            state = robust_torch_load(ck, map_location="cpu")
            fresh.load_state_dict(state if not isinstance(state, dict) or "model_state" not in state
                                  else state["model_state"])
            how = "rebuilt skeleton + state_dict from disk"
        except Exception as e:
            fresh = None
            checks["builder_reload_error"] = repr(e)
    if fresh is None and checks["model_object_exists"]:
        try:
            fresh = robust_torch_load(obj, map_location="cpu")   # fallback: whole pickled module
            how = "pickled module artifact (fallback)"
        except Exception as e:
            checks["reload_error"] = repr(e)

    if fresh is not None:
        try:
            fresh.eval()
            with torch.no_grad():
                o = fresh(ex_m, ex_d)
                got = (o["p_dual"] if isinstance(o, dict) else o).cpu()
            checks["reloaded_from_disk"] = True
            checks["reload_method"] = how
            g = int((got > 0.5).sum())
            checks["grade_in_range"] = bool(0 <= g <= NUM_CLASSES - 1)
            if ref_out is not None:
                checks["reload_max_abs_diff"] = float(torch.max(torch.abs(got - ref_out)))
                checks["reload_parity_ok"] = checks["reload_max_abs_diff"] < 1e-5
            with torch.no_grad():
                for _ in range(n_runs): fresh(ex_m, ex_d)
            checks["stable_over_100_runs"] = True
        except Exception as e:
            checks["reloaded_from_disk"] = False
            checks["reload_error"] = repr(e)
    else:
        checks["reloaded_from_disk"] = False
        checks.setdefault("reload_error", "neither checkpoint.pt+builder nor model_object.pt usable")

    pt2 = f"{model_dir}/model.pt2"
    if os.path.exists(pt2) and ref_out is not None:
        try:
            loaded = torch.export.load(pt2)
            with torch.no_grad(): got = loaded.module()(ex_m, ex_d)
            got = got["p_dual"] if isinstance(got, dict) else got
            checks["pt2_reload_max_abs_diff"] = float(torch.max(torch.abs(got.cpu() - ref_out)))
            checks["pt2_parity_ok"] = checks["pt2_reload_max_abs_diff"] < 1e-4
        except Exception as e:
            checks["pt2_parity_ok"] = False; checks["pt2_error"] = repr(e)

    checks["deployment_verified"] = bool(checks.get("reloaded_from_disk")
                                         and checks.get("stable_over_100_runs")
                                         and checks.get("grade_in_range", False)
                                         and checks.get("reload_parity_ok", True))
    return checks

def build_ptq_skeleton():
    """An architecturally identical INT8 model with placeholder qparams. Loading the saved
    state_dict into it restores the real packed weights, scales and zero-points."""
    m = _fresh_student_from(BEST_FP32_CKPT)
    m.backbone.fuse_model(qat=False)
    m.backbone = QuantizableBackbone(m.backbone)
    m.backbone.qconfig = get_default_qconfig(QUANT_ENGINE)
    prep = prepare(m, inplace=False)
    _calibrate(prep, 1)          # observers must see one batch before convert() can build the tree
    return convert(prep, inplace=False)

def build_qat_skeleton():
    m = _build_qat_prepared(BEST_FP32_CKPT, load_fp32=True)
    return convert(m.to("cpu").eval(), inplace=False)

DEPLOY_CHECKS = {}
_fp32_cpu = copy.deepcopy(BEST_FP32_MODEL).to("cpu").eval()
DEPLOY_CHECKS["best_student_fp32"] = verify_deployment(
    EXPORTS["best_student_fp32"][0],
    builder=lambda: DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS),
    on_cpu_reference=_fp32_cpu)
if PTQ_OK:
    DEPLOY_CHECKS["best_student_ptq_int8"] = verify_deployment(
        EXPORTS["best_student_ptq_int8"][0], builder=build_ptq_skeleton, on_cpu_reference=PTQ_MODEL)
if QAT_OK:
    DEPLOY_CHECKS["best_student_qat_int8"] = verify_deployment(
        EXPORTS["best_student_qat_int8"][0], builder=build_qat_skeleton, on_cpu_reference=QAT_MODEL)

# BLOCKING on the final run: if the artifact on disk cannot be reloaded and reproduce its own
# outputs, there is no deployable model, and deployability is a claim this paper makes.
record_gate("Gate8d_ArtifactReload",
            bool(DEPLOY_CHECKS) and all(v.get("deployment_verified", False)
                                        for v in DEPLOY_CHECKS.values()),
            "; ".join(f"{k}={'verified' if v.get('deployment_verified') else 'NOT verified'}"
                      f" via {v.get('reload_method', 'n/a')}" for k, v in DEPLOY_CHECKS.items()),
            blocking=not PREFLIGHT)
save_json(DEPLOY_CHECKS, f"{RESULTS_DIR}/deployment_verification.json")
print(json.dumps(DEPLOY_CHECKS, indent=2, default=str))

In [ ]:
# ---- Final inference interface (spec 21) -- what a web prototype would call ----
# Two changes from the earlier version, both about serving the RIGHT model:
#   1. it serves DEPLOY_CHOICE (the validation-selected model), not unconditionally the FP32 student.
#      If the rule chose qat_int8, a website that loads best_student_fp32 is not the studied system.
#   2. it loads that model FROM THE EXPORTED ARTIFACT, not from the training object still in RAM,
#      so the thing being demonstrated is the thing that ships.
GRADE_NAMES = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}
CLINICAL_DISCLAIMER = ("Research prototype for the DR-VERGE study. Not a medical device and not a "
                       "standalone clinical diagnosis; outputs must be reviewed by a clinician.")

DEPLOY_EXPORT_NAME = {"best_fp32": "best_student_fp32", "ptq_int8": "best_student_ptq_int8",
                      "ft_ptq_int8": "best_student_ft_ptq_int8",
                      "qat_int8": "best_student_qat_int8"}.get(DEPLOY_CHOICE, "best_student_fp32")

_INFER_CACHE = {}

def _deployment_builder():
    """Rebuild the selected model's skeleton so its saved state_dict can be loaded into it."""
    if DEPLOY_CHOICE == "qat_int8":
        return build_qat_skeleton()
    if DEPLOY_CHOICE in ("ptq_int8", "ft_ptq_int8"):
        return build_ptq_skeleton()
    return DualViewLightStudent(NUM_CLASSES, init_thresholds=INIT_THRESHOLDS)

def get_deployment_model(choice=None, device="cpu"):
    """Load the SELECTED deployment model from its artifact on disk. Cached: a server calls this
    once at start-up, never per request."""
    choice = choice or DEPLOY_CHOICE
    key = (choice, device)
    if key in _INFER_CACHE:
        return _INFER_CACHE[key]
    name = {"best_fp32": "best_student_fp32", "ptq_int8": "best_student_ptq_int8",
            "ft_ptq_int8": "best_student_ft_ptq_int8",
            "qat_int8": "best_student_qat_int8"}.get(choice, "best_student_fp32")
    ck = f"{MODELS_DIR}/{name}/checkpoint.pt"
    model = None
    if os.path.exists(ck):
        try:
            model = _deployment_builder()
            st = robust_torch_load(ck, map_location="cpu")
            model.load_state_dict(st["model_state"] if isinstance(st, dict) and "model_state" in st else st)
            model.eval()
            print(f"  deployment model '{choice}' loaded from artifact {ck}")
        except Exception as e:
            print(f"  could not load '{choice}' from disk ({e!r}) -- falling back to the in-memory FP32 model")
            model = None
    if model is None:
        model = copy.deepcopy(BEST_FP32_MODEL).to("cpu").eval()
    # INT8 models are CPU-only; asking for CUDA silently returning a CPU model would be a lie, so
    # the device actually used is reported back in every prediction.
    if device != "cpu" and DEPLOY_QUANTIZATION == "FP32":
        model = model.to(device)
    _INFER_CACHE[key] = model
    return model

def get_inference_model(device="cpu"):      # backwards-compatible name
    return get_deployment_model(device=device)

def predict_dr(macula_image, optic_disc_image, model=None, model_version=None, device="cpu"):
    """macula_image / optic_disc_image: HxWx3 uint8 RGB arrays or file paths."""
    m = model if model is not None else get_deployment_model(device=device)
    used_device = "cpu" if (DEPLOY_QUANTIZATION != "FP32" or device == "cpu") else device
    version = model_version or f"{DEPLOY_CHOICE}_seed{DEPLOY_CHOICE_SEED}_{DEPLOY_QUANTIZATION}"
    def prep(x):
        arr = _rgb(x) if isinstance(x, str) else np.asarray(x)
        return eval_transform(image=arr)["image"].unsqueeze(0).to(used_device)
    a, b = prep(macula_image), prep(optic_disc_image)
    t0 = time.perf_counter()
    with torch.no_grad():
        o = m(a, b)
        p = (o["p_dual"] if isinstance(o, dict) else o)[0]
    dt = (time.perf_counter() - t0) * 1000
    cum = p.cpu().numpy()
    grade = int((cum > 0.5).sum())
    # cumulative P(y>k) -> per-class scores
    ext = np.concatenate([[1.0], cum, [0.0]])
    scores = np.clip(ext[:-1] - ext[1:], 0, None); scores = scores / max(scores.sum(), 1e-9)
    return {
        "grade": grade,
        "grade_name": GRADE_NAMES[grade],
        "ordinal_scores": cum.tolist(),              # P(y > k), monotone by construction
        "grade_scores": scores.tolist(),
        # Deliberately NOT called a probability or a confidence: weighted-BCE training distorts the
        # sigmoid outputs, so this is an uncalibrated derived score. See OrdinalThreshold_ECE/Brier.
        "uncalibrated_score": float(scores[grade]),
        "model_version": version,
        "deployment_choice": DEPLOY_CHOICE,
        "quantization": DEPLOY_QUANTIZATION,
        "device": used_device,
        "latency_ms": dt,
        "preprocessing": PREPROCESSING_META,
        "disclaimer": CLINICAL_DISCLAIMER,
    }

_demo = pd.read_csv(DRTID_TEST_CSV).iloc[0]
_out = predict_dr(_demo["macula_path"], _demo["disc_path"])
print("predict_dr() demo ->", json.dumps({k: v for k, v in _out.items() if k != "preprocessing"},
                                          indent=2, default=str))
print("true grade:", int(_demo["grade"]))

In [ ]:
# The deployment decision was made on VALIDATION, before this section ran (see the RQ2 validation
# table). Nothing here may change it -- DEPLOY_CHOICE_FROZEN is asserted so a later edit cannot
# quietly reintroduce test-set-driven selection.
assert DEPLOY_CHOICE_FROZEN, "deployment choice must be frozen before the test set is evaluated"
print(f"Deployment model (frozen on validation): {DEPLOY_CHOICE} -- {DEPLOY_REASON}")

# ---- Model registry (spec 58) ----
reg = []
_ext_lookup = {}
if len(EXT_DF):
    _prim_ext = EXT_DF[(EXT_DF.field_order == DEEPDRID_PRIMARY_FIELD_ORDER) &
                       (EXT_DF.subset == DEEPDRID_PRIMARY_SUBSET)]
    for _, r in _prim_ext.iterrows():
        _ext_lookup[r["condition"]] = r["QWK"]

for name, (d, status, meta) in EXPORTS.items():
    cond = meta.get("condition", meta.get("role", name))
    quant = meta.get("quantization", "FP32")
    match = RAW[(RAW.condition == {"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                   "best_student_ptq_int8": "ptq_int8", "best_student_qat_int8": "qat_int8",
                                   "best_csd_fp32": "dual_csd"}.get(name, name))]
    if name == "best_csd_fp32": match = match[match.seed == BEST_CSD_SEED]
    if name == "best_student_qat_int8" and QAT_DEPLOY_SEED is not None:
        match = match[match.seed == QAT_DEPLOY_SEED]
    row = match.iloc[0] if len(match) else None
    reg.append({
        "model_id": name, "condition": cond, "seed": meta.get("training_seed", "-"),
        "checkpoint_path": f"{d}/checkpoint.pt",
        "pt2_path": f"{d}/model.pt2" if status["torch_export"] else "",
        "onnx_path": f"{d}/model.onnx" if status["onnx"] else "",
        "val_qwk": meta.get("best_val_qwk", np.nan),
        "test_qwk": float(row["QWK"]) if row is not None else np.nan,
        "test_macro_f1": float(row["MacroF1"]) if row is not None else np.nan,
        "test_accuracy": float(row["Accuracy"]) if row is not None else np.nan,
        "external_qwk": _ext_lookup.get({"teacher_fp32": "teacher", "best_student_fp32": "best_fp32",
                                          "best_student_ptq_int8": "ptq_int8",
                                          "best_student_qat_int8": "qat_int8"}.get(name, name), np.nan),
        "params": int(row["ParamCount"]) if row is not None and not pd.isna(row.get("ParamCount")) else np.nan,
        "size_mb": file_size_mb(f"{d}/checkpoint.pt"),
        "latency_median_ms": float(row["Latency_median_ms"]) if row is not None and "Latency_median_ms" in row else np.nan,
        "quantization": quant,
        "deployment_artifact_size_mb": (file_size_mb(f"{d}/model.pt2") if status["torch_export"]
                                        else file_size_mb(f"{d}/model_object.pt")),
        "deployable": bool(status["state_dict"] and (name not in DEPLOY_CHECKS or
                                                     DEPLOY_CHECKS[name].get("deployment_verified", False))),
    })
REGISTRY = pd.DataFrame(reg)
REGISTRY.to_csv(f"{ART}/model_registry.csv", index=False)
print(REGISTRY.to_string(index=False))

# ---- selected_deployment/ : the single folder a web prototype consumes ----
# Everything needed to serve and to cite the model lives here, so nobody has to reconstruct which of
# the five exported models was the studied one.
import shutil as _shutil
SELECTED_DIR = f"{MODELS_DIR}/selected_deployment"
os.makedirs(SELECTED_DIR, exist_ok=True)
_src_dir = EXPORTS.get(DEPLOY_EXPORT_NAME, (None,))[0] if DEPLOY_EXPORT_NAME in EXPORTS else None
_copied = []
if _src_dir and os.path.isdir(_src_dir):
    for fn in os.listdir(_src_dir):
        try:
            _shutil.copy2(os.path.join(_src_dir, fn), os.path.join(SELECTED_DIR, fn))
            _copied.append(fn)
        except Exception as e:
            print(f"  could not copy {fn} into selected_deployment: {e!r}")

def _metrics_for(df, cond, seed=None, cols=("QWK", "MacroF1", "Accuracy", "MAE", "SevereErrorRate")):
    if df is None or not len(df) or cond not in set(df.get("condition", [])): return {}
    sub = df[df.condition == cond]
    if seed is not None and "seed" in sub and (sub.seed == seed).any():
        sub = sub[sub.seed == seed]
    return {c: float(sub[c].mean()) for c in cols if c in sub.columns}

_ext_primary = (EXT_DF[(EXT_DF.role == "PRIMARY")] if len(EXT_DF) and "role" in EXT_DF else pd.DataFrame())
SELECTED_METADATA = {
    "source_model": DEPLOY_EXPORT_NAME,
    "deployment_choice": DEPLOY_CHOICE,
    "seed": DEPLOY_CHOICE_SEED,
    "method": BEST_CONDITION,
    "method_label": pretty(BEST_CONDITION),
    "quantization": DEPLOY_QUANTIZATION,
    "quantization_scope": QUANT_SCOPE,
    "quantization_engine": QUANT_ENGINE,
    "selection_rule": DEPLOY_RULE,
    "selection_reason": DEPLOY_REASON,
    "selected_on": "DRTiD validation split only (frozen before test/external evaluation)",
    "drtid_validation_metrics": (RQ2_VALIDATION_SUMMARY[RQ2_VALIDATION_SUMMARY.condition == DEPLOY_CHOICE]
                                 .to_dict(orient="records")),
    "drtid_test_metrics": _metrics_for(RAW, DEPLOY_CHOICE, DEPLOY_CHOICE_SEED),
    "deepdrid_primary_metrics": _metrics_for(_ext_primary, DEPLOY_CHOICE),
    "grade_mapping": {int(k): v for k, v in GRADE_NAMES.items()},
    "num_thresholds": NUM_THRESHOLDS,
    "decision_rule": "grade = sum_k [ P(y>k) > 0.5 ]",
    "preprocessing": PREPROCESSING_META,
    "artifacts": _copied,
    "torch_version": torch.__version__,
    "torchao_version": ENVIRONMENT.get("torchao"),
    "protocol_hash": PROTOCOL_HASH,
    "run_tag": RUN_TAG,
    "deterministic": FINAL_DETERMINISTIC,
    "disclaimer": CLINICAL_DISCLAIMER,
}
save_json(SELECTED_METADATA, f"{SELECTED_DIR}/metadata.json")
print(f"\nselected_deployment/ <- {DEPLOY_EXPORT_NAME} ({DEPLOY_CHOICE}, seed {DEPLOY_CHOICE_SEED}, "
      f"{DEPLOY_QUANTIZATION}); files: {_copied}")

## 38 — Figures & companion CSVs

Every figure is written as **PNG (400 dpi) + PDF + SVG**, and every figure ships a
`*_data.csv` with exactly the numbers plotted. No value exists only inside an image.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 400, "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True,
                     "figure.autolayout": False})

def save_figure(fig, stem, data_df, caption=""):
    for ext in ("png", "pdf", "svg"):
        fig.savefig(f"{FIGURES_DIR}/{stem}.{ext}", bbox_inches="tight")
    data_df.to_csv(f"{FIGURES_DIR}/{stem}_data.csv", index=False)
    if caption:
        with open(f"{FIGURES_DIR}/{stem}_caption.txt", "w") as f: f.write(caption)
    plt.close(fig)
    print(f"  saved {stem} (.png/.pdf/.svg + _data.csv)")

def order_present(order, df=None):
    df = RAW if df is None else df
    return [c for c in order if c in set(df["condition"])]


def agg_stat(df, conds, col):
    means, sds, ns = [], [], []
    for c in conds:
        v = df[df.condition == c][col].dropna()
        means.append(v.mean() if len(v) else np.nan)
        sds.append(v.std() if len(v) > 1 else 0.0)
        ns.append(len(v))
    return np.array(means), np.array(sds), np.array(ns)

In [ ]:
# ---- Figure 1: architecture / workflow schematic (hero figure) ----
fig, ax = plt.subplots(figsize=(11, 6)); ax.axis("off"); ax.grid(False)
def box(x, y, w, h, txt, fc):
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=fc, edgecolor="#333", lw=1.4, zorder=2))
    ax.text(x + w/2, y + h/2, txt, ha="center", va="center", fontsize=10, zorder=3)
def arrow(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", lw=1.6, color="#333"))

box(0.02, 0.72, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.58, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
box(0.24, 0.62, 0.20, 0.20, "Teacher\nResNet-50 dual-view\n+ CORAL heads", "#F7DCDC")
box(0.50, 0.72, 0.24, 0.10, "p_dual, p_macula, p_disc", "#FFF3CD")
box(0.50, 0.58, 0.24, 0.10, r"$\Delta^T=p_{dual}-p_{agg}$", "#FFF3CD")
box(0.24, 0.30, 0.20, 0.18, "Lightweight student\ndepthwise-separable\n+ InteractionFusion", "#DCF7E3")
box(0.50, 0.34, 0.24, 0.10, "L = task + aux\n+ α·KD + β·CSD", "#FFF3CD")
box(0.80, 0.46, 0.17, 0.09, "Best FP32 (M*)", "#DCF7E3")
box(0.80, 0.32, 0.17, 0.09, "PTQ INT8", "#E8DCF7"); box(0.80, 0.18, 0.17, 0.09, "QAT INT8", "#E8DCF7")
box(0.02, 0.30, 0.15, 0.10, "Macula view", "#DCE9F7"); box(0.02, 0.16, 0.15, 0.10, "Optic-disc view", "#DCE9F7")
arrow(0.17, 0.77, 0.24, 0.74); arrow(0.17, 0.63, 0.24, 0.68)
arrow(0.44, 0.74, 0.50, 0.77); arrow(0.44, 0.68, 0.50, 0.63)
arrow(0.17, 0.35, 0.24, 0.40); arrow(0.17, 0.21, 0.24, 0.36)
arrow(0.44, 0.39, 0.50, 0.39); arrow(0.62, 0.58, 0.62, 0.44)
arrow(0.74, 0.39, 0.80, 0.50); arrow(0.885, 0.46, 0.885, 0.41); arrow(0.885, 0.32, 0.885, 0.27)
ax.text(0.63, 0.535, "CSD", fontsize=9, ha="center", color="#B03A2E")
ax.set_xlim(0, 1); ax.set_ylim(0.1, 0.9)
ax.set_title("Figure 1 — DR-VERGE: complementarity-shift distillation and INT8 deployment", fontsize=12)
save_figure(fig, "fig_01_architecture",
            pd.DataFrame([{"component": "teacher", "detail": "ResNet-50 dual-view + CORAL"},
                          {"component": "student", "detail": f"depthwise-separable {STUDENT_CHANNELS}"},
                          {"component": "distillation", "detail": "task + aux + logit-KD + CSD"},
                          {"component": "deployment", "detail": "FP32 / PTQ INT8 / QAT INT8"}]),
            "DR-VERGE architecture. Teacher produces dual and single-view cumulative probabilities; "
            "their difference is the complementarity shift distilled into the lightweight student.")

In [ ]:
# ---- Figure 2: experimental workflow ----
fig, ax = plt.subplots(figsize=(11, 6.5)); ax.axis("off"); ax.grid(False)
steps = [("DRTiD official train (1000 eyes)", 0.86, "#DCE9F7"),
         ("stratified split -> train 800 / val 200", 0.74, "#DCE9F7"),
         ("train all conditions (5 seeds)", 0.62, "#DCF7E3"),
         ("pre-registered grids -> select on VALIDATION only", 0.50, "#FFF3CD"),
         ("M* = best FP32 (validation-selected)", 0.38, "#DCF7E3"),
         ("PTQ INT8 / QAT INT8 / FP32-FT control", 0.26, "#E8DCF7"),
         ("DRTiD official test (no selection here)", 0.14, "#F7DCDC"),
         ("DeepDRiD external -- FROZEN, evaluated last", 0.03, "#F7DCDC")]
for txt, y, fc in steps:
    ax.add_patch(plt.Rectangle((0.18, y), 0.64, 0.075, facecolor=fc, edgecolor="#333", lw=1.3))
    ax.text(0.5, y + 0.037, txt, ha="center", va="center", fontsize=10)
for i in range(len(steps) - 1):
    y1 = steps[i][1]; y2 = steps[i + 1][1] + 0.075
    ax.annotate("", xy=(0.5, y2), xytext=(0.5, y1), arrowprops=dict(arrowstyle="<-", lw=1.5, color="#333"))
ax.set_xlim(0, 1); ax.set_ylim(0, 0.95)
ax.set_title("Figure 2 — Experimental workflow (selection never touches test or external data)", fontsize=12)
save_figure(fig, "fig_02_experimental_workflow", pd.DataFrame([{"step": t} for t, _, _ in steps]),
            "Experimental workflow. All selection happens on validation; DRTiD test is evaluated once "
            "within this run and DeepDRiD is frozen until the very end.")

In [ ]:
# ---- Figure 3: predictive performance comparison (QWK / Macro-F1 / Accuracy) ----
conds = order_present(["teacher", "macula_only", "disc_only", "dual_no_distill",
                       "dual_logitkd", "dual_featkd", "dual_csd"])
metrics3 = ["QWK", "MacroF1", "Accuracy"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
rows3 = []
for ax, met in zip(axes, metrics3):
    mu, sd, ns = agg_stat(RAW, conds, met)
    ax.bar(range(len(conds)), mu, yerr=sd, capsize=4, color="#4C72B0", edgecolor="#25405e")
    for i, c in enumerate(conds):
        pts = RAW[RAW.condition == c][met].dropna().values
        if len(pts) > 1: ax.scatter([i] * len(pts), pts, s=16, color="#C44E52", zorder=3, alpha=.85)
    ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds, rotation=35, ha="right", fontsize=9)
    ax.set_title(f"{met} (higher is better)"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(conds, mu, sd, ns):
        rows3.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 3 — Predictive performance on the internal DRTiD test set (mean ± SD over seeds; dots = individual seeds)", y=1.02)
save_figure(fig, "fig_03_performance_comparison", pd.DataFrame(rows3),
            "Predictive performance. Error bars are SD across seeds; red dots are individual seed values.")

In [ ]:
# ---- Figure 4: efficiency Pareto frontier ----
eff_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows4 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (xcol, xlabel) in zip(axes, [("Latency_median_ms", "CPU latency, median (ms) — lower is better"),
                                     ("CheckpointSize_MB", "Serialized size (MB) — lower is better")]):
    for c in eff_conds:
        sub = RAW[RAW.condition == c]
        if not len(sub) or xcol not in sub: continue
        x, y = sub[xcol].mean(), sub["QWK"].mean()
        if pd.isna(x): continue
        mk = "D" if c == "teacher" else ("*" if "int8" in c else "o")
        ax.scatter(x, y, s=320 if mk != "o" else 150, marker=mk, label=c, edgecolor="#222", zorder=3)
        ax.annotate(c, (x, y), textcoords="offset points", xytext=(8, 6), fontsize=9)
        rows4.append({"axis": xcol, "condition": c, "x": x, "QWK": y})
    ax.set_xlabel(xlabel); ax.set_ylabel("QWK (higher is better)"); ax.set_xscale("log")
    ax.set_title("Top-left is better")
fig.suptitle("Figure 4 — Efficiency–Performance Pareto Frontier", y=1.02)
save_figure(fig, "fig_04_efficiency_pareto", pd.DataFrame(rows4),
            "Efficiency-performance frontier: QWK against CPU latency and serialized state_dict size (log x-axis).")

In [ ]:
# ---- Figure 5: quantization retention (FP32 vs PTQ vs QAT) ----
qconds = order_present(["best_fp32", "ptq_int8", "qat_int8"])
qmetrics = ["QWK", "Accuracy", "MacroF1", "MacroRecall"]
rows5 = []
if len(qconds) >= 2:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    w = 0.8 / len(qconds)
    for i, c in enumerate(qconds):
        vals = [RAW[RAW.condition == c][m].mean() for m in qmetrics]
        ax.bar(np.arange(len(qmetrics)) + i * w, vals, width=w, label=c, edgecolor="#25405e")
        for m, v in zip(qmetrics, vals): rows5.append({"condition": c, "metric": m, "value": v})
    ax.set_xticks(np.arange(len(qmetrics)) + w * (len(qconds) - 1) / 2); ax.set_xticklabels(qmetrics)
    ax.set_ylabel("score (higher is better)"); ax.legend()
    ax.set_title("Figure 5 — Quantization: FP32 vs PTQ INT8 vs QAT INT8")
    save_figure(fig, "fig_05_quantization_retention", pd.DataFrame(rows5),
                "Diagnostic performance retained after INT8 quantization.")
else:
    print("  fig_05 skipped: fewer than two quantization variants available")

In [ ]:
# ---- Figure 6: per-grade sensitivity (the rev2 failure mode made permanently visible) ----
pg_conds = order_present(["teacher", "best_fp32", "dual_csd", "ptq_int8", "qat_int8"])
rows6 = []
fig, ax = plt.subplots(figsize=(11, 5.5))
w = 0.8 / max(len(pg_conds), 1)
for i, c in enumerate(pg_conds):
    vals = [RAW[RAW.condition == c][f"Sensitivity_Grade{g}"].mean() for g in range(NUM_CLASSES)]
    ax.bar(np.arange(NUM_CLASSES) + i * w, vals, width=w, label=c, edgecolor="#25405e")
    for g, v in enumerate(vals): rows6.append({"condition": c, "grade": g, "sensitivity": v})
ax.axhline(0.05, color="#C44E52", ls="--", lw=1.2, label="collapse floor (0.05)")
ax.set_xticks(np.arange(NUM_CLASSES) + w * (len(pg_conds) - 1) / 2)
ax.set_xticklabels([f"Grade {g}" for g in range(NUM_CLASSES)])
ax.set_ylabel("Sensitivity / recall (higher is better)"); ax.set_ylim(0, 1); ax.legend(fontsize=9)
ax.set_title("Figure 6 — Per-grade sensitivity: are intermediate grades actually predicted?")
save_figure(fig, "fig_06_per_grade_sensitivity", pd.DataFrame(rows6),
            "Per-grade recall. Grades 1-3 near zero indicates the ordinal collapse seen in rev2.")

In [ ]:
# ---- Figure 7: normalized confusion matrices ----
cm_conds = order_present(["teacher", "best_fp32", "ptq_int8", "qat_int8"])
rows7 = []
if cm_conds:
    fig, axes = plt.subplots(1, len(cm_conds), figsize=(4.6 * len(cm_conds), 4.4))
    if len(cm_conds) == 1: axes = [axes]
    for ax, c in zip(axes, cm_conds):
        key = next((k for k in PRED_STORE if k[0] == c), None)
        if key is None: ax.axis("off"); continue
        d = PRED_STORE[key]
        cm = confusion_matrix(d["y_true"], d["y_pred"], labels=list(range(NUM_CLASSES)))
        cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        ax.set_title(c, fontsize=11); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if cmn[i, j] > .5 else "black")
                rows7.append({"condition": c, "true": i, "pred": j, "count": int(cm[i, j]),
                              "normalized": float(cmn[i, j])})
    fig.suptitle("Figure 7 — Row-normalized confusion matrices (internal test set)", y=1.03)
    save_figure(fig, "fig_07_confusion_matrices", pd.DataFrame(rows7),
                "Row-normalized confusion matrices; diagonal = per-grade recall.")

In [ ]:
# ---- Figure 8: CSD mechanism (the primary RQ1 mechanism figure) ----
mech_conds = order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd",
                            "abl_csd_counterfactual"])
mech = ["ShiftL1", "CosAgree", "BenefitCorr"]
rows8 = []
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, met in zip(axes, mech):
    mu, sd, ns = agg_stat(RAW, mech_conds, met)
    better = "lower is better" if met in ("ShiftL1", "ShiftMAE") else "higher is better"
    ax.bar(range(len(mech_conds)), mu, yerr=sd, capsize=4, color="#55A868", edgecolor="#2f5d3f")
    ax.set_xticks(range(len(mech_conds)))
    ax.set_xticklabels([pretty(c) for c in mech_conds], rotation=35, ha="right", fontsize=8)
    ax.set_title(f"{met} ({better})"); ax.set_ylabel(met)
    for c, m_, s_, n_ in zip(mech_conds, mu, sd, ns):
        rows8.append({"metric": met, "condition": c, "mean": m_, "sd": s_, "n_seeds": n_})
fig.suptitle("Figure 8 — CSD mechanism: is the teacher's complementarity shift actually transferred?", y=1.02)
save_figure(fig, "fig_08_csd_mechanism", pd.DataFrame(rows8),
            "Mechanism fidelity. ShiftMAE lower = student shift closer to teacher; CosAgree higher = same "
            "shift direction; BenefitCorr higher = student gains from dual-view on the same samples as the teacher.")

In [ ]:
# ---- Figure 9: gradient contributions over training ----
rows9 = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cond in ["dual_csd", "dual_logitkd", "dual_no_distill", "dual_featkd"]:
    f = f"{LOGS_DIR}/gradient_contributions_{cond}_{PRIMARY_SEED}.csv"
    if not os.path.exists(f): continue
    h = pd.read_csv(f)
    if "gnorm_task" in h:
        axes[0].plot(h["epoch"], h["gnorm_task"], label=f"{cond}: task", lw=1.4)
    if "gnorm_csd" in h:
        axes[0].plot(h["epoch"], h["gnorm_csd"], label=f"{cond}: CSD", lw=1.8, ls="--")
    if "gnorm_ratio_csd_over_task" in h:
        axes[1].plot(h["epoch"], h["gnorm_ratio_csd_over_task"], label=cond, lw=1.8)
    for _, r in h.iterrows():
        rows9.append({"condition": cond, **{k: r[k] for k in h.columns}})
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("gradient L2 norm"); axes[0].set_yscale("log")
axes[0].set_title("Per-component gradient norm"); axes[0].legend(fontsize=8)
axes[1].axhline(1.0, color="#888", ls=":", lw=1)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("‖∇L_CSD‖ / ‖∇L_task‖")
axes[1].set_title("CSD gradient share (rev2 was ≈0 — CSD had no influence)"); axes[1].legend(fontsize=8)
fig.suptitle("Figure 9 — Optimization signal: does CSD actually contribute gradient?", y=1.02)
save_figure(fig, "fig_09_gradient_contribution", pd.DataFrame(rows9),
            "Gradient norms per loss component. The ratio panel shows whether CSD is a real training "
            "signal or numerical decoration.")

In [ ]:
# ---- Figure 10: reliability diagrams (calibration) ----
rel_conds = order_present(["best_fp32", "fp32_ft_control", "ptq_int8", "qat_int8"])
rows10 = []
if rel_conds:
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot([0, 1], [0, 1], ls="--", color="#888", label="perfectly calibrated")
    for c in rel_conds:
        # Resolve the prediction file from PRED_STORE's OWN keys. The earlier code hard-coded
        # seed{BEST_SEED} for every condition, but QAT is trained on SEEDS_QAT and need not contain
        # BEST_SEED at all -- so the QAT reliability curve could silently disappear from the figure.
        keys = [k for k in PRED_STORE if k[0] == c]
        if not keys: continue
        key = next((k for k in keys if k[1] == QAT_DEPLOY_SEED), keys[0]) if c == "qat_int8" else keys[0]
        _c, _sd, _q = key
        pf = f"{PREDS_DIR}/DRTiD_test_{_c}_seed{_sd}_{_q}.csv"
        if not os.path.exists(pf): continue
        dfp = pd.read_csv(pf)
        p = torch.tensor(dfp[[f"p_threshold_{k}" for k in range(NUM_THRESHOLDS)]].values, dtype=torch.float32)
        rc = reliability_curve(p, dfp["true_grade"].values)
        ax.plot(rc["mean_predicted"], rc["observed_frequency"], marker="o", lw=1.8,
                label=f"{pretty(c)} (seed {_sd})")
        cal = compute_calibration(p, dfp["true_grade"].values)
        for _, r in rc.iterrows():
            rows10.append({"condition": c, "seed": _sd, **r.to_dict(),
                           "OrdinalThreshold_ECE": cal["OrdinalThreshold_ECE"],
                           "OrdinalThreshold_Brier": cal["OrdinalThreshold_Brier"]})
    ax.set_xlabel("Mean predicted P(y>k)"); ax.set_ylabel("Observed frequency")
    ax.set_title("Figure 10 — Reliability diagram (pooled cumulative thresholds)")
    ax.legend(fontsize=9)
    save_figure(fig, "fig_10_calibration_reliability", pd.DataFrame(rows10),
                "Reliability diagram. Deviation from the diagonal indicates miscalibration; "
                "companion CSV carries ECE and Brier per condition.")
print(f"\nAll figures written to {FIGURES_DIR}")

In [ ]:
# ---- Figure 11: internal vs external generalization ----
if len(EXT_DF):
    prim = EXT_DF[(EXT_DF.field_order == DEEPDRID_PRIMARY_FIELD_ORDER) &
                  (EXT_DF.subset == DEEPDRID_PRIMARY_SUBSET)]
    mets = ["QWK", "MacroF1", "Accuracy"]
    conds = [c for c in ["teacher", "best_fp32", "best_csd_fp32", "ptq_int8", "qat_int8"]
             if c in set(prim.condition)]
    rows = []
    fig, axes = plt.subplots(1, len(mets), figsize=(5.2 * len(mets), 5))
    for ax, met in zip(axes, mets):
        internal = [RAW[RAW.condition == c][met].mean() for c in conds]
        external = [prim[prim.condition == c][met].mean() for c in conds]
        x = np.arange(len(conds)); w = 0.38
        ax.bar(x - w/2, internal, w, label="DRTiD (internal)", edgecolor="#25405e")
        ax.bar(x + w/2, external, w, label="DeepDRiD (external, frozen)", edgecolor="#5e2540")
        ax.set_xticks(x); ax.set_xticklabels([pretty(c) for c in conds], rotation=30, ha="right", fontsize=8)
        ax.set_title(f"{met} (higher is better)"); ax.legend(fontsize=8)
        for c, i_, e_ in zip(conds, internal, external):
            rows.append({"metric": met, "condition": c, "internal_DRTiD": i_, "external_DeepDRiD": e_,
                         "delta_external_minus_internal": e_ - i_})
    fig.suptitle("Figure 11 — Internal vs external generalization "
                 f"(DeepDRiD {DEEPDRID_PRIMARY_SUBSET} partition, primary field ordering)", y=1.02)
    save_figure(fig, "fig_11_external_generalization", pd.DataFrame(rows),
                "Internal (DRTiD) vs external (DeepDRiD) performance. A drop is a domain-shift "
                "finding to report, not something to tune away.")
else:
    print("  fig_11 skipped: no external validation results available")

## 39 — Final research tables

In [ ]:
# table_diagnostic_performance
diag_cols = ["QWK", "Accuracy", "MacroPrecision", "MacroRecall", "MacroF1", "WeightedF1",
             "BalancedAccuracy", "MAE", "SevereErrorRate", "ECE", "Brier"]
t_diag = RAW.groupby("condition")[diag_cols].agg(["mean", "std"]).round(4)
t_diag.columns = ["_".join(c) for c in t_diag.columns]
t_diag = t_diag.reindex([c for c in DISPLAY_ORDER if c in t_diag.index])
t_diag.to_csv(f"{TABLES_DIR}/table_diagnostic_performance.csv")

# table_efficiency
eff_cols = [c for c in ["ParamCount", "CheckpointSize_MB", "Latency_mean_ms", "Latency_median_ms",
                        "Latency_sd_ms", "Latency_p95_ms", "Latency_p99_ms",
                        "Throughput_pairs_per_s", "Throughput_images_per_s"]
            if c in RAW.columns]
t_eff = RAW.groupby("condition")[eff_cols].mean().round(4)
ref_lat = t_eff.loc["teacher", "Latency_median_ms"] if "teacher" in t_eff.index else np.nan
ref_size = t_eff.loc["teacher", "CheckpointSize_MB"] if "teacher" in t_eff.index else np.nan
t_eff["Speedup_vs_teacher"] = (ref_lat / t_eff["Latency_median_ms"]).round(2)
t_eff["CompressionRatio_vs_teacher"] = (ref_size / t_eff["CheckpointSize_MB"]).round(2)
t_eff["SizeReduction_vs_teacher_pct"] = ((1 - t_eff["CheckpointSize_MB"] / ref_size) * 100).round(2)
t_eff = t_eff.reindex([c for c in DISPLAY_ORDER if c in t_eff.index])
t_eff.to_csv(f"{TABLES_DIR}/table_efficiency.csv")

# table_quantization (FP32 vs PTQ vs QAT) + retention
qrows = []
fp32_m = RAW[RAW.condition == "best_fp32"][diag_cols].mean().to_dict() if "best_fp32" in set(RAW.condition) else {}
for c in ["best_fp32", "fp32_ft_control", "fp32_ft_plain", "ptq_int8", "ft_ptq_int8", "qat_int8",
          "ptq_int8_pt2e"]:
    if c not in set(RAW.condition): continue
    m = RAW[RAW.condition == c][diag_cols].mean().to_dict()
    e = RAW[RAW.condition == c][eff_cols].mean().to_dict()
    row = {"model": c, **{k: round(v, 4) for k, v in m.items()}, **{k: round(v, 4) for k, v in e.items()}}
    if fp32_m and c != "best_fp32":
        row.update({k: round(v, 3) for k, v in retention_metrics(m, fp32_m).items()})
        row.update(efficiency_derived(e.get("CheckpointSize_MB"), fp32_m and RAW[RAW.condition=="best_fp32"]["CheckpointSize_MB"].mean(),
                                      e.get("Latency_median_ms"), RAW[RAW.condition=="best_fp32"]["Latency_median_ms"].mean()))
    qrows.append(row)
t_quant = pd.DataFrame(qrows)
t_quant.to_csv(f"{TABLES_DIR}/table_quantization.csv", index=False)

# table_csd_mechanism
mech_cols = ["ShiftL1", "ShiftMAE", "CosAgree", "BenefitCorr", "BenefitCorrSpearman",
             "DualViewGain_G_internal", "DualViewGain_G_external"]
t_mech = RAW[RAW.view_mode == "dual"].groupby("condition")[
    [c for c in mech_cols if c in RAW.columns]].agg(["mean", "std"]).round(4)
t_mech.columns = ["_".join(c) for c in t_mech.columns]
t_mech.to_csv(f"{TABLES_DIR}/table_csd_mechanism.csv")

print("Diagnostic performance:\n", t_diag[[c for c in t_diag.columns if c.endswith("_mean")]].to_string())
print("\nEfficiency:\n", t_eff.to_string())
print("\nQuantization:\n", t_quant.to_string(index=False))
print("\nCSD mechanism:\n", t_mech.to_string())

## 40–41 — Automatic headline generator & final gate report

In [ ]:
def _mean(cond, col):
    v = RAW[RAW.condition == cond][col].dropna()
    return float(v.mean()) if len(v) else float("nan")

headlines = []
t_q, s_q = _mean("teacher", "QWK"), _mean("best_fp32", "QWK")
t_p, s_p = _mean("teacher", "ParamCount"), _mean("best_fp32", "ParamCount")
t_l, s_l = _mean("teacher", "Latency_median_ms"), _mean("best_fp32", "Latency_median_ms")
t_s, s_s = _mean("teacher", "CheckpointSize_MB"), _mean("best_fp32", "CheckpointSize_MB")
if not math.isnan(t_p) and s_p:
    headlines.append(f"Teacher -> Student: {t_p/s_p:.0f}x fewer parameters, {t_s/s_s:.1f}x smaller artifact, "
                     f"{t_l/s_l:.1f}x faster CPU inference, {100*s_q/t_q:.1f}% of teacher QWK retained")
for q in ["ptq_int8", "qat_int8"]:
    if q not in set(RAW.condition): continue
    qq, ql, qs = _mean(q, "QWK"), _mean(q, "Latency_median_ms"), _mean(q, "CheckpointSize_MB")
    headlines.append(f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, "
                     f"{s_l/ql:.2f}x CPU speedup, {s_s/qs:.2f}x smaller"
                     if not math.isnan(qs) and qs else
                     f"FP32 -> {q.upper()}: {100*qq/s_q:.1f}% QWK retained, {s_l/ql:.2f}x CPU speedup")

print("=" * 78); print("AUTOMATIC HEADLINES (computed, NOT significance claims)"); print("=" * 78)
for h in headlines: print("  " + h)
pd.DataFrame({"headline": headlines}).to_csv(f"{TABLES_DIR}/table_headlines.csv", index=False)

In [ ]:
# ---- Gate 5: RQ1 verdict on predictive AND mechanistic axes ----
print("=" * 78); print("GATE 5 -- RQ1 VERDICT"); print("=" * 78)
csd_q = _mean("dual_csd", "QWK")
verdict = {}
for base in ["dual_no_distill", "dual_logitkd", "dual_featkd"]:
    if base not in set(RAW.condition): continue
    bq = _mean(base, "QWK")
    st = STATS[(STATS.comparison == f"dual_csd_vs_{base}") & (STATS.metric == "QWK")]
    ci = f"[{st.iloc[0]['ci_low']:+.4f}, {st.iloc[0]['ci_high']:+.4f}]" if len(st) else "n/a"
    cred = bool(st.iloc[0]["excludes_zero"]) if len(st) else None
    verdict[base] = {"csd_qwk": csd_q, "baseline_qwk": bq, "diff": csd_q - bq,
                     "ci_95": ci, "credible": cred}
    print(f"  CSD vs {base:18s}: {csd_q:.4f} vs {bq:.4f} (diff {csd_q-bq:+.4f}, 95% CI {ci}, credible={cred})")

print("\n  Mechanism (does CSD transfer the shift, independently of QWK?)")
for c in order_present(["dual_no_distill", "dual_logitkd", "dual_featkd", "dual_csd"]):
    print(f"    {c:18s} ShiftL1={_mean(c,'ShiftL1'):.4f}  CosAgree={_mean(c,'CosAgree'):+.4f}  "
          f"BenefitCorr(r)={_mean(c,'BenefitCorr'):+.4f}  (rho)={_mean(c,'BenefitCorrSpearman'):+.4f}")
save_json(verdict, f"{RESULTS_DIR}/rq1_verdict.json")
record_gate("Gate5_RQ1_Comparison", True, "RQ1 comparisons computed on predictive and mechanistic axes")
print("\n  A negative or null RQ1 result is a valid, reportable finding -- do not re-tune the")
print("  method in response to this table. The protocol was locked before the run.")

In [ ]:
# ---- Final consolidated gate report ----
gate_df = pd.DataFrame([{"gate": k, "passed": v["passed"], "detail": v["detail"]} for k, v in GATES.items()])
gate_df.to_csv(f"{TABLES_DIR}/table_gate_report.csv", index=False)
print("=" * 78); print("FINAL GATE REPORT"); print("=" * 78)
print(gate_df.to_string(index=False))
n_pass = int(gate_df.passed.sum())
print(f"\n{n_pass}/{len(gate_df)} gates passed.")
failed = gate_df[~gate_df.passed]
if len(failed):
    print("\nFAILED / NOT-RUN gates (report these honestly rather than hiding them):")
    for _, r in failed.iterrows(): print(f"  - {r['gate']}: {r['detail']}")

RUN_SUMMARY = {
    "environment": ENVIRONMENT, "config": CONFIG_SNAPSHOT,
    "selection": {"best_condition": BEST_CONDITION, "best_seed": BEST_SEED,
                  "best_val_qwk": float(BEST_ROW["QWK"]), "best_csd_seed": BEST_CSD_SEED},
    "csd_selected": {"variant": BEST_CSD_VARIANT, "alpha": BEST_ALPHA, "beta": BEST_BETA},
    "gates": GATES, "headlines": headlines, "rq1_verdict": verdict,
    "n_evaluated_runs": int(len(RAW)),
    "external_validation": "completed" if len(EXT_DF) else "skipped/unavailable",
}
save_json(RUN_SUMMARY, f"{RESULTS_DIR}/run_summary.json")

print(f"""
{'='*78}
ARTIFACTS
{'='*78}
  checkpoints : {CKPT_DIR}
  models      : {MODELS_DIR}   (checkpoint.pt / model.pt2 / model.onnx / metadata.json)
  figures     : {FIGURES_DIR}  (png+pdf+svg + *_data.csv per figure)
  tables      : {TABLES_DIR}
  metrics     : {METRICS_DIR}
  predictions : {PREDS_DIR}    (per-sample, so metrics can be recomputed without re-inference)
  logs        : {LOGS_DIR}
  registry    : {ART}/model_registry.csv
  summary     : {RESULTS_DIR}/run_summary.json
""")

## Done

Read in this order when writing the paper:

1. **`table_gate_report.csv`** — did anything fail? Report failures honestly.
2. **Gate 5 / `rq1_verdict.json`** — RQ1 on both axes (predictive *and* mechanistic).
3. **`table_quantization.csv`** — RQ2: FP32 vs PTQ vs QAT, plus both FP32 fine-tuning controls.
4. **`table_05_statistical_tests.csv`** — effect sizes with CIs; a difference is not a claim unless
   the CI excludes zero. Check `matched_seeds` before describing a row as paired.
5. **`table_06_external_validation_deepdrid.csv`** — external generalization. Quote the
   `role=PRIMARY` rows (validation partition, primary field ordering) as the headline; the other
   rows are supplementary robustness.
6. **`table_condition_labels.csv`** — the exact label to use for each condition in every table.
7. **`configs/quantization_info.json`** — the locked scope, the per-seed RQ2 design, the PTQ
   calibration manifest hash, the selected QAT learning rate, and the note that
   `quantization_coverage_pct` is an integrity check rather than a reportable number.
8. **`deployment_verification.json`** — which artifacts were actually re-loaded from disk and passed.
9. **`table_04_rq2_validation.csv` + `results/deployment_choice.json`** — the deployment decision and
   the *validation* evidence it was made on. Quote these when describing model selection; the test
   set played no part in it.
10. **`table_06c_external_patient_clustered_ci.csv`** — Set-C results with patient-clustered 95% CIs.
    Set-C is the confirmatory partition; Set-B/Set-A rows are labelled supplementary for a reason.
11. **`metrics/deepdrid_exclusion_audit.csv`** — every external eye that was dropped or flagged, with
    the reason. Cite the counts rather than implying the partitions were used whole.
12. **`metrics/csd_final_gradient_diagnostic.json`** — evidence that CSD contributed real gradient at
    the selected β, measured with the frozen teacher.

**Wording that must not drift:**

* CSD transfers *an operational proxy of the dual-view ordinal decision shift* — never "pure
  anatomical complementarity".
* Set-C is *confirmatory external evaluation*; Set-B/Set-A are *external validation* (their aggregate
  performance was seen during development).
* Student variability is reported *conditional on a fixed teacher checkpoint*.
* Effect sizes with CIs are the headline; p-values are secondary.
* INT8 compression is modest by design — only the CNN backbone is quantized. The large compression
  story is teacher → student; the INT8 story is latency and mixed-precision deployability.

Use `docs/judge.md` Section I's safe phrasing for every claim. Do not describe CSD as successful on
mechanism metrics alone if predictive performance did not move; do not call the model
deployment-ready on the basis of size alone.